# Day23A — MJ1 0.4C sub-DC-amplitude protocol-mode audit at fixed 1τ

## Scope

This notebook audits the MJ1 0.4C DC-rate group under sub-DC-amplitude DC–AC excitation.

Active protocols:

- 0.4C DC reference
- 0.4C + 0.1C, fixed 1τ
- 0.4C + 0.3C, fixed 1τ

Excluded protocol:

- 0.4C + 0.2C, fixed 1τ

The 0.4C + 0.2C record is excluded because the raw file lacks the required pre-Vmax CC segment. It cannot support Q_Vmax extraction, first-passage comparison, or boundary-segment assignment.

All active DC–AC protocols are interpreted as fixed 1τ excitation:

f = 1 / (2π · τ_label)

τ_label = 11.1 s

f ≈ 0.014338 Hz

## Scientific motivation

Day21A and Day22A focused on protocols where the AC amplitude exceeded the DC component:

- Day21A: 0.3C + 0.7C, κ = 2.33
- Day22A: 0.3C + 0.4C, κ = 1.33

Day23A examines the opposite amplitude regime:

κ = AC_C / DC_C < 1

Active κ values:

| Protocol | DC_C | AC_C | κ = AC_C / DC_C | Role |
|---|---:|---:|---:|---|
| 0.4C + 0.1C | 0.4 | 0.1 | 0.25 | small sub-DC perturbation |
| 0.4C + 0.3C | 0.4 | 0.3 | 0.75 | near-DC but still sub-DC perturbation |

The central question is whether a boundary-related first-passage effect persists when the AC component becomes smaller than the DC component.

## Protocol-mode discovery

Before applying the Day21A/Day22A segmentation logic, this notebook audits whether the active Day23A DC–AC files follow the same protocol assumption:

AC applied only during CC, then switched off after first V ≥ 4.2 V.

The protocol-mode audit shows that this assumption is not satisfied.

Both active DC–AC files retain substantial fixed-frequency current components after Vmax:

- 0.4C + 0.1C: post-Vmax AC component remains detectable
- 0.4C + 0.3C: post-Vmax AC component remains large

Therefore, Day23A must not use the original Day21A/Day22A Segment A/B/D framework without modification.

## Framework switch

Day21A/Day22A used:

- Segment A: shared prescribed-current region
- Segment B: DCAC AC-off / voltage-limited while DC remains CC
- Segment D: late-CV feedback region

This framework assumes AC-off after Vmax.

Day23A violates that assumption and therefore uses a generalized boundary-ordering framework.

## Generalized boundary-ordering framework

Define:

Q_DC,Vmax = charge at which the DC reference first reaches 4.2 V

Q_DCAC,Vmax = charge at which the DC–AC protocol first reaches 4.2 V

The generalized regions are:

### G0 — shared pre-boundary region

Q ≤ min(Q_DC,Vmax, Q_DCAC,Vmax)

Both trajectories are pre-boundary.

This is the only region where geometry-corrected residual analysis can be considered.

### G1 — boundary-ordering split region

min(Q_DC,Vmax, Q_DCAC,Vmax) < Q ≤ max(Q_DC,Vmax, Q_DCAC,Vmax)

One trajectory has reached the voltage boundary, while the other has not.

The ordering must be explicitly recorded:

- DCAC-first
- DC-first
- degenerate / nearly simultaneous

This region is not equivalent to Day21A/Day22A Segment B unless DCAC is confirmed to be AC-off after Vmax.

### G2 — post-boundary region

Q > max(Q_DC,Vmax, Q_DCAC,Vmax)

Both trajectories have reached the voltage boundary.

In Day23A, this region may still contain continued AC modulation and must not be interpreted as pure late-CV preservation without protocol-mode caveats.

## Methodological constraints

The following rules remain active:

1. strict-net Q integration from signed measured current;
2. no current rectification;
3. no monotonic forcing of Q_net(t);
4. first-passage time is used for t(Q);
5. raw Δt(Q) is real but not mechanism-pure;
6. geometry-corrected residuals are only meaningful in the shared pre-boundary region G0;
7. post-boundary gains must carry continued-AC-after-Vmax caveats;
8. Day23A is not directly comparable to Day21A/Day22A as the same protocol family.

## Day23A-specific caveats

Temperature summaries are missing for this subset and are carried as NaN.

Therefore, Day23A can audit:

- timebase integrity;
- protocol-mode status;
- Q_Vmax ordering;
- generalized G0/G1/G2 assignment;
- raw Δt(Q);
- geometry residual in G0, if waveform fidelity allows.

Day23A cannot support:

- quantitative thermal attribution;
- AC-off Segment B interpretation;
- direct comparison to Day21A/Day22A as identical protocol logic;
- claims of non-geometric Segment-A electrochemical acceleration.

## Interpretive outcomes

Possible outcomes:

### Case 1 — DCAC-first boundary ordering

DC–AC reaches Vmax earlier than DC.

This would indicate a boundary-leading effect, but not necessarily the same AC-off control-state split observed in Day21A/Day22A.

### Case 2 — DC-first boundary ordering

DC reaches Vmax earlier than DC–AC.

This would be a counterexample to the high-amplitude boundary-acceleration pathway and would indicate that sub-DC modulation can delay or redistribute the voltage-boundary event.

### Case 3 — nearly degenerate boundary ordering

Both protocols reach Vmax at similar charge.

This would suggest that sub-DC AC modulation does not substantially shift the voltage-boundary location.

## Non-goals

This notebook does not search for new mechanisms.

It does not assume Day21A/Day22A AC-off segmentation.

It does not interpret post-Vmax gains as late-CV preservation unless AC-off is verified.

It does not treat raw Δt(Q) as mechanism-pure evidence.

In [1]:
# Day23A Cell 0A — raw CSV format inventory
#
# Purpose:
# - Inspect raw CSV files for the 0.4C sub-DC-amplitude group
# - Detect NGU201 raw vs processed CSV format
# - Identify header line, delimiter, time column, and basic metadata
#
# Explicitly NOT done here:
# - No trajectory trimming
# - No Q integration
# - No event detection
# - No segment assignment
# - No verdict

from pathlib import Path
from datetime import datetime, timezone
import re
import csv
import numpy as np
import pandas as pd

REPO = Path("/Users/louislu/pybamm-dcac-superimposed")
DATA_DIR = REPO / "data"
RAW_DIR_DAY23A = DATA_DIR / "raw_mj1_ngu201_day23A_0p4C_subDC"

DAY23A_NOTEBOOK_NAME = "27_day23A_MJ1_0p4C_subDC_1tau_audit.ipynb"
DAY23A_GROUP_ID = "Day23A_MJ1_0p4C_subDC_1tau"

ONE_C_A = 3.4
TAU_LABEL_S = 11.1
FREQ_1TAU_HZ = 1.0 / (2.0 * np.pi * TAU_LABEL_S)

OUT_DAY23A_FORMAT_INVENTORY = DATA_DIR / "day23A_step0A_raw_csv_format_inventory.csv"

EXPECTED_DAY23A_RAW_FILES = [
    "0.4C DC.csv",
    "DC0.4C+AC0.2C f=0.0143Hz.csv",
    "DC0.4C+AC0.3C f=0.0143Hz.csv",
    "处理后DC0.4C+AC0.1C f=0.0143Hz.csv",
]

DAY23A_PROTOCOL_MAP = {
    "0.4C DC.csv": {
        "protocol_label": "0.4C DC",
        "protocol_role": "DC_reference",
        "DC_C": 0.4,
        "AC_C": 0.0,
        "m_tau": np.nan,
        "frequency_Hz": 0.0,
        "kappa": 0.0,
        "candidate_for_DC_reference": True,
        "candidate_for_DCAC": False,
    },
    "处理后DC0.4C+AC0.1C f=0.0143Hz.csv": {
        "protocol_label": "0.4C+0.1C 1tau",
        "protocol_role": "DCAC",
        "DC_C": 0.4,
        "AC_C": 0.1,
        "m_tau": 1.0,
        "frequency_Hz": FREQ_1TAU_HZ,
        "kappa": 0.1 / 0.4,
        "candidate_for_DC_reference": False,
        "candidate_for_DCAC": True,
    },
    "DC0.4C+AC0.2C f=0.0143Hz.csv": {
        "protocol_label": "0.4C+0.2C 1tau",
        "protocol_role": "DCAC",
        "DC_C": 0.4,
        "AC_C": 0.2,
        "m_tau": 1.0,
        "frequency_Hz": FREQ_1TAU_HZ,
        "kappa": 0.2 / 0.4,
        "candidate_for_DC_reference": False,
        "candidate_for_DCAC": True,
    },
    "DC0.4C+AC0.3C f=0.0143Hz.csv": {
        "protocol_label": "0.4C+0.3C 1tau",
        "protocol_role": "DCAC",
        "DC_C": 0.4,
        "AC_C": 0.3,
        "m_tau": 1.0,
        "frequency_Hz": FREQ_1TAU_HZ,
        "kappa": 0.3 / 0.4,
        "candidate_for_DC_reference": False,
        "candidate_for_DCAC": True,
    },
}

print("[OK] Day23A minimal setup loaded.")
print(f"[OK] REPO = {REPO}")
print(f"[OK] DATA_DIR = {DATA_DIR}")
print(f"[OK] RAW_DIR_DAY23A = {RAW_DIR_DAY23A}")
print(f"[OK] FREQ_1TAU_HZ = {FREQ_1TAU_HZ:.9f} Hz")


# =============================================================================
# Helpers
# =============================================================================

def read_text_lines_fallback(path: Path, max_lines: int = 80):
    encodings = ["utf-8-sig", "utf-8", "gbk", "latin1"]
    last_error = None

    for enc in encodings:
        try:
            with open(path, "r", encoding=enc, errors="replace") as f:
                lines = []
                for i, line in enumerate(f):
                    if i >= max_lines:
                        break
                    lines.append(line.rstrip("\n\r"))
            return lines, enc
        except Exception as exc:
            last_error = exc

    raise RuntimeError(f"Could not read {path}: {last_error}")


def infer_delimiter_from_lines(lines):
    candidates = [",", ";", "\t"]
    scores = {d: 0 for d in candidates}

    for line in lines[:30]:
        if not line.strip():
            continue
        for d in candidates:
            scores[d] += line.count(d)

    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else ","


def split_line(line, delimiter):
    return [x.strip() for x in line.split(delimiter)]


def find_header_line_idx(lines, delimiter):
    for idx, line in enumerate(lines):
        tokens = split_line(line, delimiter)
        tokens_lower = [t.lower() for t in tokens]

        has_time = any(
            ("timestamp" in t) or (t in ["time", "time[s]", "time(s)", "time_seconds"])
            for t in tokens_lower
        )
        has_voltage = any(("u1" in t and "v" in t) or ("voltage" in t) for t in tokens_lower)
        has_current = any(("i1" in t and "a" in t) or ("current" in t) for t in tokens_lower)

        if has_time and has_voltage and has_current:
            return idx

    return None


def extract_metadata_from_lines(lines, delimiter):
    meta = {
        "source_date": "unknown_not_recorded",
        "start_time": "unknown_not_recorded",
        "logging_interval_s": np.nan,
    }

    for line in lines:
        tokens = split_line(line, delimiter)
        if len(tokens) < 2:
            continue

        key = tokens[0].strip().lower()
        val = tokens[1].strip()

        if key == "#date":
            meta["source_date"] = val
        elif key == "#start time":
            meta["start_time"] = val
        elif key == "#logging interval[s]":
            try:
                meta["logging_interval_s"] = float(val)
            except Exception:
                meta["logging_interval_s"] = np.nan

    return meta


def classify_csv_format(header_line_idx, lines):
    if header_line_idx is None:
        return "unresolved_no_header_detected"

    if header_line_idx == 0:
        return "processed_1Hz_aligned_or_header_only_csv"

    has_device_metadata = any(line.startswith("#Device") for line in lines[:15])
    has_log_marker = any("#Format" in line and "LOG" in line for line in lines[:15])

    if has_device_metadata and has_log_marker:
        return "NGU201_LOG_raw"

    return "metadata_prefixed_csv"


def classify_time_style(values):
    vals = [str(v).strip() for v in values if str(v).strip() and str(v).strip().lower() != "nan"]

    if not vals:
        return "unknown_no_valid_examples"

    if all(re.match(r"^\d+:\d{2}:\d{2}(\.\d+)?$", v) for v in vals[:5]):
        return "timestamp_string_hh_mm_ss"

    if all(re.match(r"^\d{1,2}:\d{2}(\.\d+)?$", v) for v in vals[:5]):
        return "timestamp_string_mm_ss_or_truncated_hour"

    if all(re.match(r"^-?\d+(\.\d+)?$", v) for v in vals[:5]):
        return "numeric_seconds_or_index"

    return "mixed_or_unrecognized_time_style"


def read_header_and_time_examples(path, delimiter, header_line_idx):
    if header_line_idx is None:
        return [], None, "unknown", []

    try:
        df_head = pd.read_csv(
            path,
            sep=delimiter,
            skiprows=header_line_idx,
            nrows=8,
            engine="python",
        )
    except Exception as exc:
        return [], None, f"read_error:{type(exc).__name__}", []

    cols = list(df_head.columns)

    time_col = None
    for c in cols:
        c_lower = str(c).lower()
        if "timestamp" in c_lower or c_lower in ["time", "time[s]", "time(s)", "time_seconds"]:
            time_col = c
            break

    if time_col is None:
        return cols, None, "unknown_no_time_column", []

    examples = df_head[time_col].dropna().astype(str).head(8).tolist()
    style = classify_time_style(examples)

    return cols, time_col, style, examples


# =============================================================================
# Scan raw directory
# =============================================================================

if not RAW_DIR_DAY23A.exists():
    raise FileNotFoundError(f"RAW_DIR_DAY23A does not exist: {RAW_DIR_DAY23A}")

found_files = sorted([p.name for p in RAW_DIR_DAY23A.glob("*.csv")])
missing_files = [f for f in EXPECTED_DAY23A_RAW_FILES if f not in found_files]
unexpected_files = [f for f in found_files if f not in EXPECTED_DAY23A_RAW_FILES]

print(f"\n[scan] RAW_DIR_DAY23A = {RAW_DIR_DAY23A}")
print(f"[scan] found CSV files = {len(found_files)}")
print(f"[scan] expected files missing = {missing_files}")
print(f"[scan] unexpected CSV files = {unexpected_files}")

if missing_files:
    raise FileNotFoundError(f"Missing expected Day23A raw files: {missing_files}")

if unexpected_files:
    print("[warning] Unexpected CSV files found. They will not be used unless added to EXPECTED_DAY23A_RAW_FILES.")


# =============================================================================
# Build format inventory
# =============================================================================

rows = []

for file_name in EXPECTED_DAY23A_RAW_FILES:
    path = RAW_DIR_DAY23A / file_name

    lines, encoding_used = read_text_lines_fallback(path, max_lines=80)
    delimiter = infer_delimiter_from_lines(lines)
    header_line_idx = find_header_line_idx(lines, delimiter)
    meta = extract_metadata_from_lines(lines, delimiter)
    csv_format = classify_csv_format(header_line_idx, lines)

    cols, time_col, time_style, time_examples = read_header_and_time_examples(
        path=path,
        delimiter=delimiter,
        header_line_idx=header_line_idx,
    )

    protocol = DAY23A_PROTOCOL_MAP[file_name].copy()

    notes = []
    notes.append(f"encoding={encoding_used}")
    notes.append(f"delimiter={repr(delimiter)}")
    notes.append(f"header_line_idx={header_line_idx}")
    notes.append(f"columns={cols}")
    notes.append("frequency_label=1tau_fixed_by_protocol")
    notes.append("temperature_summary_missing")

    row = {
        "file_name": file_name,
        "file_path": str(path),
        "csv_format_refined": csv_format,
        "encoding_used": encoding_used,
        "delimiter": delimiter,
        "header_line_idx": header_line_idx,
        "columns": "|".join(map(str, cols)),
        "time_column_name": time_col if time_col is not None else "unknown_not_detected",
        "time_column_style": time_style,
        "time_value_examples": "|".join(time_examples),
        "source_session_date": meta["source_date"],
        "start_time": meta["start_time"],
        "logging_interval_s": meta["logging_interval_s"],

        "protocol_label": protocol["protocol_label"],
        "protocol_role": protocol["protocol_role"],
        "DC_C": protocol["DC_C"],
        "AC_C": protocol["AC_C"],
        "m_tau": protocol["m_tau"],
        "tau_label_s": TAU_LABEL_S,
        "frequency_Hz": protocol["frequency_Hz"],
        "frequency_source": "fixed_1tau_protocol_metadata",
        "kappa": protocol["kappa"],
        "candidate_for_DC_reference": protocol["candidate_for_DC_reference"],
        "candidate_for_DCAC": protocol["candidate_for_DCAC"],

        "T_surface_max_C": np.nan,
        "T_surface_mean_C": np.nan,
        "temperature_sensor_type": "unknown_or_not_available",
        "temperature_alignment_method": "not_available",
        "temperature_data_status": "missing_temperature_summary",

        "notes": "; ".join(notes),
    }

    rows.append(row)

    print("\n" + "=" * 120)
    print(f"[FILE] {file_name}")
    print("=" * 120)
    print(f"csv_format_refined: {csv_format}")
    print(f"encoding_used: {encoding_used}")
    print(f"delimiter: {repr(delimiter)}")
    print(f"header_line_idx: {header_line_idx}")
    print(f"columns: {cols}")
    print(f"source_session_date: {meta['source_date']}")
    print(f"start_time: {meta['start_time']}")
    print(f"logging_interval_s: {meta['logging_interval_s']}")
    print(f"time_column_name: {time_col}")
    print(f"time_column_style: {time_style}")
    print(f"time_value_examples: {time_examples}")

    print("\n[first 30 lines]")
    for i, line in enumerate(lines[:30]):
        print(f"{i:03d}: {line}")

df_day23_format_inventory = pd.DataFrame(rows)
df_day23_format_inventory.to_csv(OUT_DAY23A_FORMAT_INVENTORY, index=False)

print("\n" + "=" * 120)
print("[SUMMARY]")
print("=" * 120)

display_cols = [
    "file_name",
    "protocol_label",
    "protocol_role",
    "DC_C",
    "AC_C",
    "kappa",
    "m_tau",
    "frequency_Hz",
    "csv_format_refined",
    "header_line_idx",
    "time_column_name",
    "time_column_style",
    "source_session_date",
    "start_time",
    "logging_interval_s",
    "temperature_data_status",
]

print(df_day23_format_inventory[display_cols].to_string(index=False))

print(f"\n[OK] Wrote Day23A raw CSV format inventory: {OUT_DAY23A_FORMAT_INVENTORY}")
print("[OK] Cell 0A completed. No trajectory processing was performed.")

[OK] Day23A minimal setup loaded.
[OK] REPO = /Users/louislu/pybamm-dcac-superimposed
[OK] DATA_DIR = /Users/louislu/pybamm-dcac-superimposed/data
[OK] RAW_DIR_DAY23A = /Users/louislu/pybamm-dcac-superimposed/data/raw_mj1_ngu201_day23A_0p4C_subDC
[OK] FREQ_1TAU_HZ = 0.014338283 Hz

[scan] RAW_DIR_DAY23A = /Users/louislu/pybamm-dcac-superimposed/data/raw_mj1_ngu201_day23A_0p4C_subDC
[scan] found CSV files = 4
[scan] expected files missing = []
[scan] unexpected CSV files = []

[FILE] 0.4C DC.csv
csv_format_refined: NGU201_LOG_raw
encoding_used: utf-8-sig
delimiter: ','
header_line_idx: 13
columns: ['Timestamp', 'U1[V]', 'I1[A]', 'P1[W]', 'DVM1[V]']
source_session_date: 2025-10-20
start_time: 09:41:02
logging_interval_s: 1.0
time_column_name: Timestamp
time_column_style: timestamp_string_hh_mm_ss
time_value_examples: ['09:41:02.5', '09:41:03.5', '09:41:04.5', '09:41:05.5', '09:41:06.5', '09:41:07.5', '09:41:08.5', '09:41:09.5']

[first 30 lines]
000: #Device,NGU201
001: #Device Name,RS-N

In [2]:
# Day23A Cell 0A.5 — Exclusion registry and active-file inventory
#
# Purpose:
# - Exclude incomplete raw records before downstream processing
# - Keep an explicit audit trail for excluded files
# - Generate an active format inventory for timebase / trajectory processing
#
# Explicitly NOT done here:
# - No trajectory trimming
# - No Q integration
# - No event detection
# - No segment assignment
# - No verdict

OUT_DAY23A_EXCLUSION_AUDIT = DATA_DIR / "day23A_step0A5_file_exclusion_audit.csv"
OUT_DAY23A_ACTIVE_FORMAT_INVENTORY = DATA_DIR / "day23A_step0A5_active_raw_csv_format_inventory.csv"

DAY23A_EXCLUDED_FILES = {
    "DC0.4C+AC0.2C f=0.0143Hz.csv": {
        "use_for_audit": False,
        "exclusion_status": "excluded_missing_CC_raw",
        "exclusion_reason": (
            "Raw record starts near 4.2 V / CV-like region and lacks the required "
            "pre-Vmax CC segment. It cannot support Q_Vmax_DCAC, Segment A/B/D "
            "assignment, Q80/Q90 first-passage, or Segment-A residual audit."
        ),
    }
}

df_day23_format_inventory = pd.read_csv(OUT_DAY23A_FORMAT_INVENTORY)

rows = []

for _, row in df_day23_format_inventory.iterrows():
    file_name = row["file_name"]

    if file_name in DAY23A_EXCLUDED_FILES:
        use_for_audit = False
        exclusion_status = DAY23A_EXCLUDED_FILES[file_name]["exclusion_status"]
        exclusion_reason = DAY23A_EXCLUDED_FILES[file_name]["exclusion_reason"]
    else:
        use_for_audit = True
        exclusion_status = "active"
        exclusion_reason = ""

    out = row.to_dict()
    out["use_for_audit"] = use_for_audit
    out["exclusion_status"] = exclusion_status
    out["exclusion_reason"] = exclusion_reason
    rows.append(out)

df_day23_exclusion = pd.DataFrame(rows)
df_day23_active_format_inventory = df_day23_exclusion[
    df_day23_exclusion["use_for_audit"].map(lambda x: bool(x))
].copy()

df_day23_exclusion.to_csv(OUT_DAY23A_EXCLUSION_AUDIT, index=False)
df_day23_active_format_inventory.to_csv(OUT_DAY23A_ACTIVE_FORMAT_INVENTORY, index=False)

print(f"[OK] Wrote Day23A exclusion audit: {OUT_DAY23A_EXCLUSION_AUDIT}")
print(f"[OK] Wrote Day23A active format inventory: {OUT_DAY23A_ACTIVE_FORMAT_INVENTORY}")

display_cols = [
    "file_name",
    "protocol_label",
    "protocol_role",
    "DC_C",
    "AC_C",
    "kappa",
    "frequency_Hz",
    "csv_format_refined",
    "use_for_audit",
    "exclusion_status",
]

print("\n[Day23A exclusion audit]")
print(df_day23_exclusion[display_cols].to_string(index=False))

print("\n[Day23A active files]")
print(df_day23_active_format_inventory[display_cols].to_string(index=False))

n_ref = int((df_day23_active_format_inventory["candidate_for_DC_reference"] == True).sum())
n_dcac = int((df_day23_active_format_inventory["candidate_for_DCAC"] == True).sum())

print(f"\n[OK] Active DC reference candidates = {n_ref}")
print(f"[OK] Active DCAC candidates = {n_dcac}")

if n_ref != 1:
    raise ValueError(f"Expected exactly one active DC reference, found {n_ref}")

if n_dcac < 1:
    raise ValueError("Expected at least one active DCAC file.")

if "DC0.4C+AC0.2C f=0.0143Hz.csv" in df_day23_active_format_inventory["file_name"].tolist():
    raise ValueError("Excluded 0.4C+0.2C file still appears in active inventory.")

print("[OK] Cell 0A.5 completed.")
print("[OK] Downstream Day23A processing must use OUT_DAY23A_ACTIVE_FORMAT_INVENTORY.")

[OK] Wrote Day23A exclusion audit: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step0A5_file_exclusion_audit.csv
[OK] Wrote Day23A active format inventory: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step0A5_active_raw_csv_format_inventory.csv

[Day23A exclusion audit]
                      file_name protocol_label protocol_role  DC_C  AC_C  kappa  frequency_Hz                       csv_format_refined  use_for_audit        exclusion_status
                    0.4C DC.csv        0.4C DC  DC_reference   0.4   0.0   0.00      0.000000                           NGU201_LOG_raw           True                  active
   DC0.4C+AC0.2C f=0.0143Hz.csv 0.4C+0.2C 1tau          DCAC   0.4   0.2   0.50      0.014338                           NGU201_LOG_raw          False excluded_missing_CC_raw
   DC0.4C+AC0.3C f=0.0143Hz.csv 0.4C+0.3C 1tau          DCAC   0.4   0.3   0.75      0.014338                           NGU201_LOG_raw           True                  active
处理后DC0.4C+AC0.1C f=

In [3]:
# Day23A Cell 0B — active-file timebase audit
#
# Purpose:
# - Audit timestamp parsing and monotonicity for active Day23A files only
# - Support both NGU201 LOG raw files and processed 1 Hz aligned files
# - Write a timebase audit table for downstream processing
#
# Explicitly NOT done here:
# - No charge-onset trimming
# - No Q integration
# - No event detection
# - No segment assignment
# - No verdict

OUT_DAY23A_TIMEBASE_AUDIT = DATA_DIR / "day23A_step0B_timebase_audit.csv"

if not OUT_DAY23A_ACTIVE_FORMAT_INVENTORY.exists():
    raise FileNotFoundError(
        f"Active format inventory missing: {OUT_DAY23A_ACTIVE_FORMAT_INVENTORY}\n"
        "Run Day23A Cell 0A.5 first."
    )

df_day23_active_format_inventory = pd.read_csv(OUT_DAY23A_ACTIVE_FORMAT_INVENTORY)

print(f"[OK] Loaded Day23A active format inventory: {OUT_DAY23A_ACTIVE_FORMAT_INVENTORY}")
print(f"[OK] active files = {len(df_day23_active_format_inventory)}")


# =============================================================================
# 0B.1 Helpers
# =============================================================================

def parse_timestamp_to_seconds_day23(value):
    """
    Parse timestamp into seconds.

    Supported:
    - HH:MM:SS(.s)
    - H:MM:SS(.s)
    - MM:SS(.s)
    - numeric seconds

    For strings with three fields, the first field is treated as hours.
    For strings with two fields, the first field is treated as minutes.
    """
    if pd.isna(value):
        return np.nan

    s = str(value).strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    try:
        if ":" not in s:
            return float(s)

        parts = s.split(":")
        parts = [float(p) for p in parts]

        if len(parts) == 3:
            h, m, sec = parts
            return h * 3600.0 + m * 60.0 + sec

        if len(parts) == 2:
            m, sec = parts
            return m * 60.0 + sec

        return np.nan

    except Exception:
        return np.nan


def unwrap_timestamp_day23(t_raw_s):
    """
    Unwrap timestamp rollovers.

    The active Day23A files are expected to be monotonic after parsing.
    This function still supports 3600 s and 86400 s rollovers for future files.
    """
    t = np.asarray(t_raw_s, dtype=float)
    out = np.full_like(t, np.nan, dtype=float)

    finite_idx = np.where(np.isfinite(t))[0]
    if len(finite_idx) == 0:
        return out, 0, np.nan

    finite_vals = t[finite_idx]
    max_val = np.nanmax(finite_vals)

    # If values are clearly clock seconds, use 24 h rollover.
    # If values are short/truncated, use 1 h rollover.
    rollover_period_s = 86400.0 if max_val > 3600.0 else 3600.0

    offset = 0.0
    unwrap_count = 0

    first_idx = finite_idx[0]
    out[first_idx] = t[first_idx]
    prev = out[first_idx]

    for idx in finite_idx[1:]:
        val = t[idx]

        if val + offset < prev - 10.0:
            offset += rollover_period_s
            unwrap_count += 1

        out[idx] = val + offset
        prev = out[idx]

    return out, unwrap_count, rollover_period_s


def read_active_csv_for_timebase_day23(row):
    path = Path(row["file_path"])
    delimiter = row["delimiter"]
    header_line_idx = int(row["header_line_idx"])

    df = pd.read_csv(
        path,
        sep=delimiter,
        skiprows=header_line_idx,
        engine="python",
    )

    # Remove fully empty unnamed / blank columns.
    keep_cols = []
    for c in df.columns:
        c_str = str(c).strip()
        if c_str.startswith("Unnamed") and df[c].isna().all():
            continue
        if c_str == "" and df[c].isna().all():
            continue
        keep_cols.append(c)

    df = df[keep_cols].copy()

    if "Timestamp" not in df.columns:
        raise ValueError(f"{path.name}: missing Timestamp column. columns={df.columns.tolist()}")

    df["t_raw_s"] = df["Timestamp"].map(parse_timestamp_to_seconds_day23)
    t_unwrapped, unwrap_count, rollover_period_s = unwrap_timestamp_day23(df["t_raw_s"].to_numpy())
    df["t_unwrapped_s"] = t_unwrapped

    finite_t = np.isfinite(df["t_unwrapped_s"].to_numpy())
    if finite_t.any():
        t0 = np.nanmin(df.loc[finite_t, "t_unwrapped_s"].to_numpy(dtype=float))
        df["t_s"] = df["t_unwrapped_s"] - t0
    else:
        df["t_s"] = np.nan

    return df, unwrap_count, rollover_period_s


def classify_timebase_day23(dt):
    arr = np.asarray(dt, dtype=float)
    arr = arr[np.isfinite(arr)]

    if len(arr) == 0:
        return "invalid_no_finite_dt"

    if np.any(arr <= 0):
        return "invalid_non_monotonic_or_duplicate_time"

    dt_median = float(np.median(arr))
    dt_min = float(np.min(arr))
    dt_max = float(np.max(arr))

    if abs(dt_median - 1.0) <= 0.05 and dt_min >= 0.5 and dt_max <= 5.0:
        return "timebase_ok_approximately_1Hz"

    if dt_min > 0:
        return "timebase_ok_nonstandard_sampling"

    return "invalid_timebase"


# =============================================================================
# 0B.2 Audit active files
# =============================================================================

rows = []

for _, row in df_day23_active_format_inventory.iterrows():
    file_name = row["file_name"]
    df_raw, unwrap_count, rollover_period_s = read_active_csv_for_timebase_day23(row)

    finite_t = df_raw["t_s"].notna()
    t = df_raw.loc[finite_t, "t_s"].to_numpy(dtype=float)

    if len(t) >= 2:
        dt = np.diff(t)
        dt_median = float(np.median(dt))
        dt_min = float(np.min(dt))
        dt_max = float(np.max(dt))
        duration_s = float(t[-1] - t[0])
        t_start_s = float(t[0])
        t_end_s = float(t[-1])
        time_status = classify_timebase_day23(dt)
    else:
        dt_median = np.nan
        dt_min = np.nan
        dt_max = np.nan
        duration_s = np.nan
        t_start_s = np.nan
        t_end_s = np.nan
        time_status = "invalid_insufficient_time_points"

    reconstructed = False

    notes = []
    notes.append(f"source_format={row['csv_format_refined']}")
    notes.append(f"source_date={row['source_session_date']}")
    notes.append(f"start_time={row['start_time']}")
    notes.append(f"excluded_files_not_processed=True")

    audit_row = {
        "file_name": file_name,
        "protocol_label": row["protocol_label"],
        "protocol_role": row["protocol_role"],
        "DC_C": row["DC_C"],
        "AC_C": row["AC_C"],
        "kappa": row["kappa"],
        "m_tau": row["m_tau"],
        "frequency_Hz": row["frequency_Hz"],
        "csv_format_refined": row["csv_format_refined"],
        "header_line_idx": int(row["header_line_idx"]),
        "time_column_name": row["time_column_name"],
        "n_rows": int(len(df_raw)),
        "n_finite_time_rows": int(finite_t.sum()),
        "time_parse_method": "parsed_timestamp_with_unwrap",
        "time_monotonic_status": time_status,
        "time_unwrap_count": int(unwrap_count),
        "time_rollover_period_s": rollover_period_s,
        "time_reconstructed_from_row_index": reconstructed,
        "dt_median_s": dt_median,
        "dt_min_s": dt_min,
        "dt_max_s": dt_max,
        "t_start_s": t_start_s,
        "t_end_s": t_end_s,
        "duration_s": duration_s,
        "source_session_date": row["source_session_date"],
        "start_time": row["start_time"],
        "temperature_data_status": row["temperature_data_status"],
        "notes": "; ".join(notes),
    }

    rows.append(audit_row)

df_day23_timebase_audit = pd.DataFrame(rows)
df_day23_timebase_audit.to_csv(OUT_DAY23A_TIMEBASE_AUDIT, index=False)

print(f"[OK] Wrote Day23A timebase audit: {OUT_DAY23A_TIMEBASE_AUDIT}")

display_cols = [
    "file_name",
    "protocol_label",
    "protocol_role",
    "csv_format_refined",
    "n_rows",
    "n_finite_time_rows",
    "time_parse_method",
    "time_monotonic_status",
    "time_unwrap_count",
    "time_rollover_period_s",
    "time_reconstructed_from_row_index",
    "dt_median_s",
    "dt_min_s",
    "dt_max_s",
    "t_start_s",
    "t_end_s",
    "duration_s",
    "temperature_data_status",
]

print(df_day23_timebase_audit[display_cols].to_string(index=False))


# =============================================================================
# 0B.3 Hard guards
# =============================================================================

bad = df_day23_timebase_audit[
    ~df_day23_timebase_audit["time_monotonic_status"].isin([
        "timebase_ok_approximately_1Hz",
        "timebase_ok_nonstandard_sampling",
    ])
]

if len(bad) > 0:
    print("\n[ERROR] Invalid Day23A timebase rows:")
    print(bad[display_cols].to_string(index=False))
    raise ValueError("Day23A timebase audit failed.")

if "DC0.4C+AC0.2C f=0.0143Hz.csv" in df_day23_timebase_audit["file_name"].tolist():
    raise ValueError("Excluded 0.4C+0.2C file appeared in timebase audit.")

print("[OK] Cell 0B Day23A timebase audit passed.")
print("[OK] No trajectory trimming, Q integration, event detection, segment assignment, or verdict performed.")

[OK] Loaded Day23A active format inventory: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step0A5_active_raw_csv_format_inventory.csv
[OK] active files = 3
[OK] Wrote Day23A timebase audit: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step0B_timebase_audit.csv
                      file_name protocol_label protocol_role                       csv_format_refined  n_rows  n_finite_time_rows            time_parse_method         time_monotonic_status  time_unwrap_count  time_rollover_period_s  time_reconstructed_from_row_index  dt_median_s  dt_min_s  dt_max_s  t_start_s  t_end_s  duration_s     temperature_data_status
                    0.4C DC.csv        0.4C DC  DC_reference                           NGU201_LOG_raw   11505               11505 parsed_timestamp_with_unwrap timebase_ok_approximately_1Hz                  0                 86400.0                              False          1.0       0.9       2.1        0.0  11504.5     11504.5 missing_temperature_summary
   DC0

In [4]:
# Day23A Cell 0C — AC-off / full-DCAC protocol-mode audit
#
# Purpose:
# - Check whether active Day23A DCAC files follow the Day21A/Day22A protocol:
#   AC applied only during CC, then switched off after first V >= 4.2 V.
# - Detect possible full-protocol DCAC continuation after Vmax.
#
# Method:
# - Detect first Vmax event: first finite U >= 4.2 V.
# - Fit fixed-frequency sinusoidal current component before Vmax and after Vmax.
# - Compare post-Vmax fitted AC amplitude to expected AC amplitude.
#
# Explicitly NOT done here:
# - No Q integration
# - No segment assignment
# - No Δt computation
# - No verdict

from pathlib import Path
import numpy as np
import pandas as pd

OUT_DAY23A_PROTOCOL_MODE_AUDIT = DATA_DIR / "day23A_step0C_protocol_mode_acoff_audit.csv"

if not OUT_DAY23A_ACTIVE_FORMAT_INVENTORY.exists():
    raise FileNotFoundError(
        f"Active format inventory missing: {OUT_DAY23A_ACTIVE_FORMAT_INVENTORY}\n"
        "Run Day23A Cell 0A.5 first."
    )

df_day23_active_format_inventory = pd.read_csv(OUT_DAY23A_ACTIVE_FORMAT_INVENTORY)

print(f"[OK] Loaded active Day23A inventory: {OUT_DAY23A_ACTIVE_FORMAT_INVENTORY}")
print(f"[OK] active rows = {len(df_day23_active_format_inventory)}")


# =============================================================================
# 0C.1 Helpers
# =============================================================================

def parse_timestamp_to_seconds_day23_protocol(value):
    if pd.isna(value):
        return np.nan

    s = str(value).strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    try:
        if ":" not in s:
            return float(s)

        parts = [float(p) for p in s.split(":")]

        if len(parts) == 3:
            h, m, sec = parts
            return h * 3600.0 + m * 60.0 + sec

        if len(parts) == 2:
            m, sec = parts
            return m * 60.0 + sec

        return np.nan

    except Exception:
        return np.nan


def unwrap_timestamp_day23_protocol(t_raw_s):
    t = np.asarray(t_raw_s, dtype=float)
    out = np.full_like(t, np.nan, dtype=float)

    finite_idx = np.where(np.isfinite(t))[0]
    if len(finite_idx) == 0:
        return out, 0, np.nan

    finite_vals = t[finite_idx]
    max_val = np.nanmax(finite_vals)
    rollover_period_s = 86400.0 if max_val > 3600.0 else 3600.0

    offset = 0.0
    unwrap_count = 0

    first_idx = finite_idx[0]
    out[first_idx] = t[first_idx]
    prev = out[first_idx]

    for idx in finite_idx[1:]:
        val = t[idx]

        if val + offset < prev - 10.0:
            offset += rollover_period_s
            unwrap_count += 1

        out[idx] = val + offset
        prev = out[idx]

    return out, unwrap_count, rollover_period_s


def load_active_csv_day23_protocol(row):
    path = Path(row["file_path"])
    delimiter = row["delimiter"]
    header_line_idx = int(row["header_line_idx"])

    df = pd.read_csv(
        path,
        sep=delimiter,
        skiprows=header_line_idx,
        engine="python",
    )

    keep_cols = []
    for c in df.columns:
        c_str = str(c).strip()
        if c_str.startswith("Unnamed") and df[c].isna().all():
            continue
        if c_str == "" and df[c].isna().all():
            continue
        keep_cols.append(c)

    df = df[keep_cols].copy()

    required = ["Timestamp", "U1[V]", "I1[A]"]
    for c in required:
        if c not in df.columns:
            raise ValueError(f"{path.name}: missing required column {c}. columns={df.columns.tolist()}")

    df["t_raw_s"] = df["Timestamp"].map(parse_timestamp_to_seconds_day23_protocol)
    t_unwrapped, unwrap_count, rollover_period_s = unwrap_timestamp_day23_protocol(
        df["t_raw_s"].to_numpy()
    )
    df["t_unwrapped_s"] = t_unwrapped

    finite_t = np.isfinite(df["t_unwrapped_s"].to_numpy(dtype=float))
    if finite_t.any():
        t0 = np.nanmin(df.loc[finite_t, "t_unwrapped_s"].to_numpy(dtype=float))
        df["t_s"] = df["t_unwrapped_s"] - t0
    else:
        df["t_s"] = np.nan

    df["U_V"] = pd.to_numeric(df["U1[V]"], errors="coerce")
    df["I_A"] = pd.to_numeric(df["I1[A]"], errors="coerce")

    return df


def detect_first_vmax_day23_protocol(df, vmax_v=4.2):
    finite = df["t_s"].notna() & df["U_V"].notna()
    d = df.loc[finite].copy()

    if len(d) == 0:
        return {
            "t_Vmax_s": np.nan,
            "U_at_Vmax_V": np.nan,
            "idx_Vmax": np.nan,
            "vmax_detected": False,
        }

    hit = d[d["U_V"] >= vmax_v]
    if len(hit) == 0:
        return {
            "t_Vmax_s": np.nan,
            "U_at_Vmax_V": np.nan,
            "idx_Vmax": np.nan,
            "vmax_detected": False,
        }

    first = hit.iloc[0]

    return {
        "t_Vmax_s": float(first["t_s"]),
        "U_at_Vmax_V": float(first["U_V"]),
        "idx_Vmax": int(first.name),
        "vmax_detected": True,
    }


def fit_fixed_frequency_current_component(t_s, i_a, frequency_hz):
    """
    Fit:
        I(t) = c0 + c1*t_rel + a*sin(2πft) + b*cos(2πft)

    Returns fitted sinusoidal amplitude sqrt(a^2 + b^2).
    Linear trend term reduces false AC detection in CV current decay.
    """
    t = np.asarray(t_s, dtype=float)
    i = np.asarray(i_a, dtype=float)

    finite = np.isfinite(t) & np.isfinite(i)
    t = t[finite]
    i = i[finite]

    if len(t) < 20:
        return {
            "fit_status": "insufficient_samples",
            "n": int(len(t)),
            "amp_fit_A": np.nan,
            "offset_fit_A": np.nan,
            "slope_fit_A_per_s": np.nan,
            "phase_fit_rad": np.nan,
            "rmse_A": np.nan,
            "i_range_A": np.nan,
            "i_std_A": np.nan,
            "window_duration_s": np.nan,
        }

    if not np.isfinite(frequency_hz) or frequency_hz <= 0:
        return {
            "fit_status": "invalid_frequency",
            "n": int(len(t)),
            "amp_fit_A": np.nan,
            "offset_fit_A": np.nan,
            "slope_fit_A_per_s": np.nan,
            "phase_fit_rad": np.nan,
            "rmse_A": np.nan,
            "i_range_A": float(np.nanmax(i) - np.nanmin(i)),
            "i_std_A": float(np.nanstd(i)),
            "window_duration_s": float(np.nanmax(t) - np.nanmin(t)),
        }

    t_rel = t - t[0]
    omega = 2.0 * np.pi * float(frequency_hz)

    X = np.column_stack([
        np.ones_like(t_rel),
        t_rel,
        np.sin(omega * t_rel),
        np.cos(omega * t_rel),
    ])

    try:
        beta, *_ = np.linalg.lstsq(X, i, rcond=None)
        pred = X @ beta

        c0, c1, a_sin, b_cos = beta
        amp = float(np.sqrt(a_sin**2 + b_cos**2))
        phase = float(np.arctan2(b_cos, a_sin))
        rmse = float(np.sqrt(np.mean((i - pred) ** 2)))

        return {
            "fit_status": "ok",
            "n": int(len(t)),
            "amp_fit_A": amp,
            "offset_fit_A": float(c0),
            "slope_fit_A_per_s": float(c1),
            "phase_fit_rad": phase,
            "rmse_A": rmse,
            "i_range_A": float(np.nanmax(i) - np.nanmin(i)),
            "i_std_A": float(np.nanstd(i)),
            "window_duration_s": float(np.nanmax(t) - np.nanmin(t)),
        }

    except Exception as exc:
        return {
            "fit_status": f"fit_error:{type(exc).__name__}",
            "n": int(len(t)),
            "amp_fit_A": np.nan,
            "offset_fit_A": np.nan,
            "slope_fit_A_per_s": np.nan,
            "phase_fit_rad": np.nan,
            "rmse_A": np.nan,
            "i_range_A": float(np.nanmax(i) - np.nanmin(i)),
            "i_std_A": float(np.nanstd(i)),
            "window_duration_s": float(np.nanmax(t) - np.nanmin(t)),
        }


def window_data(df, t_lo, t_hi):
    return df[
        df["t_s"].notna()
        & df["I_A"].notna()
        & (df["t_s"] >= float(t_lo))
        & (df["t_s"] <= float(t_hi))
    ].copy()


def classify_acoff_day23(
    expected_ac_A,
    pre_amp_A,
    post_amp_A,
    post_late_amp_A,
    pre_fit_status,
    post_fit_status,
):
    """
    Conservative classification:
    - If post-Vmax fixed-frequency amplitude is small relative to expected AC,
      classify as AC-off.
    - If it remains large, classify as possible full-DCAC continuation.
    """
    if expected_ac_A <= 0:
        return "not_applicable_DC_reference"

    if pre_fit_status != "ok":
        return "unresolved_pre_Vmax_waveform_fit_failed"

    if post_fit_status != "ok":
        return "unresolved_post_Vmax_waveform_fit_failed"

    if not np.isfinite(post_amp_A):
        return "unresolved_post_Vmax_amp_nan"

    post_ratio = post_amp_A / expected_ac_A
    late_ratio = post_late_amp_A / expected_ac_A if np.isfinite(post_late_amp_A) else np.nan

    # Strong AC-off: post periodic component nearly gone.
    if post_ratio <= 0.15 and (not np.isfinite(late_ratio) or late_ratio <= 0.15):
        return "AC_off_after_Vmax_supported"

    # Strong full continuation: periodic component remains close to expected AC.
    if post_ratio >= 0.50 or (np.isfinite(late_ratio) and late_ratio >= 0.50):
        return "possible_full_DCAC_after_Vmax"

    return "ambiguous_post_Vmax_AC_component"


# =============================================================================
# 0C.2 Audit active files
# =============================================================================

rows = []

for _, row in df_day23_active_format_inventory.iterrows():
    file_name = row["file_name"]
    protocol_label = row["protocol_label"]
    protocol_role = row["protocol_role"]

    df = load_active_csv_day23_protocol(row)

    finite_vi = df["t_s"].notna() & df["U_V"].notna() & df["I_A"].notna()
    dfin = df.loc[finite_vi].copy()

    expected_dc_A = float(row["DC_C"]) * ONE_C_A
    expected_ac_A = float(row["AC_C"]) * ONE_C_A
    frequency_hz = float(row["frequency_Hz"])

    vmax = detect_first_vmax_day23_protocol(df)

    if protocol_role == "DC_reference":
        audit_status = "not_applicable_DC_reference"
        t_vmax = vmax["t_Vmax_s"]
        pre_fit = {
            "fit_status": "not_applicable",
            "n": np.nan,
            "amp_fit_A": np.nan,
            "offset_fit_A": np.nan,
            "slope_fit_A_per_s": np.nan,
            "phase_fit_rad": np.nan,
            "rmse_A": np.nan,
            "i_range_A": np.nan,
            "i_std_A": np.nan,
            "window_duration_s": np.nan,
        }
        post_fit = pre_fit.copy()
        post_late_fit = pre_fit.copy()
        pre_window = (np.nan, np.nan)
        post_window = (np.nan, np.nan)
        post_late_window = (np.nan, np.nan)

    else:
        if not vmax["vmax_detected"]:
            audit_status = "unresolved_no_Vmax_detected"
            t_vmax = np.nan
            pre_fit = post_fit = post_late_fit = {
                "fit_status": "unresolved_no_Vmax_detected",
                "n": np.nan,
                "amp_fit_A": np.nan,
                "offset_fit_A": np.nan,
                "slope_fit_A_per_s": np.nan,
                "phase_fit_rad": np.nan,
                "rmse_A": np.nan,
                "i_range_A": np.nan,
                "i_std_A": np.nan,
                "window_duration_s": np.nan,
            }
            pre_window = post_window = post_late_window = (np.nan, np.nan)

        else:
            t_vmax = float(vmax["t_Vmax_s"])
            T_AC_s = 1.0 / frequency_hz if frequency_hz > 0 else np.nan

            # Use up to five periods before Vmax, excluding the final 5 s.
            pre_len_s = min(5.0 * T_AC_s, max(0.0, t_vmax - 10.0))
            pre_lo = max(0.0, t_vmax - pre_len_s)
            pre_hi = max(0.0, t_vmax - 5.0)

            # Immediate post-Vmax window: three periods after Vmax.
            post_lo = t_vmax
            post_hi = min(float(dfin["t_s"].max()), t_vmax + 3.0 * T_AC_s)

            # Late post-Vmax window: later CV region, if available.
            post_late_lo = min(float(dfin["t_s"].max()), t_vmax + 3.0 * T_AC_s)
            post_late_hi = min(float(dfin["t_s"].max()), t_vmax + 6.0 * T_AC_s)

            pre_df = window_data(df, pre_lo, pre_hi)
            post_df = window_data(df, post_lo, post_hi)
            post_late_df = window_data(df, post_late_lo, post_late_hi)

            pre_fit = fit_fixed_frequency_current_component(
                pre_df["t_s"].to_numpy(),
                pre_df["I_A"].to_numpy(),
                frequency_hz,
            )
            post_fit = fit_fixed_frequency_current_component(
                post_df["t_s"].to_numpy(),
                post_df["I_A"].to_numpy(),
                frequency_hz,
            )
            post_late_fit = fit_fixed_frequency_current_component(
                post_late_df["t_s"].to_numpy(),
                post_late_df["I_A"].to_numpy(),
                frequency_hz,
            )

            audit_status = classify_acoff_day23(
                expected_ac_A=expected_ac_A,
                pre_amp_A=pre_fit["amp_fit_A"],
                post_amp_A=post_fit["amp_fit_A"],
                post_late_amp_A=post_late_fit["amp_fit_A"],
                pre_fit_status=pre_fit["fit_status"],
                post_fit_status=post_fit["fit_status"],
            )

            pre_window = (pre_lo, pre_hi)
            post_window = (post_lo, post_hi)
            post_late_window = (post_late_lo, post_late_hi)

    row_out = {
        "file_name": file_name,
        "protocol_label": protocol_label,
        "protocol_role": protocol_role,
        "DC_C": float(row["DC_C"]),
        "AC_C": float(row["AC_C"]),
        "kappa": float(row["kappa"]),
        "frequency_Hz": frequency_hz,
        "expected_DC_A": expected_dc_A,
        "expected_AC_A": expected_ac_A,

        "vmax_detected": bool(vmax["vmax_detected"]),
        "t_Vmax_s": t_vmax,
        "U_at_Vmax_V": vmax["U_at_Vmax_V"],

        "pre_window_lo_s": pre_window[0],
        "pre_window_hi_s": pre_window[1],
        "pre_fit_status": pre_fit["fit_status"],
        "pre_n": pre_fit["n"],
        "pre_amp_fit_A": pre_fit["amp_fit_A"],
        "pre_amp_ratio_to_expected": (
            pre_fit["amp_fit_A"] / expected_ac_A
            if expected_ac_A > 0 and np.isfinite(pre_fit["amp_fit_A"])
            else np.nan
        ),
        "pre_i_range_A": pre_fit["i_range_A"],
        "pre_i_std_A": pre_fit["i_std_A"],
        "pre_rmse_A": pre_fit["rmse_A"],

        "post_window_lo_s": post_window[0],
        "post_window_hi_s": post_window[1],
        "post_fit_status": post_fit["fit_status"],
        "post_n": post_fit["n"],
        "post_amp_fit_A": post_fit["amp_fit_A"],
        "post_amp_ratio_to_expected": (
            post_fit["amp_fit_A"] / expected_ac_A
            if expected_ac_A > 0 and np.isfinite(post_fit["amp_fit_A"])
            else np.nan
        ),
        "post_i_range_A": post_fit["i_range_A"],
        "post_i_std_A": post_fit["i_std_A"],
        "post_rmse_A": post_fit["rmse_A"],

        "post_late_window_lo_s": post_late_window[0],
        "post_late_window_hi_s": post_late_window[1],
        "post_late_fit_status": post_late_fit["fit_status"],
        "post_late_n": post_late_fit["n"],
        "post_late_amp_fit_A": post_late_fit["amp_fit_A"],
        "post_late_amp_ratio_to_expected": (
            post_late_fit["amp_fit_A"] / expected_ac_A
            if expected_ac_A > 0 and np.isfinite(post_late_fit["amp_fit_A"])
            else np.nan
        ),
        "post_late_i_range_A": post_late_fit["i_range_A"],
        "post_late_i_std_A": post_late_fit["i_std_A"],
        "post_late_rmse_A": post_late_fit["rmse_A"],

        "protocol_mode_status": audit_status,
        "interpretation": (
            "DC reference; AC-off audit not applicable."
            if protocol_role == "DC_reference"
            else (
                "Post-Vmax fixed-frequency current component is strongly suppressed; "
                "Day21A/Day22A AC-off segmentation assumption is supported."
                if audit_status == "AC_off_after_Vmax_supported"
                else (
                    "Post-Vmax fixed-frequency current component remains large; "
                    "this file may represent full-protocol DCAC or continued modulation after Vmax."
                    if audit_status == "possible_full_DCAC_after_Vmax"
                    else "Post-Vmax AC-off status is ambiguous and requires manual inspection."
                )
            )
        ),
    }

    rows.append(row_out)

df_day23_protocol_mode = pd.DataFrame(rows)
df_day23_protocol_mode.to_csv(OUT_DAY23A_PROTOCOL_MODE_AUDIT, index=False)

print(f"[OK] Wrote Day23A protocol-mode audit: {OUT_DAY23A_PROTOCOL_MODE_AUDIT}")

display_cols = [
    "file_name",
    "protocol_label",
    "protocol_role",
    "t_Vmax_s",
    "U_at_Vmax_V",
    "expected_AC_A",
    "pre_amp_fit_A",
    "pre_amp_ratio_to_expected",
    "post_amp_fit_A",
    "post_amp_ratio_to_expected",
    "post_late_amp_fit_A",
    "post_late_amp_ratio_to_expected",
    "protocol_mode_status",
]

print(df_day23_protocol_mode[display_cols].to_string(index=False))


# =============================================================================
# 0C.3 Hard guard
# =============================================================================

dcac_rows = df_day23_protocol_mode[df_day23_protocol_mode["protocol_role"] == "DCAC"]

bad = dcac_rows[
    dcac_rows["protocol_mode_status"].isin([
        "possible_full_DCAC_after_Vmax",
        "ambiguous_post_Vmax_AC_component",
        "unresolved_pre_Vmax_waveform_fit_failed",
        "unresolved_post_Vmax_waveform_fit_failed",
        "unresolved_no_Vmax_detected",
        "unresolved_post_Vmax_amp_nan",
    ])
]

if len(bad) > 0:
    print("\n[WARNING] Some active Day23A DCAC files do not clearly support AC-off after Vmax:")
    print(bad[display_cols].to_string(index=False))
    print("\n[STOP] Do not proceed with Day21A/Day22A segmentation assumptions until this is resolved.")
    raise ValueError("Day23A protocol-mode audit did not fully support AC-off-after-Vmax assumption.")

print("[OK] Cell 0C protocol-mode audit passed.")
print("[OK] Active DCAC files support AC-off after Vmax under fixed-frequency current-component audit.")
print("[OK] It is safe to proceed with Day21A/Day22A Segment A/B/D framework.")

[OK] Loaded active Day23A inventory: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step0A5_active_raw_csv_format_inventory.csv
[OK] active rows = 3
[OK] Wrote Day23A protocol-mode audit: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step0C_protocol_mode_acoff_audit.csv
                      file_name protocol_label protocol_role  t_Vmax_s  U_at_Vmax_V  expected_AC_A  pre_amp_fit_A  pre_amp_ratio_to_expected  post_amp_fit_A  post_amp_ratio_to_expected  post_late_amp_fit_A  post_late_amp_ratio_to_expected          protocol_mode_status
                    0.4C DC.csv        0.4C DC  DC_reference    7688.6     4.200031           0.00            NaN                        NaN             NaN                         NaN                  NaN                              NaN   not_applicable_DC_reference
   DC0.4C+AC0.3C f=0.0143Hz.csv 0.4C+0.3C 1tau          DCAC   10288.0     4.201665           1.02       1.018700                   0.998725        0.896275                    0.87

ValueError: Day23A protocol-mode audit did not fully support AC-off-after-Vmax assumption.

In [5]:
# Day23A Cell 0D — protocol-mode decision and framework switch
#
# Purpose:
# - Convert the Cell 0C STOP into an explicit audit decision
# - Disable Day21A/Day22A AC-off segmentation assumptions for Day23A
# - Activate generalized boundary-ordering framework
#
# Explicitly NOT done here:
# - No trajectory trimming
# - No Q integration
# - No event detection beyond reading Cell 0C output
# - No Δt computation
# - No verdict

OUT_DAY23A_FRAMEWORK_DECISION = DATA_DIR / "day23A_step0D_framework_decision.csv"

if not OUT_DAY23A_PROTOCOL_MODE_AUDIT.exists():
    raise FileNotFoundError(
        f"Protocol-mode audit missing: {OUT_DAY23A_PROTOCOL_MODE_AUDIT}\n"
        "Run Day23A Cell 0C first."
    )

df_day23_protocol_mode = pd.read_csv(OUT_DAY23A_PROTOCOL_MODE_AUDIT)

dcac_rows = df_day23_protocol_mode[df_day23_protocol_mode["protocol_role"] == "DCAC"].copy()

n_dcac = len(dcac_rows)
n_acoff_supported = int((dcac_rows["protocol_mode_status"] == "AC_off_after_Vmax_supported").sum())
n_possible_full = int((dcac_rows["protocol_mode_status"] == "possible_full_DCAC_after_Vmax").sum())
n_ambiguous = int(dcac_rows["protocol_mode_status"].str.contains("ambiguous|unresolved", regex=True).sum())

if n_dcac == 0:
    raise ValueError("No active DCAC rows found in protocol-mode audit.")

if n_acoff_supported == n_dcac:
    day23a_framework_mode = "day21A_day22A_AC_off_segmentation_allowed"
    generalized_boundary_framework_required = False
    downstream_segmentation_rule = "use_original_A_B_D"
    decision_reason = (
        "All active DCAC files support AC-off after Vmax. "
        "Day21A/Day22A Segment A/B/D framework can be reused."
    )
else:
    day23a_framework_mode = "generalized_boundary_ordering_required"
    generalized_boundary_framework_required = True
    downstream_segmentation_rule = "use_G0_G1_G2_boundary_ordering"
    decision_reason = (
        "At least one active DCAC file retains substantial fixed-frequency current "
        "component after Vmax. Day21A/Day22A AC-off Segment B assumption is not valid. "
        "Use generalized boundary-ordering framework instead."
    )

decision_row = {
    "group_id": DAY23A_GROUP_ID,
    "notebook": DAY23A_NOTEBOOK_NAME,
    "n_active_DCAC": n_dcac,
    "n_AC_off_after_Vmax_supported": n_acoff_supported,
    "n_possible_full_DCAC_after_Vmax": n_possible_full,
    "n_ambiguous_or_unresolved": n_ambiguous,
    "day23a_framework_mode": day23a_framework_mode,
    "generalized_boundary_framework_required": generalized_boundary_framework_required,
    "downstream_segmentation_rule": downstream_segmentation_rule,
    "original_A_B_D_framework_allowed": not generalized_boundary_framework_required,
    "decision_reason": decision_reason,
}

df_day23_framework_decision = pd.DataFrame([decision_row])
df_day23_framework_decision.to_csv(OUT_DAY23A_FRAMEWORK_DECISION, index=False)

print(f"[OK] Wrote Day23A framework decision: {OUT_DAY23A_FRAMEWORK_DECISION}")
print(df_day23_framework_decision.to_string(index=False))

print("\n[Protocol-mode rows]")
display_cols = [
    "file_name",
    "protocol_label",
    "protocol_role",
    "t_Vmax_s",
    "expected_AC_A",
    "pre_amp_ratio_to_expected",
    "post_amp_ratio_to_expected",
    "post_late_amp_ratio_to_expected",
    "protocol_mode_status",
]
print(df_day23_protocol_mode[display_cols].to_string(index=False))

if generalized_boundary_framework_required:
    print("\n[DECISION]")
    print("Day23A must NOT use Day21A/Day22A AC-off Segment A/B/D framework.")
    print("Use generalized G0/G1/G2 boundary-ordering framework.")
else:
    print("\n[DECISION]")
    print("Day23A can reuse Day21A/Day22A Segment A/B/D framework.")

print("[OK] Cell 0D completed.")

[OK] Wrote Day23A framework decision: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step0D_framework_decision.csv
                  group_id                                  notebook  n_active_DCAC  n_AC_off_after_Vmax_supported  n_possible_full_DCAC_after_Vmax  n_ambiguous_or_unresolved                  day23a_framework_mode  generalized_boundary_framework_required   downstream_segmentation_rule  original_A_B_D_framework_allowed                                                                                                                                                                                              decision_reason
Day23A_MJ1_0p4C_subDC_1tau 27_day23A_MJ1_0p4C_subDC_1tau_audit.ipynb              2                              0                                2                          0 generalized_boundary_ordering_required                                     True use_G0_G1_G2_boundary_ordering                             False At least one active DCAC file retai

In [6]:
# Day23A Cell 1A — generalized boundary-ordering contract freeze
#
# Purpose:
# - Freeze Day23A as a generalized boundary-ordering audit
# - Record that original Day21A/Day22A AC-off Segment A/B/D framework is disabled
# - Freeze constants, output paths, active/excluded protocols, and interpretation caveats
#
# Explicitly NOT done here:
# - No raw trajectory loading
# - No Q integration
# - No event detection
# - No generalized G0/G1/G2 assignment
# - No Δt computation
# - No verdict

import json
import subprocess
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd

# =============================================================================
# 1A.1 Required previous outputs
# =============================================================================

OUT_DAY23A_AUDIT_CONTRACT_JSON = DATA_DIR / "day23A_audit_contract_schema_thresholds.json"

required_previous_day23A_files = [
    OUT_DAY23A_FORMAT_INVENTORY,
    OUT_DAY23A_ACTIVE_FORMAT_INVENTORY,
    OUT_DAY23A_EXCLUSION_AUDIT,
    OUT_DAY23A_TIMEBASE_AUDIT,
    OUT_DAY23A_PROTOCOL_MODE_AUDIT,
    OUT_DAY23A_FRAMEWORK_DECISION,
]

missing_previous = [p for p in required_previous_day23A_files if not Path(p).exists()]
if missing_previous:
    raise FileNotFoundError(
        "Cannot freeze Day23A contract. Missing previous audit files:\n"
        + "\n".join(str(p) for p in missing_previous)
    )

df_day23_framework_decision = pd.read_csv(OUT_DAY23A_FRAMEWORK_DECISION)
framework_mode = str(df_day23_framework_decision["day23a_framework_mode"].iloc[0])
segmentation_rule = str(df_day23_framework_decision["downstream_segmentation_rule"].iloc[0])
original_abd_allowed = bool(df_day23_framework_decision["original_A_B_D_framework_allowed"].iloc[0])

if framework_mode != "generalized_boundary_ordering_required":
    raise ValueError(
        f"Unexpected Day23A framework mode: {framework_mode}. "
        "This Cell 1A is designed for generalized boundary ordering only."
    )

if original_abd_allowed:
    raise ValueError("Original Day21A/Day22A A/B/D framework is unexpectedly allowed.")

print("[OK] Previous Day23A framework decision loaded.")
print(f"[OK] framework_mode = {framework_mode}")
print(f"[OK] segmentation_rule = {segmentation_rule}")
print(f"[OK] original_A_B_D_framework_allowed = {original_abd_allowed}")


# =============================================================================
# 1A.2 Basic constants
# =============================================================================

DAY23A_GROUP_ID = "Day23A_MJ1_0p4C_subDC_1tau"
DAY23A_NOTEBOOK_NAME = "27_day23A_MJ1_0p4C_subDC_1tau_audit.ipynb"

CELL_ID = "LG_INR18650_MJ1"
SOURCE_TYPE_MJ1 = "experimental_MJ1"

ONE_C_A = 3.4
Q_NOM_AH = 3.4

TAU_LABEL_S = 11.1
FREQ_1TAU_HZ = 1.0 / (2.0 * np.pi * TAU_LABEL_S)

Q80_NOMINAL_FRACTION_OF_Q_NOM = 0.80
Q90_NOMINAL_FRACTION_OF_Q_NOM = 0.90

Q80_NOMINAL_AH = Q80_NOMINAL_FRACTION_OF_Q_NOM * Q_NOM_AH
Q90_NOMINAL_AH = Q90_NOMINAL_FRACTION_OF_Q_NOM * Q_NOM_AH

VMAX_V = 4.2
ICUTOFF_A = 0.050

Q_GRID_STEP_AH = 0.010
Q_GRID_MIN_COUNT_G0 = 30

# G0 residual lower bound, inherited from Day21A / Day22A Segment-A convention.
G0_Q_LO_AH = 0.050

# Boundary degeneracy tolerance
Q_BOUNDARY_DEGENERATE_TOLERANCE_AH = 0.001

# Residual thresholds, inherited but now applied only to G0.
G0_RESID_FLOOR_COMPATIBLE_THRESHOLD_S = 2.70
G0_RESID_REOPEN_THRESHOLD_S = 6.75

# Post-boundary preservation threshold, only descriptive in Day23A.
POST_BOUNDARY_PRESERVATION_THRESHOLD_S = 2.70

# No repeat-based experimental noise floor.
DAY23A_NO_REPEAT_NOISE_FLOOR = True
DAY23A_NOISE_FLOOR_CAVEAT = "no_independent_repeat_based_experimental_noise_floor"
DAY23A_EFFECT_SIZE_LIMITATION = "effect_size_interpreted_relative_to_audit_resolution_not_strict_disappearance"

# Temperature missing
DAY23A_TEMPERATURE_STATUS = "missing_temperature_summary"
DAY23A_TEMPERATURE_CAVEAT = "temperature_summary_missing"
DAY23A_THERMAL_ATTRIBUTION_ALLOWED = False

# Protocol-mode caveat
DAY23A_CONTINUED_AC_CAVEAT = "continued_AC_after_Vmax"
DAY23A_NOT_SAME_PROTOCOL_FAMILY_CAVEAT = "not_same_protocol_family_as_Day21A_Day22A"


# =============================================================================
# 1A.3 Generalized G0/G1/G2 region labels
# =============================================================================

REGION_G0 = "G0_shared_pre_boundary"
REGION_G1 = "G1_boundary_ordering_split"
REGION_G2 = "G2_post_boundary"
REGION_OUTSIDE = "outside_common_Q_window"
REGION_UNRESOLVED = "region_unresolved"

BOUNDARY_ORDER_DCAC_FIRST = "DCAC_first"
BOUNDARY_ORDER_DC_FIRST = "DC_first"
BOUNDARY_ORDER_DEGENERATE = "boundary_degenerate"
BOUNDARY_ORDER_UNRESOLVED = "boundary_order_unresolved"

PROTOCOL_MODE_AC_OFF_SUPPORTED = "AC_off_after_Vmax_supported"
PROTOCOL_MODE_CONTINUED_AC = "possible_full_DCAC_after_Vmax"
PROTOCOL_MODE_AMBIGUOUS = "ambiguous_post_Vmax_AC_component"
PROTOCOL_MODE_UNRESOLVED = "protocol_mode_unresolved"

GEOM_PHASE_VERIFIED = "verified"
GEOM_PHASE_ESTIMATED = "estimated_from_current_waveform"
GEOM_PHASE_UNRESOLVED = "geometry_phase_unresolved"

FINAL_Q_CONSISTENT = "final_Q_consistent"
FINAL_Q_MISMATCH_WARNING = "final_Q_mismatch_warning"
FINAL_Q_UNRESOLVED = "final_Q_unresolved"

G0_RESID_FLOOR_COMPATIBLE = "floor_compatible"
G0_RESID_INTERMEDIATE = "intermediate_between_floor_and_reopen_threshold"
G0_RESID_SPIKE = "spike_or_transition_artifact"
G0_RESID_ABOVE_FLOOR = "above_floor"
G0_RESID_UNRESOLVED = "G0_residual_unresolved"

POST_BOUNDARY_SATISFIED = "satisfied"
POST_BOUNDARY_NOT_SATISFIED = "not_satisfied"
POST_BOUNDARY_NOT_REQUIRED = "not_required"
POST_BOUNDARY_UNRESOLVED = "unresolved"


# =============================================================================
# 1A.4 Active / excluded protocol registry
# =============================================================================

DAY23A_ACTIVE_FILES = [
    "0.4C DC.csv",
    "处理后DC0.4C+AC0.1C f=0.0143Hz.csv",
    "DC0.4C+AC0.3C f=0.0143Hz.csv",
]

DAY23A_EXCLUDED_FILES_LOCKED = [
    "DC0.4C+AC0.2C f=0.0143Hz.csv",
]

DAY23A_ACTIVE_PROTOCOLS = [
    {
        "file_name": "0.4C DC.csv",
        "protocol_label": "0.4C DC",
        "protocol_role": "DC_reference",
        "DC_C": 0.4,
        "AC_C": 0.0,
        "kappa": 0.0,
        "m_tau": np.nan,
        "frequency_Hz": 0.0,
    },
    {
        "file_name": "处理后DC0.4C+AC0.1C f=0.0143Hz.csv",
        "protocol_label": "0.4C+0.1C 1tau",
        "protocol_role": "DCAC",
        "DC_C": 0.4,
        "AC_C": 0.1,
        "kappa": 0.25,
        "m_tau": 1.0,
        "frequency_Hz": FREQ_1TAU_HZ,
    },
    {
        "file_name": "DC0.4C+AC0.3C f=0.0143Hz.csv",
        "protocol_label": "0.4C+0.3C 1tau",
        "protocol_role": "DCAC",
        "DC_C": 0.4,
        "AC_C": 0.3,
        "kappa": 0.75,
        "m_tau": 1.0,
        "frequency_Hz": FREQ_1TAU_HZ,
    },
]


# =============================================================================
# 1A.5 Output paths
# =============================================================================

OUT_DAY23A_FILE_INVENTORY = DATA_DIR / "day23A_step0_MJ1_0p4C_subDC_file_inventory.csv"
OUT_DAY23A_LOAD_SUMMARY = DATA_DIR / "day23A_step1_MJ1_0p4C_subDC_loaded_trajectory_sanity.csv"
OUT_DAY23A_EVENT_AUDIT = DATA_DIR / "day23A_step2_MJ1_0p4C_subDC_event_protocol_boundary_audit.csv"
OUT_DAY23A_Q_SUMMARY = DATA_DIR / "day23A_step3_MJ1_0p4C_subDC_Q_integration_summary.csv"
OUT_DAY23A_FINALQ_PAIR_AUDIT = DATA_DIR / "day23A_step3_MJ1_0p4C_subDC_finalQ_pair_audit.csv"

OUT_DAY23A_RESOLUTION_LONG = DATA_DIR / "day23A_step3A_MJ1_0p4C_subDC_resolution_floor_long.csv"
OUT_DAY23A_RESOLUTION_SUMMARY = DATA_DIR / "day23A_step3A_MJ1_0p4C_subDC_resolution_floor_summary.csv"

OUT_DAY23A_G_REGION_ASSIGNMENT = DATA_DIR / "day23A_step4_MJ1_0p4C_subDC_G0G1G2_assignment.csv"
OUT_DAY23A_DTQ_LONG = DATA_DIR / "day23A_step5_MJ1_0p4C_subDC_dtQ_Gregion_audit_long.csv"
OUT_DAY23A_DTQ_SUMMARY = DATA_DIR / "day23A_step5_MJ1_0p4C_subDC_dtQ_Gregion_summary.csv"
OUT_DAY23A_VERDICT = DATA_DIR / "day23A_step6_MJ1_0p4C_subDC_generalized_verdict.csv"
OUT_DAY23A_CLOSURE_CSV = DATA_DIR / "day23A_step7_closure_summary.csv"
OUT_DAY23A_CLOSURE_MD = DATA_DIR / "day23A_step7_closure_note.md"

OUT_DAY23A_MANUAL_METADATA = DATA_DIR / "metadata" / "day23A_MJ1_0p4C_subDC_manual_metadata.csv"


# =============================================================================
# 1A.6 Helper functions
# =============================================================================

def git_head_or_unknown_day23(repo_path: Path):
    try:
        out = subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=repo_path,
            stderr=subprocess.DEVNULL,
        )
        return out.decode("utf-8").strip()
    except Exception:
        return "unknown_git_head"


def is_finite_number_day23(x):
    try:
        return np.isfinite(float(x))
    except Exception:
        return False


def parse_bool_strict_day23(x):
    if isinstance(x, (bool, np.bool_)):
        return bool(x)

    if isinstance(x, (int, np.integer)) and x in [0, 1]:
        return bool(x)

    if isinstance(x, str):
        s = x.strip().lower()
        if s == "true":
            return True
        if s == "false":
            return False

    raise ValueError(f"Cannot parse strict boolean from value: {x!r}")


def compute_t_ac_s_day23(frequency_Hz):
    if not is_finite_number_day23(frequency_Hz) or float(frequency_Hz) <= 0:
        return np.nan
    return 1.0 / float(frequency_Hz)


def boundary_ordering_status_day23(q_vmax_dc_ah, q_vmax_dcac_ah, tol_ah=Q_BOUNDARY_DEGENERATE_TOLERANCE_AH):
    if not is_finite_number_day23(q_vmax_dc_ah) or not is_finite_number_day23(q_vmax_dcac_ah):
        return BOUNDARY_ORDER_UNRESOLVED

    q_dc = float(q_vmax_dc_ah)
    q_dcac = float(q_vmax_dcac_ah)

    if q_dcac < q_dc - tol_ah:
        return BOUNDARY_ORDER_DCAC_FIRST

    if q_dc < q_dcac - tol_ah:
        return BOUNDARY_ORDER_DC_FIRST

    return BOUNDARY_ORDER_DEGENERATE


def assign_generalized_region_day23(q_ah, q_vmax_dc_ah, q_vmax_dcac_ah, q_final_dc_ah, q_final_dcac_ah, eps_ah=1e-9):
    required = [q_ah, q_vmax_dc_ah, q_vmax_dcac_ah, q_final_dc_ah, q_final_dcac_ah]
    if not all(is_finite_number_day23(x) for x in required):
        return REGION_UNRESOLVED

    q = float(q_ah)
    q_dc = float(q_vmax_dc_ah)
    q_dcac = float(q_vmax_dcac_ah)
    q_common_final = min(float(q_final_dc_ah), float(q_final_dcac_ah))

    q_boundary_lo = min(q_dc, q_dcac)
    q_boundary_hi = max(q_dc, q_dcac)

    if q > q_common_final + eps_ah:
        return REGION_OUTSIDE

    if q <= q_boundary_lo + eps_ah:
        return REGION_G0

    if q <= q_boundary_hi + eps_ah:
        return REGION_G1

    return REGION_G2


def final_q_status_day23(q_final_dc_ah, q_final_dcac_ah, warning_threshold_mAh=10.0):
    if not is_finite_number_day23(q_final_dc_ah) or not is_finite_number_day23(q_final_dcac_ah):
        return FINAL_Q_UNRESOLVED

    diff_mAh = abs(float(q_final_dc_ah) - float(q_final_dcac_ah)) * 1000.0
    if diff_mAh <= warning_threshold_mAh:
        return FINAL_Q_CONSISTENT
    return FINAL_Q_MISMATCH_WARNING


def compute_common_anchors_day23(q_final_dc_ah, q_final_dcac_ah, q_nom_ah=Q_NOM_AH):
    if not is_finite_number_day23(q_final_dc_ah) or not is_finite_number_day23(q_final_dcac_ah):
        return {
            "Q_common_final_Ah": np.nan,
            "Q80_common_Ah": np.nan,
            "Q90_common_Ah": np.nan,
            "Q80_common_fraction_of_Q_nom": np.nan,
            "Q90_common_fraction_of_Q_nom": np.nan,
        }

    q_common_final = min(float(q_final_dc_ah), float(q_final_dcac_ah))
    q80 = 0.80 * q_common_final
    q90 = 0.90 * q_common_final

    return {
        "Q_common_final_Ah": q_common_final,
        "Q80_common_Ah": q80,
        "Q90_common_Ah": q90,
        "Q80_common_fraction_of_Q_nom": q80 / q_nom_ah,
        "Q90_common_fraction_of_Q_nom": q90 / q_nom_ah,
    }


GIT_HEAD_DAY23A = git_head_or_unknown_day23(REPO)


# =============================================================================
# 1A.7 Audit contract JSON
# =============================================================================

audit_contract_day23 = {
    "group_id": DAY23A_GROUP_ID,
    "notebook": DAY23A_NOTEBOOK_NAME,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "git_head": GIT_HEAD_DAY23A,

    "framework_decision": {
        "mode": framework_mode,
        "downstream_segmentation_rule": segmentation_rule,
        "original_A_B_D_framework_allowed": original_abd_allowed,
        "generalized_boundary_ordering_required": True,
        "reason": (
            "Cell 0C shows continued fixed-frequency current components after Vmax "
            "for all active DCAC files. Day21A/Day22A AC-off Segment B assumption is invalid."
        ),
    },

    "scientific_scope": {
        "purpose": (
            "Audit fixed-1tau 0.4C sub-DC-amplitude MJ1 protocols under generalized "
            "boundary ordering because post-Vmax AC continues."
        ),
        "central_question": (
            "Does a boundary-related first-passage effect persist when AC_C < DC_C, "
            "and how does boundary ordering behave when AC is not switched off after Vmax?"
        ),
        "active_kappa_values": [0.25, 0.75],
        "excluded_kappa_values": [0.50],
    },

    "active_protocols": DAY23A_ACTIVE_PROTOCOLS,
    "excluded_files": {
        "files": DAY23A_EXCLUDED_FILES_LOCKED,
        "reason": "0.4C+0.2C raw record lacks required pre-Vmax CC segment.",
    },

    "constants": {
        "ONE_C_A": ONE_C_A,
        "Q_NOM_AH": Q_NOM_AH,
        "TAU_LABEL_S": TAU_LABEL_S,
        "FREQ_1TAU_HZ": FREQ_1TAU_HZ,
        "Q80_NOMINAL_AH": Q80_NOMINAL_AH,
        "Q90_NOMINAL_AH": Q90_NOMINAL_AH,
        "VMAX_V": VMAX_V,
        "ICUTOFF_A": ICUTOFF_A,
        "G0_Q_LO_AH": G0_Q_LO_AH,
        "Q_GRID_STEP_AH": Q_GRID_STEP_AH,
        "Q_GRID_MIN_COUNT_G0": Q_GRID_MIN_COUNT_G0,
    },

    "thresholds": {
        "Q_BOUNDARY_DEGENERATE_TOLERANCE_AH": Q_BOUNDARY_DEGENERATE_TOLERANCE_AH,
        "G0_RESID_FLOOR_COMPATIBLE_THRESHOLD_S": G0_RESID_FLOOR_COMPATIBLE_THRESHOLD_S,
        "G0_RESID_REOPEN_THRESHOLD_S": G0_RESID_REOPEN_THRESHOLD_S,
        "POST_BOUNDARY_PRESERVATION_THRESHOLD_S": POST_BOUNDARY_PRESERVATION_THRESHOLD_S,
    },

    "generalized_regions": {
        "G0": "Q <= min(Q_DC,Vmax, Q_DCAC,Vmax)",
        "G1": "min(Q_DC,Vmax, Q_DCAC,Vmax) < Q <= max(Q_DC,Vmax, Q_DCAC,Vmax)",
        "G2": "Q > max(Q_DC,Vmax, Q_DCAC,Vmax)",
        "residual_allowed_only_in": "G0",
    },

    "integration_rule": {
        "name": "strict_net_signed_current",
        "rectification": False,
        "monotonic_forcing": False,
        "first_passage": True,
    },

    "formal_vs_diagnostic": {
        "formal_geometry": "metadata_prescribed_waveform_in_G0_only",
        "diagnostic_geometry": "fitted_waveform",
        "rule": "fitted_waveform_residual_does_not_replace_prescribed_geometry_residual",
    },

    "caveats": {
        "continued_AC_after_Vmax": DAY23A_CONTINUED_AC_CAVEAT,
        "not_same_protocol_family_as_Day21A_Day22A": DAY23A_NOT_SAME_PROTOCOL_FAMILY_CAVEAT,
        "temperature_summary_missing": DAY23A_TEMPERATURE_CAVEAT,
        "no_repeat_noise_floor": DAY23A_NOISE_FLOOR_CAVEAT,
    },

    "temperature_handling": {
        "status": DAY23A_TEMPERATURE_STATUS,
        "all_temperature_values": "NaN",
        "thermal_attribution_allowed": DAY23A_THERMAL_ATTRIBUTION_ALLOWED,
    },
}

OUT_DAY23A_AUDIT_CONTRACT_JSON.write_text(
    json.dumps(audit_contract_day23, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("[OK] Day23A Cell 1A generalized boundary-ordering contract frozen.")
print(f"[OK] Git HEAD: {GIT_HEAD_DAY23A}")
print(f"[OK] Audit contract JSON: {OUT_DAY23A_AUDIT_CONTRACT_JSON}")
print(f"[OK] Framework mode: {framework_mode}")
print(f"[OK] Downstream segmentation rule: {segmentation_rule}")
print(f"[OK] Original A/B/D framework allowed: {original_abd_allowed}")
print(f"[OK] Active files: {DAY23A_ACTIVE_FILES}")
print(f"[OK] Excluded files: {DAY23A_EXCLUDED_FILES_LOCKED}")
print(f"[OK] FREQ_1TAU_HZ = {FREQ_1TAU_HZ:.9f} Hz")
print(f"[OK] Q80_nominal_Ah = {Q80_NOMINAL_AH:.4f}")
print(f"[OK] Q90_nominal_Ah = {Q90_NOMINAL_AH:.4f}")
print(f"[OK] Temperature summary: {DAY23A_TEMPERATURE_STATUS}")
print("[OK] No trajectory processing performed in Cell 1A.")

[OK] Previous Day23A framework decision loaded.
[OK] framework_mode = generalized_boundary_ordering_required
[OK] segmentation_rule = use_G0_G1_G2_boundary_ordering
[OK] original_A_B_D_framework_allowed = False
[OK] Day23A Cell 1A generalized boundary-ordering contract frozen.
[OK] Git HEAD: 70e8d6a405ec267f566bab1d7fa4829900b7826d
[OK] Audit contract JSON: /Users/louislu/pybamm-dcac-superimposed/data/day23A_audit_contract_schema_thresholds.json
[OK] Framework mode: generalized_boundary_ordering_required
[OK] Downstream segmentation rule: use_G0_G1_G2_boundary_ordering
[OK] Original A/B/D framework allowed: False
[OK] Active files: ['0.4C DC.csv', '处理后DC0.4C+AC0.1C f=0.0143Hz.csv', 'DC0.4C+AC0.3C f=0.0143Hz.csv']
[OK] Excluded files: ['DC0.4C+AC0.2C f=0.0143Hz.csv']
[OK] FREQ_1TAU_HZ = 0.014338283 Hz
[OK] Q80_nominal_Ah = 2.7200
[OK] Q90_nominal_Ah = 3.0600
[OK] Temperature summary: missing_temperature_summary
[OK] No trajectory processing performed in Cell 1A.


In [7]:
# Day23A Cell 2 — active file inventory + metadata merge
#
# Purpose:
# - Build the active Day23A file inventory from active format inventory, timebase audit,
#   protocol-mode audit, and exclusion registry
# - Create or merge manual metadata
# - Carry missing temperature summaries as NaN
# - Carry protocol-mode status and framework decision into each active row
#
# Explicitly NOT done here:
# - No trajectory loading
# - No Q integration
# - No event detection
# - No generalized G0/G1/G2 assignment
# - No Δt computation
# - No verdict

required_day23A_cell2_inputs = [
    OUT_DAY23A_ACTIVE_FORMAT_INVENTORY,
    OUT_DAY23A_EXCLUSION_AUDIT,
    OUT_DAY23A_TIMEBASE_AUDIT,
    OUT_DAY23A_PROTOCOL_MODE_AUDIT,
    OUT_DAY23A_FRAMEWORK_DECISION,
    OUT_DAY23A_AUDIT_CONTRACT_JSON,
]

missing = [p for p in required_day23A_cell2_inputs if not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        "Cannot build Day23A inventory. Missing required files:\n"
        + "\n".join(str(p) for p in missing)
    )

df_active_fmt = pd.read_csv(OUT_DAY23A_ACTIVE_FORMAT_INVENTORY)
df_exclusion = pd.read_csv(OUT_DAY23A_EXCLUSION_AUDIT)
df_timebase = pd.read_csv(OUT_DAY23A_TIMEBASE_AUDIT)
df_protocol_mode = pd.read_csv(OUT_DAY23A_PROTOCOL_MODE_AUDIT)
df_framework = pd.read_csv(OUT_DAY23A_FRAMEWORK_DECISION)

print(f"[OK] Loaded active format inventory: {OUT_DAY23A_ACTIVE_FORMAT_INVENTORY}")
print(f"[OK] Loaded exclusion audit: {OUT_DAY23A_EXCLUSION_AUDIT}")
print(f"[OK] Loaded timebase audit: {OUT_DAY23A_TIMEBASE_AUDIT}")
print(f"[OK] Loaded protocol-mode audit: {OUT_DAY23A_PROTOCOL_MODE_AUDIT}")
print(f"[OK] Loaded framework decision: {OUT_DAY23A_FRAMEWORK_DECISION}")


# =============================================================================
# 2.1 Create or load manual metadata
# =============================================================================

metadata_cols = [
    "file_name",
    "T_surface_max_C",
    "T_surface_mean_C",
    "temperature_sensor_type",
    "temperature_alignment_method",
    "temperature_data_status",
    "U00_12h_V",
    "metadata_status",
    "metadata_notes",
]

default_meta_rows = []

for _, row in df_active_fmt.iterrows():
    default_meta_rows.append({
        "file_name": row["file_name"],
        "T_surface_max_C": np.nan,
        "T_surface_mean_C": np.nan,
        "temperature_sensor_type": "unknown_or_not_available",
        "temperature_alignment_method": "not_available",
        "temperature_data_status": "missing_temperature_summary",
        "U00_12h_V": np.nan,
        "metadata_status": "auto_created_missing_temperature_summary",
        "metadata_notes": "Temperature summary unavailable for Day23A active subset; U00 optional and currently NaN unless manually patched.",
    })

df_default_meta = pd.DataFrame(default_meta_rows, columns=metadata_cols)

OUT_DAY23A_MANUAL_METADATA.parent.mkdir(parents=True, exist_ok=True)

if not OUT_DAY23A_MANUAL_METADATA.exists():
    df_default_meta.to_csv(OUT_DAY23A_MANUAL_METADATA, index=False)
    print(f"[metadata] Created Day23A manual metadata template: {OUT_DAY23A_MANUAL_METADATA}")
else:
    df_existing_meta = pd.read_csv(OUT_DAY23A_MANUAL_METADATA)

    # Ensure all required metadata columns exist.
    for c in metadata_cols:
        if c not in df_existing_meta.columns:
            df_existing_meta[c] = np.nan

    # Add missing active rows if needed.
    existing_files = set(df_existing_meta["file_name"].astype(str))
    missing_meta_rows = df_default_meta[
        ~df_default_meta["file_name"].astype(str).isin(existing_files)
    ]

    if len(missing_meta_rows) > 0:
        df_existing_meta = pd.concat(
            [df_existing_meta[metadata_cols], missing_meta_rows[metadata_cols]],
            ignore_index=True,
        )

    df_existing_meta = df_existing_meta[metadata_cols].copy()
    df_existing_meta.to_csv(OUT_DAY23A_MANUAL_METADATA, index=False)
    print(f"[metadata] Merged Day23A manual metadata: {OUT_DAY23A_MANUAL_METADATA}")

df_meta = pd.read_csv(OUT_DAY23A_MANUAL_METADATA)

# Restrict metadata to active files only.
active_files = set(df_active_fmt["file_name"].astype(str))
df_meta = df_meta[df_meta["file_name"].astype(str).isin(active_files)].copy()

if df_meta["file_name"].duplicated().any():
    dupes = df_meta.loc[df_meta["file_name"].duplicated(), "file_name"].tolist()
    raise ValueError(f"Duplicate file_name rows in Day23A metadata: {dupes}")


# =============================================================================
# 2.2 Merge active inventory
# =============================================================================

# Minimal selected columns to avoid duplicate-column chaos.
fmt_cols = [
    "file_name",
    "file_path",
    "protocol_label",
    "protocol_role",
    "DC_C",
    "AC_C",
    "kappa",
    "m_tau",
    "tau_label_s",
    "frequency_Hz",
    "frequency_source",
    "candidate_for_DC_reference",
    "candidate_for_DCAC",
    "csv_format_refined",
    "encoding_used",
    "delimiter",
    "header_line_idx",
    "columns",
    "time_column_name",
    "time_column_style",
    "source_session_date",
    "start_time",
    "logging_interval_s",
]

time_cols = [
    "file_name",
    "time_parse_method",
    "time_monotonic_status",
    "time_unwrap_count",
    "time_rollover_period_s",
    "time_reconstructed_from_row_index",
    "dt_median_s",
    "dt_min_s",
    "dt_max_s",
    "duration_s",
]

proto_cols = [
    "file_name",
    "vmax_detected",
    "t_Vmax_s",
    "U_at_Vmax_V",
    "expected_DC_A",
    "expected_AC_A",
    "pre_amp_ratio_to_expected",
    "post_amp_ratio_to_expected",
    "post_late_amp_ratio_to_expected",
    "protocol_mode_status",
    "interpretation",
]

excl_cols = [
    "file_name",
    "use_for_audit",
    "exclusion_status",
    "exclusion_reason",
]

df_inv = df_active_fmt[fmt_cols].copy()

df_inv = df_inv.merge(
    df_exclusion[excl_cols],
    on="file_name",
    how="left",
    validate="one_to_one",
)

df_inv = df_inv.merge(
    df_timebase[time_cols],
    on="file_name",
    how="left",
    validate="one_to_one",
)

df_inv = df_inv.merge(
    df_protocol_mode[proto_cols],
    on="file_name",
    how="left",
    validate="one_to_one",
)

df_inv = df_inv.merge(
    df_meta[metadata_cols],
    on="file_name",
    how="left",
    validate="one_to_one",
)

df_inv["source_type"] = SOURCE_TYPE_MJ1
df_inv["cell_or_param_set"] = CELL_ID
df_inv["group_id"] = DAY23A_GROUP_ID
df_inv["sampling_rate_Hz"] = 1.0 / df_inv["dt_median_s"]

framework_mode = str(df_framework["day23a_framework_mode"].iloc[0])
segmentation_rule = str(df_framework["downstream_segmentation_rule"].iloc[0])
original_abd_allowed = bool(df_framework["original_A_B_D_framework_allowed"].iloc[0])

df_inv["day23a_framework_mode"] = framework_mode
df_inv["downstream_segmentation_rule"] = segmentation_rule
df_inv["original_A_B_D_framework_allowed"] = original_abd_allowed

df_inv["notes"] = df_inv.apply(
    lambda r: (
        f"framework={framework_mode}; "
        f"segmentation={segmentation_rule}; "
        f"original_A_B_D_allowed={original_abd_allowed}; "
        f"protocol_mode={r['protocol_mode_status']}; "
        f"temperature={r['temperature_data_status']}; "
        f"excluded_0p4C_0p2C_missing_CC_raw=True"
    ),
    axis=1,
)


# =============================================================================
# 2.3 Type normalization and hard guards
# =============================================================================

bool_cols = [
    "candidate_for_DC_reference",
    "candidate_for_DCAC",
    "use_for_audit",
    "time_reconstructed_from_row_index",
    "original_A_B_D_framework_allowed",
]

for c in bool_cols:
    if c in df_inv.columns:
        df_inv[c] = df_inv[c].map(parse_bool_strict_day23)

numeric_cols = [
    "DC_C",
    "AC_C",
    "kappa",
    "m_tau",
    "tau_label_s",
    "frequency_Hz",
    "header_line_idx",
    "logging_interval_s",
    "time_unwrap_count",
    "time_rollover_period_s",
    "dt_median_s",
    "dt_min_s",
    "dt_max_s",
    "duration_s",
    "sampling_rate_Hz",
    "t_Vmax_s",
    "U_at_Vmax_V",
    "expected_DC_A",
    "expected_AC_A",
    "pre_amp_ratio_to_expected",
    "post_amp_ratio_to_expected",
    "post_late_amp_ratio_to_expected",
    "T_surface_max_C",
    "T_surface_mean_C",
    "U00_12h_V",
]

for c in numeric_cols:
    if c in df_inv.columns:
        df_inv[c] = pd.to_numeric(df_inv[c], errors="coerce")

# Hard guards
if len(df_inv) != 3:
    raise ValueError(f"Expected exactly 3 active Day23A files, got {len(df_inv)}")

if "DC0.4C+AC0.2C f=0.0143Hz.csv" in df_inv["file_name"].tolist():
    raise ValueError("Excluded 0.4C+0.2C file appeared in active Day23A inventory.")

n_ref = int(df_inv["candidate_for_DC_reference"].sum())
n_dcac = int(df_inv["candidate_for_DCAC"].sum())

if n_ref != 1:
    raise ValueError(f"Expected exactly one active DC reference, found {n_ref}")

if n_dcac != 2:
    raise ValueError(f"Expected exactly two active DCAC files, found {n_dcac}")

if bool(df_inv["original_A_B_D_framework_allowed"].any()):
    raise ValueError("Original A/B/D framework unexpectedly allowed in Day23A inventory.")

if not (df_inv["downstream_segmentation_rule"] == "use_G0_G1_G2_boundary_ordering").all():
    raise ValueError("Day23A inventory does not consistently use G0/G1/G2 boundary ordering.")

if not (df_inv["temperature_data_status"] == "missing_temperature_summary").all():
    raise ValueError("Day23A temperature status is not consistently marked as missing.")

dcac_proto = df_inv[df_inv["protocol_role"] == "DCAC"]
if not (dcac_proto["protocol_mode_status"] == PROTOCOL_MODE_CONTINUED_AC).all():
    raise ValueError("Day23A DCAC protocol-mode status should be possible_full_DCAC_after_Vmax for all active DCAC rows.")

if not df_inv["time_monotonic_status"].isin([
    "timebase_ok_approximately_1Hz",
    "timebase_ok_nonstandard_sampling",
]).all():
    raise ValueError("At least one Day23A active file has invalid timebase status.")


# =============================================================================
# 2.4 Save inventory
# =============================================================================

# Put key columns first; keep any extra columns afterward.
preferred_cols = [
    "file_name",
    "file_path",
    "source_type",
    "cell_or_param_set",
    "group_id",

    "protocol_label",
    "protocol_role",
    "DC_C",
    "AC_C",
    "kappa",
    "m_tau",
    "tau_label_s",
    "frequency_Hz",
    "frequency_source",

    "candidate_for_DC_reference",
    "candidate_for_DCAC",
    "use_for_audit",
    "exclusion_status",
    "exclusion_reason",

    "day23a_framework_mode",
    "downstream_segmentation_rule",
    "original_A_B_D_framework_allowed",

    "protocol_mode_status",
    "pre_amp_ratio_to_expected",
    "post_amp_ratio_to_expected",
    "post_late_amp_ratio_to_expected",

    "csv_format_refined",
    "encoding_used",
    "delimiter",
    "header_line_idx",
    "columns",
    "time_column_name",
    "time_column_style",
    "source_session_date",
    "start_time",
    "logging_interval_s",

    "time_parse_method",
    "time_monotonic_status",
    "time_unwrap_count",
    "time_rollover_period_s",
    "time_reconstructed_from_row_index",
    "dt_median_s",
    "dt_min_s",
    "dt_max_s",
    "duration_s",
    "sampling_rate_Hz",

    "T_surface_max_C",
    "T_surface_mean_C",
    "temperature_sensor_type",
    "temperature_alignment_method",
    "temperature_data_status",
    "U00_12h_V",
    "metadata_status",

    "notes",
]

all_cols = preferred_cols + [c for c in df_inv.columns if c not in preferred_cols]
df_inv = df_inv[all_cols].copy()

df_inv.to_csv(OUT_DAY23A_FILE_INVENTORY, index=False)

print(f"[OK] Wrote Day23A file inventory: {OUT_DAY23A_FILE_INVENTORY}")
print(f"[OK] inventory shape = {df_inv.shape}")
print(f"[OK] Active DC reference candidates = {n_ref}")
print(f"[OK] Active DCAC candidates = {n_dcac}")
print(f"[OK] Manual metadata file = {OUT_DAY23A_MANUAL_METADATA}")

display_cols = [
    "file_name",
    "protocol_label",
    "protocol_role",
    "DC_C",
    "AC_C",
    "kappa",
    "frequency_Hz",
    "day23a_framework_mode",
    "downstream_segmentation_rule",
    "protocol_mode_status",
    "pre_amp_ratio_to_expected",
    "post_amp_ratio_to_expected",
    "post_late_amp_ratio_to_expected",
    "csv_format_refined",
    "time_monotonic_status",
    "dt_median_s",
    "sampling_rate_Hz",
    "temperature_data_status",
]

print(df_inv[display_cols].to_string(index=False))

print("[OK] Cell 2 Day23A active file inventory completed.")
print("[OK] No trajectory loading, Q integration, event detection, G-region assignment, or verdict performed.")

[OK] Loaded active format inventory: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step0A5_active_raw_csv_format_inventory.csv
[OK] Loaded exclusion audit: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step0A5_file_exclusion_audit.csv
[OK] Loaded timebase audit: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step0B_timebase_audit.csv
[OK] Loaded protocol-mode audit: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step0C_protocol_mode_acoff_audit.csv
[OK] Loaded framework decision: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step0D_framework_decision.csv
[metadata] Created Day23A manual metadata template: /Users/louislu/pybamm-dcac-superimposed/data/metadata/day23A_MJ1_0p4C_subDC_manual_metadata.csv
[OK] Wrote Day23A file inventory: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step0_MJ1_0p4C_subDC_file_inventory.csv
[OK] inventory shape = (3, 61)
[OK] Active DC reference candidates = 1
[OK] Active DCAC candidates = 2
[OK] Manual metadata file = /Users

In [8]:
# Day23A Cell 3 — trajectory loading and sanity checks
#
# Purpose:
# - Load active Day23A raw CSV trajectories
# - Parse and normalize timebase
# - Remove leading invalid / NaN rows
# - Retain full active charging trajectory after charge onset
# - Store retained trajectories in TRAJ23
#
# Explicitly NOT done here:
# - No Q integration
# - No Vmax event extraction
# - No generalized G0/G1/G2 assignment
# - No Δt computation
# - No verdict

OUT_DAY23A_LOAD_SUMMARY = DATA_DIR / "day23A_step1_MJ1_0p4C_subDC_loaded_trajectory_sanity.csv"

if not OUT_DAY23A_FILE_INVENTORY.exists():
    raise FileNotFoundError(
        f"Day23A file inventory missing: {OUT_DAY23A_FILE_INVENTORY}\n"
        "Run Day23A Cell 2 first."
    )

df_day23_inventory = pd.read_csv(OUT_DAY23A_FILE_INVENTORY)

print(f"[OK] Loaded Day23A inventory: {OUT_DAY23A_FILE_INVENTORY}")
print(f"[OK] inventory rows = {len(df_day23_inventory)}")


# =============================================================================
# 3.1 Helpers
# =============================================================================

def parse_timestamp_to_seconds_day23_load(value):
    if pd.isna(value):
        return np.nan

    s = str(value).strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    try:
        if ":" not in s:
            return float(s)

        parts = [float(p) for p in s.split(":")]

        if len(parts) == 3:
            h, m, sec = parts
            return h * 3600.0 + m * 60.0 + sec

        if len(parts) == 2:
            m, sec = parts
            return m * 60.0 + sec

        return np.nan

    except Exception:
        return np.nan


def unwrap_timestamp_day23_load(t_raw_s):
    t = np.asarray(t_raw_s, dtype=float)
    out = np.full_like(t, np.nan, dtype=float)

    finite_idx = np.where(np.isfinite(t))[0]
    if len(finite_idx) == 0:
        return out, 0, np.nan

    finite_vals = t[finite_idx]
    max_val = np.nanmax(finite_vals)
    rollover_period_s = 86400.0 if max_val > 3600.0 else 3600.0

    offset = 0.0
    unwrap_count = 0

    first_idx = finite_idx[0]
    out[first_idx] = t[first_idx]
    prev = out[first_idx]

    for idx in finite_idx[1:]:
        val = t[idx]

        if val + offset < prev - 10.0:
            offset += rollover_period_s
            unwrap_count += 1

        out[idx] = val + offset
        prev = out[idx]

    return out, unwrap_count, rollover_period_s


def read_day23_raw_csv(row):
    path = Path(row["file_path"])
    delimiter = row["delimiter"]
    header_line_idx = int(row["header_line_idx"])

    df = pd.read_csv(
        path,
        sep=delimiter,
        skiprows=header_line_idx,
        engine="python",
    )

    keep_cols = []
    for c in df.columns:
        c_str = str(c).strip()
        if c_str.startswith("Unnamed") and df[c].isna().all():
            continue
        if c_str == "" and df[c].isna().all():
            continue
        keep_cols.append(c)

    df = df[keep_cols].copy()

    required = ["Timestamp", "U1[V]", "I1[A]"]
    for c in required:
        if c not in df.columns:
            raise ValueError(f"{path.name}: missing required column {c}. columns={df.columns.tolist()}")

    df["t_raw_s"] = df["Timestamp"].map(parse_timestamp_to_seconds_day23_load)
    t_unwrapped, unwrap_count, rollover_period_s = unwrap_timestamp_day23_load(
        df["t_raw_s"].to_numpy()
    )

    df["t_unwrapped_s"] = t_unwrapped

    finite_t = np.isfinite(df["t_unwrapped_s"].to_numpy(dtype=float))
    if finite_t.any():
        t0 = np.nanmin(df.loc[finite_t, "t_unwrapped_s"].to_numpy(dtype=float))
        df["t_elapsed_raw_s"] = df["t_unwrapped_s"] - t0
    else:
        df["t_elapsed_raw_s"] = np.nan

    df["U_V"] = pd.to_numeric(df["U1[V]"], errors="coerce")
    df["I_Q_A"] = pd.to_numeric(df["I1[A]"], errors="coerce")

    return df, unwrap_count, rollover_period_s


def find_charge_onset_idx_day23(df, expected_dc_a, expected_ac_a):
    """
    Day23A active files should start either with leading NaNs or directly
    with valid charge current.

    Charge onset is defined as first row with finite t/U/I and current above 50 mA.
    """
    finite = (
        df["t_elapsed_raw_s"].notna()
        & df["U_V"].notna()
        & df["I_Q_A"].notna()
    )

    charge_threshold_A = 0.050

    candidates = df.index[
        finite
        & (df["I_Q_A"] > charge_threshold_A)
        & (df["U_V"] > 2.0)
        & (df["U_V"] < 4.25)
    ].to_numpy()

    if len(candidates) == 0:
        return None

    return int(candidates[0])


def retain_from_onset_day23(df, onset_idx):
    retained = df.loc[onset_idx:].copy()
    retained = retained[
        retained["t_elapsed_raw_s"].notna()
        & retained["U_V"].notna()
        & retained["I_Q_A"].notna()
    ].copy()

    retained = retained.reset_index(drop=False).rename(columns={"index": "raw_index"})

    t0 = float(retained["t_elapsed_raw_s"].iloc[0])
    retained["t_s"] = retained["t_elapsed_raw_s"] - t0

    return retained


def summarize_dt_day23(t_s):
    t = np.asarray(t_s, dtype=float)
    t = t[np.isfinite(t)]

    if len(t) < 2:
        return np.nan, np.nan, np.nan

    dt = np.diff(t)
    dt = dt[np.isfinite(dt)]

    if len(dt) == 0:
        return np.nan, np.nan, np.nan

    return float(np.median(dt)), float(np.min(dt)), float(np.max(dt))


# =============================================================================
# 3.2 Load active trajectories
# =============================================================================

TRAJ23 = {}
RAW23 = {}
summary_rows = []

for _, row in df_day23_inventory.iterrows():
    file_name = row["file_name"]

    if not parse_bool_strict_day23(row["use_for_audit"]):
        continue

    if file_name in DAY23A_EXCLUDED_FILES_LOCKED:
        raise ValueError(f"Excluded file unexpectedly reached loader: {file_name}")

    raw, unwrap_count, rollover_period_s = read_day23_raw_csv(row)

    expected_dc_a = float(row["DC_C"]) * ONE_C_A
    expected_ac_a = float(row["AC_C"]) * ONE_C_A

    onset_idx = find_charge_onset_idx_day23(
        raw,
        expected_dc_a=expected_dc_a,
        expected_ac_a=expected_ac_a,
    )

    if onset_idx is None:
        raise ValueError(f"Could not detect charge onset for {file_name}")

    retained = retain_from_onset_day23(raw, onset_idx)

    if len(retained) < 100:
        raise ValueError(f"Retained trajectory too short for {file_name}: n={len(retained)}")

    RAW23[file_name] = raw
    TRAJ23[file_name] = retained

    dt_median, dt_min, dt_max = summarize_dt_day23(retained["t_s"].to_numpy())

    finite_voltage_fraction = float(retained["U_V"].notna().mean())
    finite_current_fraction = float(retained["I_Q_A"].notna().mean())

    n_negative_current = int((retained["I_Q_A"] < 0).sum())
    frac_negative_current = float(n_negative_current / len(retained))

    n_current_below_50mA = int((retained["I_Q_A"] <= ICUTOFF_A).sum())

    summary = {
        "file_name": file_name,
        "protocol_label": row["protocol_label"],
        "protocol_role": row["protocol_role"],
        "DC_C": float(row["DC_C"]),
        "AC_C": float(row["AC_C"]),
        "kappa": float(row["kappa"]),
        "frequency_Hz": float(row["frequency_Hz"]),
        "protocol_mode_status": row["protocol_mode_status"],
        "day23a_framework_mode": row["day23a_framework_mode"],

        "csv_format_refined": row["csv_format_refined"],
        "n_raw_rows": int(len(raw)),
        "n_retained_rows": int(len(retained)),
        "charge_onset_idx_raw": onset_idx,
        "charge_onset_timestamp_raw": str(raw.loc[onset_idx, "Timestamp"]),

        "retained_t_start_s": float(retained["t_s"].iloc[0]),
        "retained_t_end_s": float(retained["t_s"].iloc[-1]),
        "retained_duration_s": float(retained["t_s"].iloc[-1] - retained["t_s"].iloc[0]),

        "dt_median_s": dt_median,
        "dt_min_s": dt_min,
        "dt_max_s": dt_max,

        "finite_voltage_fraction": finite_voltage_fraction,
        "finite_current_fraction": finite_current_fraction,

        "U_start_V": float(retained["U_V"].iloc[0]),
        "U_min_V": float(retained["U_V"].min()),
        "U_max_V": float(retained["U_V"].max()),
        "U_end_V": float(retained["U_V"].iloc[-1]),

        "I_start_A": float(retained["I_Q_A"].iloc[0]),
        "I_min_A": float(retained["I_Q_A"].min()),
        "I_max_A": float(retained["I_Q_A"].max()),
        "I_mean_A": float(retained["I_Q_A"].mean()),
        "I_end_A": float(retained["I_Q_A"].iloc[-1]),

        "n_negative_current": n_negative_current,
        "fraction_negative_current": frac_negative_current,
        "n_current_below_or_equal_50mA": n_current_below_50mA,

        "temperature_data_status": row["temperature_data_status"],
        "time_unwrap_count_loader": int(unwrap_count),
        "time_rollover_period_s_loader": rollover_period_s,

        "load_status": "loaded_retained_active_trajectory",
    }

    summary_rows.append(summary)

df_day23_load_summary = pd.DataFrame(summary_rows)
df_day23_load_summary.to_csv(OUT_DAY23A_LOAD_SUMMARY, index=False)

print(f"[OK] Loaded Day23A retained trajectories: {len(TRAJ23)}")
print(f"[OK] Wrote Day23A load sanity summary: {OUT_DAY23A_LOAD_SUMMARY}")

display_cols = [
    "file_name",
    "protocol_label",
    "protocol_role",
    "csv_format_refined",
    "n_raw_rows",
    "n_retained_rows",
    "charge_onset_idx_raw",
    "charge_onset_timestamp_raw",
    "retained_duration_s",
    "dt_median_s",
    "dt_min_s",
    "dt_max_s",
    "U_start_V",
    "U_min_V",
    "U_max_V",
    "U_end_V",
    "I_start_A",
    "I_min_A",
    "I_max_A",
    "I_mean_A",
    "I_end_A",
    "fraction_negative_current",
    "protocol_mode_status",
]

print(df_day23_load_summary[display_cols].to_string(index=False))


# =============================================================================
# 3.3 Hard guards and warnings
# =============================================================================

if len(TRAJ23) != 3:
    raise ValueError(f"Expected 3 retained active trajectories, got {len(TRAJ23)}")

if "DC0.4C+AC0.2C f=0.0143Hz.csv" in TRAJ23:
    raise ValueError("Excluded 0.4C+0.2C file was loaded unexpectedly.")

if not (df_day23_load_summary["dt_median_s"].between(0.8, 1.2)).all():
    raise ValueError("At least one Day23A retained trajectory has non-1Hz median sampling.")

# Warning only: because continued-AC files may not end exactly at 50 mA in a clean CV sense.
if (df_day23_load_summary["U_start_V"] > 3.3).any():
    print("\n[warning] At least one retained trajectory starts above 3.3 V.")
    print("          Confirm that the file contains the intended complete charging record.")

if (df_day23_load_summary["fraction_negative_current"] > 0.01).any():
    print("\n[warning] Negative current intervals detected despite AC_C < DC_C.")
    print("          This may indicate waveform clipping, measurement offset, or sign convention issues.")

if not (df_day23_load_summary["temperature_data_status"] == "missing_temperature_summary").all():
    raise ValueError("Temperature status should remain missing_temperature_summary for Day23A.")

print("[OK] Cell 3 Day23A trajectory loading and sanity checks passed.")
print("[OK] No Q integration, event extraction, G-region assignment, Δt computation, or verdict performed.")

[OK] Loaded Day23A inventory: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step0_MJ1_0p4C_subDC_file_inventory.csv
[OK] inventory rows = 3
[OK] Loaded Day23A retained trajectories: 3
[OK] Wrote Day23A load sanity summary: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step1_MJ1_0p4C_subDC_loaded_trajectory_sanity.csv
                      file_name protocol_label protocol_role                       csv_format_refined  n_raw_rows  n_retained_rows  charge_onset_idx_raw charge_onset_timestamp_raw  retained_duration_s  dt_median_s  dt_min_s  dt_max_s  U_start_V  U_min_V  U_max_V  U_end_V  I_start_A  I_min_A  I_max_A  I_mean_A  I_end_A  fraction_negative_current          protocol_mode_status
                    0.4C DC.csv        0.4C DC  DC_reference                           NGU201_LOG_raw       11505            11051                    58                 09:42:00.5              11050.5          1.0       0.9       2.1   3.070943 3.070943 4.200868 4.200095   1.030244 0.048737 

In [9]:
# Day23A Cell 4 — retained-trajectory event / boundary-time audit
#
# Purpose:
# - Detect Vmax timing on retained Day23A trajectories
# - Detect first post-Vmax I <= 50 mA timing as an audit marker
# - Preserve protocol-mode status: continued AC after Vmax
# - Prepare for later Q_Vmax extraction after Q integration
#
# Explicitly NOT done here:
# - No Q integration
# - No Q_Vmax extraction
# - No generalized G0/G1/G2 assignment
# - No Δt computation
# - No verdict

OUT_DAY23A_EVENT_AUDIT = DATA_DIR / "day23A_step2_MJ1_0p4C_subDC_event_protocol_boundary_audit.csv"

if "TRAJ23" not in globals():
    raise RuntimeError("TRAJ23 not found. Run Day23A Cell 3 first.")

if not OUT_DAY23A_FILE_INVENTORY.exists():
    raise FileNotFoundError(
        f"Day23A file inventory missing: {OUT_DAY23A_FILE_INVENTORY}\n"
        "Run Day23A Cell 2 first."
    )

df_day23_inventory = pd.read_csv(OUT_DAY23A_FILE_INVENTORY)

print(f"[OK] Loaded Day23A inventory: {OUT_DAY23A_FILE_INVENTORY}")
print(f"[OK] retained trajectories available = {len(TRAJ23)}")


# =============================================================================
# 4.1 Helpers
# =============================================================================

def detect_first_vmax_retained_day23(df, vmax_v=VMAX_V, deglitch_n=1):
    """
    Detect first V >= vmax_v on retained trajectory.

    deglitch_n = 1 means first crossing.
    For future noisy files, deglitch_n can require consecutive samples.
    """
    d = df[
        df["t_s"].notna()
        & df["U_V"].notna()
        & df["I_Q_A"].notna()
    ].copy()

    if len(d) == 0:
        return {
            "vmax_detected": False,
            "t_Vmax_detected_s": np.nan,
            "U_at_Vmax_detected_V": np.nan,
            "I_at_Vmax_detected_A": np.nan,
            "raw_index_at_Vmax": np.nan,
            "Vmax_detection_method_used": "failed_no_finite_data",
        }

    u = d["U_V"].to_numpy(dtype=float)
    hit = u >= float(vmax_v)

    if deglitch_n <= 1:
        idxs = np.where(hit)[0]
    else:
        rolling = np.convolve(hit.astype(int), np.ones(deglitch_n, dtype=int), mode="valid")
        idxs = np.where(rolling >= deglitch_n)[0]

    if len(idxs) == 0:
        return {
            "vmax_detected": False,
            "t_Vmax_detected_s": np.nan,
            "U_at_Vmax_detected_V": np.nan,
            "I_at_Vmax_detected_A": np.nan,
            "raw_index_at_Vmax": np.nan,
            "Vmax_detection_method_used": "failed_no_V_ge_4p2",
        }

    local_idx = int(idxs[0])
    row = d.iloc[local_idx]

    return {
        "vmax_detected": True,
        "t_Vmax_detected_s": float(row["t_s"]),
        "U_at_Vmax_detected_V": float(row["U_V"]),
        "I_at_Vmax_detected_A": float(row["I_Q_A"]),
        "raw_index_at_Vmax": int(row["raw_index"]),
        "Vmax_detection_method_used": (
            "first_V_ge_4p2V"
            if deglitch_n <= 1
            else f"first_{deglitch_n}_consecutive_V_ge_4p2V"
        ),
    }


def detect_first_post_vmax_cutoff_marker_day23(df, t_vmax_s, icutoff_a=ICUTOFF_A):
    """
    Detect first retained sample after Vmax with I <= 50 mA.

    Important:
    In Day23A this is only an audit marker, not necessarily a clean CV cutoff,
    because fixed-frequency current modulation can persist after Vmax.
    """
    if not is_finite_number_day23(t_vmax_s):
        return {
            "cutoff_marker_detected": False,
            "t_first_I_le_50mA_after_Vmax_s": np.nan,
            "I_at_cutoff_marker_A": np.nan,
            "U_at_cutoff_marker_V": np.nan,
            "raw_index_at_cutoff_marker": np.nan,
            "cutoff_marker_interpretation": "not_available_no_Vmax",
        }

    d = df[
        df["t_s"].notna()
        & df["U_V"].notna()
        & df["I_Q_A"].notna()
        & (df["t_s"] >= float(t_vmax_s))
    ].copy()

    if len(d) == 0:
        return {
            "cutoff_marker_detected": False,
            "t_first_I_le_50mA_after_Vmax_s": np.nan,
            "I_at_cutoff_marker_A": np.nan,
            "U_at_cutoff_marker_V": np.nan,
            "raw_index_at_cutoff_marker": np.nan,
            "cutoff_marker_interpretation": "not_available_no_post_Vmax_data",
        }

    hit = d[d["I_Q_A"] <= float(icutoff_a)]

    if len(hit) == 0:
        return {
            "cutoff_marker_detected": False,
            "t_first_I_le_50mA_after_Vmax_s": np.nan,
            "I_at_cutoff_marker_A": np.nan,
            "U_at_cutoff_marker_V": np.nan,
            "raw_index_at_cutoff_marker": np.nan,
            "cutoff_marker_interpretation": "not_reached_in_retained_record",
        }

    first = hit.iloc[0]

    return {
        "cutoff_marker_detected": True,
        "t_first_I_le_50mA_after_Vmax_s": float(first["t_s"]),
        "I_at_cutoff_marker_A": float(first["I_Q_A"]),
        "U_at_cutoff_marker_V": float(first["U_V"]),
        "raw_index_at_cutoff_marker": int(first["raw_index"]),
        "cutoff_marker_interpretation": (
            "audit_marker_only_not_clean_CV_cutoff_if_AC_continues_after_Vmax"
        ),
    }


def get_inventory_row_day23(file_name):
    rows = df_day23_inventory[df_day23_inventory["file_name"] == file_name]
    if len(rows) != 1:
        raise ValueError(f"Expected exactly one inventory row for {file_name}, found {len(rows)}")
    return rows.iloc[0]


# =============================================================================
# 4.2 Event audit
# =============================================================================

event_rows = []

for file_name, traj in TRAJ23.items():
    inv = get_inventory_row_day23(file_name)

    vmax = detect_first_vmax_retained_day23(
        traj,
        vmax_v=VMAX_V,
        deglitch_n=1,
    )

    cutoff_marker = detect_first_post_vmax_cutoff_marker_day23(
        traj,
        t_vmax_s=vmax["t_Vmax_detected_s"],
        icutoff_a=ICUTOFF_A,
    )

    t_final_s = float(traj["t_s"].iloc[-1])
    U_final_V = float(traj["U_V"].iloc[-1])
    I_final_A = float(traj["I_Q_A"].iloc[-1])

    if vmax["vmax_detected"]:
        post_vmax_available_s = t_final_s - float(vmax["t_Vmax_detected_s"])
        t_vmax_fraction_of_record = float(vmax["t_Vmax_detected_s"]) / t_final_s if t_final_s > 0 else np.nan
    else:
        post_vmax_available_s = np.nan
        t_vmax_fraction_of_record = np.nan

    if cutoff_marker["cutoff_marker_detected"] and vmax["vmax_detected"]:
        cutoff_lag_after_vmax_s = (
            cutoff_marker["t_first_I_le_50mA_after_Vmax_s"]
            - vmax["t_Vmax_detected_s"]
        )
    else:
        cutoff_lag_after_vmax_s = np.nan

    protocol_mode_status = str(inv["protocol_mode_status"])

    if inv["protocol_role"] == "DC_reference":
        event_framework_status = "event_audit_ok_DC_reference"
    else:
        if protocol_mode_status == PROTOCOL_MODE_CONTINUED_AC:
            event_framework_status = "event_audit_ok_DCAC_continued_AC_after_Vmax"
        else:
            event_framework_status = "event_audit_DCAC_protocol_mode_not_continued_AC"

    row = {
        "file_name": file_name,
        "protocol_label": inv["protocol_label"],
        "protocol_role": inv["protocol_role"],
        "DC_C": float(inv["DC_C"]),
        "AC_C": float(inv["AC_C"]),
        "kappa": float(inv["kappa"]),
        "frequency_Hz": float(inv["frequency_Hz"]),

        "protocol_mode_status": protocol_mode_status,
        "day23a_framework_mode": inv["day23a_framework_mode"],
        "downstream_segmentation_rule": inv["downstream_segmentation_rule"],

        "vmax_detected": vmax["vmax_detected"],
        "t_Vmax_detected_s": vmax["t_Vmax_detected_s"],
        "U_at_Vmax_detected_V": vmax["U_at_Vmax_detected_V"],
        "I_at_Vmax_detected_A": vmax["I_at_Vmax_detected_A"],
        "raw_index_at_Vmax": vmax["raw_index_at_Vmax"],
        "Vmax_detection_method_used": vmax["Vmax_detection_method_used"],

        "cutoff_marker_detected": cutoff_marker["cutoff_marker_detected"],
        "t_first_I_le_50mA_after_Vmax_s": cutoff_marker["t_first_I_le_50mA_after_Vmax_s"],
        "I_at_cutoff_marker_A": cutoff_marker["I_at_cutoff_marker_A"],
        "U_at_cutoff_marker_V": cutoff_marker["U_at_cutoff_marker_V"],
        "raw_index_at_cutoff_marker": cutoff_marker["raw_index_at_cutoff_marker"],
        "cutoff_marker_interpretation": cutoff_marker["cutoff_marker_interpretation"],
        "cutoff_lag_after_Vmax_s": cutoff_lag_after_vmax_s,

        "t_final_s": t_final_s,
        "U_final_V": U_final_V,
        "I_final_A": I_final_A,
        "post_Vmax_available_s": post_vmax_available_s,
        "t_Vmax_fraction_of_record": t_vmax_fraction_of_record,

        "temperature_data_status": inv["temperature_data_status"],
        "event_framework_status": event_framework_status,
    }

    event_rows.append(row)

df_day23_event_audit = pd.DataFrame(event_rows)
df_day23_event_audit.to_csv(OUT_DAY23A_EVENT_AUDIT, index=False)

print(f"[OK] Wrote Day23A event / boundary-time audit: {OUT_DAY23A_EVENT_AUDIT}")

display_cols = [
    "file_name",
    "protocol_label",
    "protocol_role",
    "protocol_mode_status",
    "t_Vmax_detected_s",
    "U_at_Vmax_detected_V",
    "I_at_Vmax_detected_A",
    "t_first_I_le_50mA_after_Vmax_s",
    "cutoff_lag_after_Vmax_s",
    "t_final_s",
    "post_Vmax_available_s",
    "t_Vmax_fraction_of_record",
    "event_framework_status",
]

print(df_day23_event_audit[display_cols].to_string(index=False))


# =============================================================================
# 4.3 Hard guards and warnings
# =============================================================================

if not df_day23_event_audit["vmax_detected"].all():
    bad = df_day23_event_audit[~df_day23_event_audit["vmax_detected"]]
    print("\n[ERROR] Vmax not detected for:")
    print(bad[["file_name", "protocol_label", "Vmax_detection_method_used"]].to_string(index=False))
    raise ValueError("Day23A Vmax detection failed.")

if (df_day23_event_audit["post_Vmax_available_s"] < 60).any():
    print("\n[warning] At least one file has short post-Vmax record length.")
    print("          Post-boundary region may be weakly resolved.")

dcac_events = df_day23_event_audit[df_day23_event_audit["protocol_role"] == "DCAC"]
if not (dcac_events["protocol_mode_status"] == PROTOCOL_MODE_CONTINUED_AC).all():
    raise ValueError("Expected all active Day23A DCAC files to carry continued-AC protocol mode.")

if bool(df_day23_event_audit["downstream_segmentation_rule"].ne("use_G0_G1_G2_boundary_ordering").any()):
    raise ValueError("Day23A event audit found inconsistent downstream segmentation rule.")

print("[OK] Cell 4 Day23A event / boundary-time audit passed.")
print("[OK] No Q integration, Q_Vmax extraction, G-region assignment, Δt computation, or verdict performed.")

[OK] Loaded Day23A inventory: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step0_MJ1_0p4C_subDC_file_inventory.csv
[OK] retained trajectories available = 3
[OK] Wrote Day23A event / boundary-time audit: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step2_MJ1_0p4C_subDC_event_protocol_boundary_audit.csv
                      file_name protocol_label protocol_role          protocol_mode_status  t_Vmax_detected_s  U_at_Vmax_detected_V  I_at_Vmax_detected_A  t_first_I_le_50mA_after_Vmax_s  cutoff_lag_after_Vmax_s  t_final_s  post_Vmax_available_s  t_Vmax_fraction_of_record                      event_framework_status
                    0.4C DC.csv        0.4C DC  DC_reference   not_applicable_DC_reference             7630.6              4.200031              1.359941                         11024.5                   3393.9    11050.5                 3419.9                   0.690521                 event_audit_ok_DC_reference
   DC0.4C+AC0.3C f=0.0143Hz.csv 0.4C+0.3C 1tau     

In [10]:
# Day23A Cell 5 — strict-net Q integration and final-Q audit
#
# Purpose:
# - Integrate signed measured current to obtain Q_net(t)
# - Keep original signed current; no rectification; no monotonic forcing
# - Attach Q_net_Ah to retained trajectories in TRAJ23
# - Audit final-Q consistency for active DC-vs-DCAC pairs
#
# Explicitly NOT done here:
# - No Q_Vmax extraction
# - No G0/G1/G2 assignment
# - No Δt(Q) computation
# - No verdict

OUT_DAY23A_Q_SUMMARY = DATA_DIR / "day23A_step3_MJ1_0p4C_subDC_Q_integration_summary.csv"
OUT_DAY23A_FINALQ_PAIR_AUDIT = DATA_DIR / "day23A_step3_MJ1_0p4C_subDC_finalQ_pair_audit.csv"

if "TRAJ23" not in globals():
    raise RuntimeError("TRAJ23 not found. Run Day23A Cell 3 first.")

if not OUT_DAY23A_EVENT_AUDIT.exists():
    raise FileNotFoundError(
        f"Day23A event audit missing: {OUT_DAY23A_EVENT_AUDIT}\n"
        "Run Day23A Cell 4 first."
    )

df_day23_event_audit = pd.read_csv(OUT_DAY23A_EVENT_AUDIT)
df_day23_inventory = pd.read_csv(OUT_DAY23A_FILE_INVENTORY)

print(f"[OK] Loaded Day23A event audit: {OUT_DAY23A_EVENT_AUDIT}")
print(f"[OK] retained trajectories available = {len(TRAJ23)}")


# =============================================================================
# 5.1 Helpers
# =============================================================================

def integrate_strict_net_Q_Ah_day23(t_s, i_a):
    """
    Strict-net signed-current integration.

    Rules:
    - Use raw signed current.
    - No rectification.
    - No cumulative maximum.
    - Negative current intervals, if present, contribute negatively.
    """
    t = np.asarray(t_s, dtype=float)
    i = np.asarray(i_a, dtype=float)

    q = np.full_like(t, np.nan, dtype=float)

    finite = np.isfinite(t) & np.isfinite(i)

    if finite.sum() < 2:
        return q

    # Work on finite samples only, then map back.
    idx = np.where(finite)[0]
    tf = t[idx]
    ifin = i[idx]

    qf = np.zeros_like(tf, dtype=float)

    for k in range(1, len(tf)):
        dt_s = tf[k] - tf[k - 1]

        if not np.isfinite(dt_s) or dt_s <= 0:
            qf[k] = qf[k - 1]
            continue

        i_avg = 0.5 * (ifin[k] + ifin[k - 1])
        qf[k] = qf[k - 1] + i_avg * dt_s / 3600.0

    q[idx] = qf
    return q


def count_q_decreases_day23(q_Ah, tol_Ah=1e-9):
    q = np.asarray(q_Ah, dtype=float)
    q = q[np.isfinite(q)]

    if len(q) < 2:
        return 0, 0.0

    dq = np.diff(q)
    n_dec = int((dq < -abs(tol_Ah)).sum())
    frac_dec = float(n_dec / len(dq)) if len(dq) else np.nan

    return n_dec, frac_dec


def get_day23_inventory_row(file_name):
    rows = df_day23_inventory[df_day23_inventory["file_name"] == file_name]
    if len(rows) != 1:
        raise ValueError(f"Expected exactly one inventory row for {file_name}, found {len(rows)}")
    return rows.iloc[0]


# =============================================================================
# 5.2 Integrate Q for each retained trajectory
# =============================================================================

q_summary_rows = []

for file_name, traj in TRAJ23.items():
    inv = get_day23_inventory_row(file_name)

    t = traj["t_s"].to_numpy(dtype=float)
    i = traj["I_Q_A"].to_numpy(dtype=float)

    q_net = integrate_strict_net_Q_Ah_day23(t, i)

    # Attach Q to global retained trajectory object.
    TRAJ23[file_name] = traj.copy()
    TRAJ23[file_name]["Q_net_Ah"] = q_net
    TRAJ23[file_name]["Q_net_mAh"] = q_net * 1000.0

    q_finite = q_net[np.isfinite(q_net)]

    if len(q_finite) == 0:
        q_final_Ah = np.nan
        q_max_Ah = np.nan
        q_min_Ah = np.nan
    else:
        q_final_Ah = float(q_finite[-1])
        q_max_Ah = float(np.nanmax(q_finite))
        q_min_Ah = float(np.nanmin(q_finite))

    q_decrease_count, q_decrease_fraction = count_q_decreases_day23(q_net)

    row = {
        "file_name": file_name,
        "protocol_label": inv["protocol_label"],
        "protocol_role": inv["protocol_role"],
        "DC_C": float(inv["DC_C"]),
        "AC_C": float(inv["AC_C"]),
        "kappa": float(inv["kappa"]),
        "frequency_Hz": float(inv["frequency_Hz"]),
        "protocol_mode_status": inv["protocol_mode_status"],
        "day23a_framework_mode": inv["day23a_framework_mode"],

        "t_final_s": float(traj["t_s"].iloc[-1]),
        "Q_final_Ah": q_final_Ah,
        "Q_final_mAh": q_final_Ah * 1000.0 if np.isfinite(q_final_Ah) else np.nan,
        "Q_max_Ah": q_max_Ah,
        "Q_min_Ah": q_min_Ah,
        "Q_decrease_count": q_decrease_count,
        "Q_decrease_fraction": q_decrease_fraction,

        "I_Q_min_A": float(np.nanmin(i)),
        "I_Q_max_A": float(np.nanmax(i)),
        "I_Q_mean_A": float(np.nanmean(i)),

        "strict_net_integration_rule": "signed_current_no_rectification_no_monotonic_forcing",
        "temperature_data_status": inv["temperature_data_status"],
    }

    q_summary_rows.append(row)

df_day23_q_summary = pd.DataFrame(q_summary_rows)
df_day23_q_summary.to_csv(OUT_DAY23A_Q_SUMMARY, index=False)

print(f"[OK] Wrote Day23A Q integration summary: {OUT_DAY23A_Q_SUMMARY}")

display_q_cols = [
    "file_name",
    "protocol_label",
    "protocol_role",
    "DC_C",
    "AC_C",
    "kappa",
    "t_final_s",
    "Q_final_Ah",
    "Q_final_mAh",
    "Q_max_Ah",
    "Q_decrease_count",
    "Q_decrease_fraction",
    "I_Q_min_A",
    "I_Q_max_A",
    "I_Q_mean_A",
    "protocol_mode_status",
]

print(df_day23_q_summary[display_q_cols].to_string(index=False))


# =============================================================================
# 5.3 Final-Q pair audit against active DC reference
# =============================================================================

dc_rows = df_day23_q_summary[df_day23_q_summary["protocol_role"] == "DC_reference"]
if len(dc_rows) != 1:
    raise ValueError(f"Expected exactly one Day23A DC reference in Q summary, found {len(dc_rows)}")

dc_row = dc_rows.iloc[0]
dc_file = dc_row["file_name"]
dc_label = dc_row["protocol_label"]
q_final_dc = float(dc_row["Q_final_Ah"])

pair_rows = []

for _, dcac_row in df_day23_q_summary[df_day23_q_summary["protocol_role"] == "DCAC"].iterrows():
    q_final_dcac = float(dcac_row["Q_final_Ah"])

    final_status = final_q_status_day23(
        q_final_dc_ah=q_final_dc,
        q_final_dcac_ah=q_final_dcac,
        warning_threshold_mAh=10.0,
    )

    common = compute_common_anchors_day23(
        q_final_dc_ah=q_final_dc,
        q_final_dcac_ah=q_final_dcac,
        q_nom_ah=Q_NOM_AH,
    )

    pair_row = {
        "protocol_pair": f"{dc_label} vs {dcac_row['protocol_label']}",
        "file_name_DC": dc_file,
        "file_name_DCAC": dcac_row["file_name"],
        "protocol_label_DC": dc_label,
        "protocol_label_DCAC": dcac_row["protocol_label"],

        "DC_C": float(dcac_row["DC_C"]),
        "AC_C": float(dcac_row["AC_C"]),
        "kappa": float(dcac_row["kappa"]),
        "frequency_Hz": float(dcac_row["frequency_Hz"]),
        "protocol_mode_status_DCAC": dcac_row["protocol_mode_status"],

        "Q_final_DC_Ah": q_final_dc,
        "Q_final_DCAC_Ah": q_final_dcac,
        "Q_final_diff_Ah": q_final_dc - q_final_dcac,
        "Q_final_diff_mAh": (q_final_dc - q_final_dcac) * 1000.0,
        "Q_final_abs_diff_mAh": abs(q_final_dc - q_final_dcac) * 1000.0,
        "Q_final_diff_status": final_status,

        "Q_common_final_Ah": common["Q_common_final_Ah"],
        "Q80_common_Ah": common["Q80_common_Ah"],
        "Q90_common_Ah": common["Q90_common_Ah"],
        "Q80_common_fraction_of_Q_nom": common["Q80_common_fraction_of_Q_nom"],
        "Q90_common_fraction_of_Q_nom": common["Q90_common_fraction_of_Q_nom"],

        "Q80_nominal_Ah": Q80_NOMINAL_AH,
        "Q90_nominal_Ah": Q90_NOMINAL_AH,

        "Q80_nominal_reachable_common": bool(
            np.isfinite(common["Q_common_final_Ah"]) and Q80_NOMINAL_AH <= common["Q_common_final_Ah"]
        ),
        "Q90_nominal_reachable_common": bool(
            np.isfinite(common["Q_common_final_Ah"]) and Q90_NOMINAL_AH <= common["Q_common_final_Ah"]
        ),

        "final_Q_audit_interpretation": (
            "common anchors remain valid; final-Q mismatch caveat required"
            if final_status == FINAL_Q_MISMATCH_WARNING
            else "final-Q consistent; common anchors valid"
        ),
    }

    pair_rows.append(pair_row)

df_day23_finalq_pairs = pd.DataFrame(pair_rows)
df_day23_finalq_pairs.to_csv(OUT_DAY23A_FINALQ_PAIR_AUDIT, index=False)

print(f"\n[OK] Wrote Day23A final-Q pair audit: {OUT_DAY23A_FINALQ_PAIR_AUDIT}")

display_pair_cols = [
    "protocol_pair",
    "Q_final_DC_Ah",
    "Q_final_DCAC_Ah",
    "Q_final_diff_mAh",
    "Q_final_abs_diff_mAh",
    "Q_final_diff_status",
    "Q80_common_Ah",
    "Q90_common_Ah",
    "Q80_common_fraction_of_Q_nom",
    "Q90_common_fraction_of_Q_nom",
    "Q80_nominal_reachable_common",
    "Q90_nominal_reachable_common",
]

print(df_day23_finalq_pairs[display_pair_cols].to_string(index=False))


# =============================================================================
# 5.4 Hard guards and warnings
# =============================================================================

if len(df_day23_q_summary) != 3:
    raise ValueError(f"Expected 3 Day23A Q summary rows, got {len(df_day23_q_summary)}")

if "DC0.4C+AC0.2C f=0.0143Hz.csv" in df_day23_q_summary["file_name"].tolist():
    raise ValueError("Excluded 0.4C+0.2C file appeared in Q summary.")

if (df_day23_q_summary["Q_final_Ah"] <= 0).any():
    raise ValueError("At least one Day23A trajectory has non-positive final Q.")

if (df_day23_q_summary["Q_decrease_count"] > 0).any():
    print("\n[warning] At least one Day23A trajectory has Q decreases under strict-net integration.")
    print("          This is allowed, but unexpected for AC_C < DC_C.")

if (df_day23_finalq_pairs["Q_final_diff_status"] == FINAL_Q_MISMATCH_WARNING).any():
    print("\n[warning] Final-Q mismatch warning in Day23A pair audit.")
    print("          Common anchors remain valid, but later verdicts must carry final-Q mismatch caveat.")

if not df_day23_finalq_pairs["Q80_nominal_reachable_common"].all():
    print("\n[warning] Q80_nominal is not reachable within at least one common final-Q window.")

if not df_day23_finalq_pairs["Q90_nominal_reachable_common"].all():
    print("\n[warning] Q90_nominal is not reachable within at least one common final-Q window.")

print("[OK] Cell 5 Day23A strict-net Q integration and final-Q audit completed.")
print("[OK] No Q_Vmax extraction, G-region assignment, Δt computation, or verdict performed.")

[OK] Loaded Day23A event audit: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step2_MJ1_0p4C_subDC_event_protocol_boundary_audit.csv
[OK] retained trajectories available = 3
[OK] Wrote Day23A Q integration summary: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step3_MJ1_0p4C_subDC_Q_integration_summary.csv
                      file_name protocol_label protocol_role  DC_C  AC_C  kappa  t_final_s  Q_final_Ah  Q_final_mAh  Q_max_Ah  Q_decrease_count  Q_decrease_fraction  I_Q_min_A  I_Q_max_A  I_Q_mean_A          protocol_mode_status
                    0.4C DC.csv        0.4C DC  DC_reference   0.4   0.0   0.00    11050.5    3.333548  3333.548112  3.333548                 0                  0.0   0.048737   1.359949    1.086023   not_applicable_DC_reference
   DC0.4C+AC0.3C f=0.0143Hz.csv 0.4C+0.3C 1tau          DCAC   0.4   0.3   0.75    10956.0    3.319026  3319.025812  3.319026                 0                  0.0   0.049788   2.380011    1.090646 possible_full_DCAC_afte

In [11]:
# Day23A Cell 5A — DC self-consistency resolution-floor diagnostic
#
# Purpose:
# - Estimate Day23A first-passage audit-resolution lower bound
# - Use 0.4C DC reference even/odd row split
# - Quantify sampling / interpolation / integration sensitivity
#
# Explicitly NOT done here:
# - No Q_Vmax extraction
# - No G0/G1/G2 assignment
# - No Δt comparison between protocols
# - No verdict
#
# Interpretation:
# - This is NOT a repeat-based experimental noise floor.
# - It is a lower-bound self-consistency estimate from the DC reference.

OUT_DAY23A_RESOLUTION_LONG = DATA_DIR / "day23A_step3A_MJ1_0p4C_subDC_resolution_floor_long.csv"
OUT_DAY23A_RESOLUTION_SUMMARY = DATA_DIR / "day23A_step3A_MJ1_0p4C_subDC_resolution_floor_summary.csv"

if "TRAJ23" not in globals():
    raise RuntimeError("TRAJ23 not found. Run Day23A Cell 3 first.")

if not OUT_DAY23A_Q_SUMMARY.exists():
    raise FileNotFoundError(
        f"Day23A Q summary missing: {OUT_DAY23A_Q_SUMMARY}\n"
        "Run Day23A Cell 5 first."
    )

df_day23_q_summary = pd.read_csv(OUT_DAY23A_Q_SUMMARY)

print(f"[OK] Loaded Day23A Q summary: {OUT_DAY23A_Q_SUMMARY}")


# =============================================================================
# 5A.1 Helpers
# =============================================================================

def first_passage_time_from_Q_day23(q_target_Ah, t_s, q_Ah):
    """
    First-passage time: first t where Q(t) >= q_target.
    No monotonic correction is applied.
    """
    if not is_finite_number_day23(q_target_Ah):
        return np.nan

    t = np.asarray(t_s, dtype=float)
    q = np.asarray(q_Ah, dtype=float)

    finite = np.isfinite(t) & np.isfinite(q)
    if finite.sum() < 2:
        return np.nan

    t_f = t[finite]
    q_f = q[finite]

    hit = np.where(q_f >= float(q_target_Ah))[0]
    if len(hit) == 0:
        return np.nan

    idx = int(hit[0])

    if idx == 0:
        return float(t_f[idx])

    q0, q1 = q_f[idx - 1], q_f[idx]
    t0, t1 = t_f[idx - 1], t_f[idx]

    if not np.isfinite(q0) or not np.isfinite(q1) or q1 == q0:
        return float(t1)

    frac = (float(q_target_Ah) - q0) / (q1 - q0)
    frac = float(np.clip(frac, 0.0, 1.0))

    return float(t0 + frac * (t1 - t0))


def make_fixed_Q_grid_day23(q_lo_Ah, q_hi_Ah, step_Ah):
    """
    Fixed Q-grid from first grid value >= q_lo to last grid value <= q_hi.
    """
    if not all(is_finite_number_day23(x) for x in [q_lo_Ah, q_hi_Ah, step_Ah]):
        return np.array([], dtype=float)

    q_lo = float(q_lo_Ah)
    q_hi = float(q_hi_Ah)
    step = float(step_Ah)

    if q_hi < q_lo or step <= 0:
        return np.array([], dtype=float)

    start = np.ceil(q_lo / step) * step
    stop = np.floor(q_hi / step) * step

    if stop < start:
        return np.array([], dtype=float)

    n = int(round((stop - start) / step)) + 1
    return start + step * np.arange(n)


def integrate_subset_day23(df, subset_name, selector):
    """
    Build a self-consistency subset from one retained trajectory.

    Important:
    - original t_s is preserved;
    - Q is re-integrated only on the selected subset;
    - this is a lower-bound diagnostic for sampling/interpolation sensitivity,
      not a repeatability estimate.
    """
    out = df.loc[selector].copy().reset_index(drop=True)

    t = out["t_s"].to_numpy(dtype=float)
    i = out["I_Q_A"].to_numpy(dtype=float)

    out["Q_self_Ah"] = integrate_strict_net_Q_Ah_day23(t, i)
    out["self_subset"] = subset_name

    return out


def safe_abs_p95_day23(x):
    arr = np.asarray(x, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.percentile(np.abs(arr), 95)) if len(arr) else np.nan


def safe_abs_max_day23(x):
    arr = np.asarray(x, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.max(np.abs(arr))) if len(arr) else np.nan


def safe_mean_day23(x):
    arr = np.asarray(x, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.mean(arr)) if len(arr) else np.nan


def safe_median_day23(x):
    arr = np.asarray(x, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.median(arr)) if len(arr) else np.nan


# =============================================================================
# 5A.2 Build DC self-consistency subsets
# =============================================================================

dc_rows = df_day23_q_summary[df_day23_q_summary["protocol_role"] == "DC_reference"]
if len(dc_rows) != 1:
    raise ValueError(f"Expected exactly one Day23A DC reference, found {len(dc_rows)}")

dc_file = dc_rows.iloc[0]["file_name"]

if dc_file not in TRAJ23:
    raise KeyError(f"DC reference trajectory not found in TRAJ23: {dc_file}")

df_dc = TRAJ23[dc_file].copy()

if "Q_net_Ah" not in df_dc.columns:
    raise ValueError("DC trajectory does not contain Q_net_Ah. Run Day23A Cell 5 first.")

idx = np.arange(len(df_dc))

df_dc_full = df_dc.copy()
df_dc_full["Q_self_Ah"] = df_dc_full["Q_net_Ah"]
df_dc_full["self_subset"] = "full"

df_dc_even = integrate_subset_day23(df_dc, "even_rows", idx % 2 == 0)
df_dc_odd = integrate_subset_day23(df_dc, "odd_rows", idx % 2 == 1)

q_hi_common = min(
    float(df_dc_full["Q_self_Ah"].iloc[-1]),
    float(df_dc_even["Q_self_Ah"].iloc[-1]),
    float(df_dc_odd["Q_self_Ah"].iloc[-1]),
)

# Leave one grid step margin at the top to avoid endpoint instability.
q_lo = G0_Q_LO_AH
q_hi = q_hi_common - Q_GRID_STEP_AH

q_grid = make_fixed_Q_grid_day23(q_lo, q_hi, Q_GRID_STEP_AH)

if len(q_grid) < Q_GRID_MIN_COUNT_G0:
    raise ValueError(
        f"Day23A resolution-floor Q-grid too short: n={len(q_grid)}. "
        f"Need >= {Q_GRID_MIN_COUNT_G0}."
    )

print(f"[OK] DC self-consistency reference file: {dc_file}")
print(f"[OK] Q-grid: n={len(q_grid)}, lo={q_grid[0]:.3f} Ah, hi={q_grid[-1]:.3f} Ah")


# =============================================================================
# 5A.3 Compute self-Δt diagnostics
# =============================================================================

resolution_rows = []

for q in q_grid:
    t_full = first_passage_time_from_Q_day23(
        q,
        df_dc_full["t_s"].to_numpy(dtype=float),
        df_dc_full["Q_self_Ah"].to_numpy(dtype=float),
    )

    t_even = first_passage_time_from_Q_day23(
        q,
        df_dc_even["t_s"].to_numpy(dtype=float),
        df_dc_even["Q_self_Ah"].to_numpy(dtype=float),
    )

    t_odd = first_passage_time_from_Q_day23(
        q,
        df_dc_odd["t_s"].to_numpy(dtype=float),
        df_dc_odd["Q_self_Ah"].to_numpy(dtype=float),
    )

    resolution_rows.append({
        "Q_Ah": float(q),
        "t_full_s": t_full,
        "t_even_s": t_even,
        "t_odd_s": t_odd,
        "dt_even_minus_odd_s": (
            t_even - t_odd
            if is_finite_number_day23(t_even) and is_finite_number_day23(t_odd)
            else np.nan
        ),
        "dt_full_minus_even_s": (
            t_full - t_even
            if is_finite_number_day23(t_full) and is_finite_number_day23(t_even)
            else np.nan
        ),
        "dt_full_minus_odd_s": (
            t_full - t_odd
            if is_finite_number_day23(t_full) and is_finite_number_day23(t_odd)
            else np.nan
        ),
    })

df_day23_resolution_long = pd.DataFrame(resolution_rows)

p95_even_odd = safe_abs_p95_day23(df_day23_resolution_long["dt_even_minus_odd_s"])
p95_full_even = safe_abs_p95_day23(df_day23_resolution_long["dt_full_minus_even_s"])
p95_full_odd = safe_abs_p95_day23(df_day23_resolution_long["dt_full_minus_odd_s"])

max_even_odd = safe_abs_max_day23(df_day23_resolution_long["dt_even_minus_odd_s"])
max_full_even = safe_abs_max_day23(df_day23_resolution_long["dt_full_minus_even_s"])
max_full_odd = safe_abs_max_day23(df_day23_resolution_long["dt_full_minus_odd_s"])

audit_resolution_p95_s = np.nanmax([p95_even_odd, p95_full_even, p95_full_odd])
audit_resolution_max_s = np.nanmax([max_even_odd, max_full_even, max_full_odd])

summary = {
    "source": "0.4C_DC_self_consistency_even_odd_split",
    "dc_reference_file": dc_file,
    "n_Q_grid": int(len(q_grid)),
    "Q_lo_Ah": float(q_grid[0]),
    "Q_hi_Ah": float(q_grid[-1]),
    "Q_grid_step_Ah": Q_GRID_STEP_AH,

    "dt_even_minus_odd_mean_s": safe_mean_day23(df_day23_resolution_long["dt_even_minus_odd_s"]),
    "dt_even_minus_odd_median_s": safe_median_day23(df_day23_resolution_long["dt_even_minus_odd_s"]),
    "dt_even_minus_odd_p95_abs_s": p95_even_odd,
    "dt_even_minus_odd_max_abs_s": max_even_odd,

    "dt_full_minus_even_p95_abs_s": p95_full_even,
    "dt_full_minus_even_max_abs_s": max_full_even,

    "dt_full_minus_odd_p95_abs_s": p95_full_odd,
    "dt_full_minus_odd_max_abs_s": max_full_odd,

    "day23A_self_consistency_resolution_p95_s": audit_resolution_p95_s,
    "day23A_self_consistency_resolution_max_s": audit_resolution_max_s,

    "repeat_based_noise_floor_available": False,
    "floor_scope": "lower_bound_for_sampling_interpolation_first_passage_resolution_not_repeatability",
    "formal_disappearance_claim_allowed": False,
}

df_day23_resolution_summary = pd.DataFrame([summary])

df_day23_resolution_long.to_csv(OUT_DAY23A_RESOLUTION_LONG, index=False)
df_day23_resolution_summary.to_csv(OUT_DAY23A_RESOLUTION_SUMMARY, index=False)

print(f"[OK] Wrote Day23A resolution-floor long table: {OUT_DAY23A_RESOLUTION_LONG}")
print(f"[OK] Wrote Day23A resolution-floor summary: {OUT_DAY23A_RESOLUTION_SUMMARY}")
print(df_day23_resolution_summary.to_string(index=False))


# =============================================================================
# 5A.4 Hard guards
# =============================================================================

if not np.isfinite(audit_resolution_p95_s):
    raise ValueError("Day23A self-consistency p95 resolution estimate is not finite.")

if audit_resolution_p95_s <= 0:
    raise ValueError("Day23A self-consistency p95 resolution estimate is non-positive.")

print("[OK] Cell 5A Day23A experimental resolution-floor diagnostic completed.")
print("[OK] This is a lower-bound audit-resolution estimate, not a repeat-based noise floor.")
print("[OK] Day23A must not claim strict effect disappearance without repeat-based evidence.")

[OK] Loaded Day23A Q summary: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step3_MJ1_0p4C_subDC_Q_integration_summary.csv
[OK] DC self-consistency reference file: 0.4C DC.csv
[OK] Q-grid: n=328, lo=0.050 Ah, hi=3.320 Ah
[OK] Wrote Day23A resolution-floor long table: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step3A_MJ1_0p4C_subDC_resolution_floor_long.csv
[OK] Wrote Day23A resolution-floor summary: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step3A_MJ1_0p4C_subDC_resolution_floor_summary.csv
                                 source dc_reference_file  n_Q_grid  Q_lo_Ah  Q_hi_Ah  Q_grid_step_Ah  dt_even_minus_odd_mean_s  dt_even_minus_odd_median_s  dt_even_minus_odd_p95_abs_s  dt_even_minus_odd_max_abs_s  dt_full_minus_even_p95_abs_s  dt_full_minus_even_max_abs_s  dt_full_minus_odd_p95_abs_s  dt_full_minus_odd_max_abs_s  day23A_self_consistency_resolution_p95_s  day23A_self_consistency_resolution_max_s  repeat_based_noise_floor_available                            

In [12]:
# Day23A Cell 6 — Q_Vmax extraction and generalized G0/G1/G2 assignment
#
# Purpose:
# - Extract Q_DC,Vmax and Q_DCAC,Vmax from retained Q_net trajectories
# - Determine generalized boundary ordering:
#   DCAC-first / DC-first / degenerate
# - Assign Q80/Q90 nominal and common anchors to G0/G1/G2/outside
#
# Explicitly NOT done here:
# - No Δt(Q) computation
# - No geometry residual computation
# - No mechanism verdict

OUT_DAY23A_G_REGION_ASSIGNMENT = DATA_DIR / "day23A_step4_MJ1_0p4C_subDC_G0G1G2_assignment.csv"

required_day23A_cell6_inputs = [
    OUT_DAY23A_EVENT_AUDIT,
    OUT_DAY23A_Q_SUMMARY,
    OUT_DAY23A_FINALQ_PAIR_AUDIT,
    OUT_DAY23A_RESOLUTION_SUMMARY,
]

missing = [p for p in required_day23A_cell6_inputs if not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        "Cannot run Day23A Cell 6. Missing required files:\n"
        + "\n".join(str(p) for p in missing)
    )

if "TRAJ23" not in globals():
    raise RuntimeError("TRAJ23 not found. Run Day23A Cell 3 first.")

df_day23_event = pd.read_csv(OUT_DAY23A_EVENT_AUDIT)
df_day23_q_summary = pd.read_csv(OUT_DAY23A_Q_SUMMARY)
df_day23_finalq = pd.read_csv(OUT_DAY23A_FINALQ_PAIR_AUDIT)
df_day23_resolution = pd.read_csv(OUT_DAY23A_RESOLUTION_SUMMARY)

print(f"[OK] Loaded event audit: {OUT_DAY23A_EVENT_AUDIT}")
print(f"[OK] Loaded Q summary: {OUT_DAY23A_Q_SUMMARY}")
print(f"[OK] Loaded final-Q audit: {OUT_DAY23A_FINALQ_PAIR_AUDIT}")
print(f"[OK] Loaded resolution summary: {OUT_DAY23A_RESOLUTION_SUMMARY}")


# =============================================================================
# 6.1 Helpers
# =============================================================================

def interp_Q_at_time_day23(traj, t_target_s):
    """
    Interpolate Q_net_Ah at a given retained time.
    """
    if not is_finite_number_day23(t_target_s):
        return np.nan

    if "Q_net_Ah" not in traj.columns:
        raise ValueError("Trajectory does not contain Q_net_Ah. Run Day23A Cell 5 first.")

    t = traj["t_s"].to_numpy(dtype=float)
    q = traj["Q_net_Ah"].to_numpy(dtype=float)

    finite = np.isfinite(t) & np.isfinite(q)
    if finite.sum() < 2:
        return np.nan

    tf = t[finite]
    qf = q[finite]

    if float(t_target_s) < tf[0] or float(t_target_s) > tf[-1]:
        return np.nan

    return float(np.interp(float(t_target_s), tf, qf))


def get_event_row_day23(file_name):
    rows = df_day23_event[df_day23_event["file_name"] == file_name]
    if len(rows) != 1:
        raise ValueError(f"Expected exactly one event row for {file_name}, found {len(rows)}")
    return rows.iloc[0]


def get_q_summary_row_day23(file_name):
    rows = df_day23_q_summary[df_day23_q_summary["file_name"] == file_name]
    if len(rows) != 1:
        raise ValueError(f"Expected exactly one Q summary row for {file_name}, found {len(rows)}")
    return rows.iloc[0]


def get_finalq_pair_row_day23(file_name_dcac):
    rows = df_day23_finalq[df_day23_finalq["file_name_DCAC"] == file_name_dcac]
    if len(rows) != 1:
        raise ValueError(f"Expected exactly one final-Q pair row for {file_name_dcac}, found {len(rows)}")
    return rows.iloc[0]


def get_dc_reference_file_day23():
    rows = df_day23_q_summary[df_day23_q_summary["protocol_role"] == "DC_reference"]
    if len(rows) != 1:
        raise ValueError(f"Expected exactly one DC reference in Q summary, found {len(rows)}")
    return str(rows.iloc[0]["file_name"])


def boundary_ordering_by_time_day23(t_vmax_dc_s, t_vmax_dcac_s, tol_s=1.0):
    if not is_finite_number_day23(t_vmax_dc_s) or not is_finite_number_day23(t_vmax_dcac_s):
        return "boundary_time_order_unresolved"

    t_dc = float(t_vmax_dc_s)
    t_dcac = float(t_vmax_dcac_s)

    if t_dcac < t_dc - tol_s:
        return "DCAC_first_by_time"
    if t_dc < t_dcac - tol_s:
        return "DC_first_by_time"
    return "boundary_time_degenerate"


def region_grid_count_day23(q_lo_Ah, q_hi_Ah, step_Ah=Q_GRID_STEP_AH):
    if not all(is_finite_number_day23(x) for x in [q_lo_Ah, q_hi_Ah, step_Ah]):
        return 0

    q_lo = float(q_lo_Ah)
    q_hi = float(q_hi_Ah)
    step = float(step_Ah)

    if q_hi < q_lo or step <= 0:
        return 0

    start = np.ceil(q_lo / step) * step
    stop = np.floor(q_hi / step) * step

    if stop < start:
        return 0

    return int(round((stop - start) / step)) + 1


# =============================================================================
# 6.2 Extract DC reference boundary
# =============================================================================

dc_file = get_dc_reference_file_day23()
dc_event = get_event_row_day23(dc_file)
dc_q_row = get_q_summary_row_day23(dc_file)
dc_traj = TRAJ23[dc_file]

t_vmax_dc_s = float(dc_event["t_Vmax_detected_s"])
q_vmax_dc_ah = interp_Q_at_time_day23(dc_traj, t_vmax_dc_s)

if not is_finite_number_day23(q_vmax_dc_ah):
    raise ValueError("Could not extract Q_DC,Vmax.")

q_final_dc_ah = float(dc_q_row["Q_final_Ah"])

print(f"[OK] DC reference file = {dc_file}")
print(f"[OK] t_DC,Vmax = {t_vmax_dc_s:.3f} s")
print(f"[OK] Q_DC,Vmax = {q_vmax_dc_ah:.6f} Ah")
print(f"[OK] Q_final,DC = {q_final_dc_ah:.6f} Ah")


# =============================================================================
# 6.3 Pairwise generalized boundary assignment
# =============================================================================

assignment_rows = []

dcac_rows = df_day23_q_summary[df_day23_q_summary["protocol_role"] == "DCAC"].copy()

for _, dcac_q_row in dcac_rows.iterrows():
    dcac_file = str(dcac_q_row["file_name"])
    dcac_event = get_event_row_day23(dcac_file)
    finalq_pair = get_finalq_pair_row_day23(dcac_file)
    dcac_traj = TRAJ23[dcac_file]

    t_vmax_dcac_s = float(dcac_event["t_Vmax_detected_s"])
    q_vmax_dcac_ah = interp_Q_at_time_day23(dcac_traj, t_vmax_dcac_s)

    if not is_finite_number_day23(q_vmax_dcac_ah):
        raise ValueError(f"Could not extract Q_DCAC,Vmax for {dcac_file}")

    q_final_dcac_ah = float(dcac_q_row["Q_final_Ah"])

    boundary_order_Q = boundary_ordering_status_day23(
        q_vmax_dc_ah=q_vmax_dc_ah,
        q_vmax_dcac_ah=q_vmax_dcac_ah,
        tol_ah=Q_BOUNDARY_DEGENERATE_TOLERANCE_AH,
    )

    boundary_order_time = boundary_ordering_by_time_day23(
        t_vmax_dc_s,
        t_vmax_dcac_s,
        tol_s=1.0,
    )

    q_boundary_lo = min(q_vmax_dc_ah, q_vmax_dcac_ah)
    q_boundary_hi = max(q_vmax_dc_ah, q_vmax_dcac_ah)
    q_boundary_width = q_boundary_hi - q_boundary_lo

    q_common_final = float(finalq_pair["Q_common_final_Ah"])

    q80_common = float(finalq_pair["Q80_common_Ah"])
    q90_common = float(finalq_pair["Q90_common_Ah"])

    q80_nominal = Q80_NOMINAL_AH
    q90_nominal = Q90_NOMINAL_AH

    q80_common_region = assign_generalized_region_day23(
        q80_common,
        q_vmax_dc_ah,
        q_vmax_dcac_ah,
        q_final_dc_ah,
        q_final_dcac_ah,
    )
    q90_common_region = assign_generalized_region_day23(
        q90_common,
        q_vmax_dc_ah,
        q_vmax_dcac_ah,
        q_final_dc_ah,
        q_final_dcac_ah,
    )
    q80_nominal_region = assign_generalized_region_day23(
        q80_nominal,
        q_vmax_dc_ah,
        q_vmax_dcac_ah,
        q_final_dc_ah,
        q_final_dcac_ah,
    )
    q90_nominal_region = assign_generalized_region_day23(
        q90_nominal,
        q_vmax_dc_ah,
        q_vmax_dcac_ah,
        q_final_dc_ah,
        q_final_dcac_ah,
    )

    g0_q_lo = G0_Q_LO_AH
    g0_q_hi = q_boundary_lo

    g1_q_lo = q_boundary_lo
    g1_q_hi = q_boundary_hi

    g2_q_lo = q_boundary_hi
    g2_q_hi = q_common_final

    g0_grid_count = region_grid_count_day23(g0_q_lo, g0_q_hi, Q_GRID_STEP_AH)
    g1_grid_count = region_grid_count_day23(g1_q_lo + Q_GRID_STEP_AH, g1_q_hi, Q_GRID_STEP_AH)
    g2_grid_count = region_grid_count_day23(g2_q_lo + Q_GRID_STEP_AH, g2_q_hi, Q_GRID_STEP_AH)

    if boundary_order_Q == BOUNDARY_ORDER_DCAC_FIRST:
        boundary_interpretation = (
            "DCAC reaches Vmax at lower Q than DC. Boundary-leading effect exists, "
            "but Day23A has continued AC after Vmax, so this is not equivalent to "
            "Day21A/Day22A AC-off Segment B."
        )
    elif boundary_order_Q == BOUNDARY_ORDER_DC_FIRST:
        boundary_interpretation = (
            "DC reaches Vmax at lower Q than DCAC. This is opposite to the high-amplitude "
            "boundary-leading pathway and indicates delayed or redistributed voltage-boundary behavior."
        )
    elif boundary_order_Q == BOUNDARY_ORDER_DEGENERATE:
        boundary_interpretation = (
            "DC and DCAC reach Vmax at nearly the same Q; boundary split is degenerate."
        )
    else:
        boundary_interpretation = "Boundary ordering unresolved."

    row = {
        "protocol_pair": str(finalq_pair["protocol_pair"]),
        "file_name_DC": dc_file,
        "file_name_DCAC": dcac_file,
        "protocol_label_DC": str(finalq_pair["protocol_label_DC"]),
        "protocol_label_DCAC": str(finalq_pair["protocol_label_DCAC"]),

        "DC_C": float(finalq_pair["DC_C"]),
        "AC_C": float(finalq_pair["AC_C"]),
        "kappa": float(finalq_pair["kappa"]),
        "frequency_Hz": float(finalq_pair["frequency_Hz"]),
        "protocol_mode_status_DCAC": str(finalq_pair["protocol_mode_status_DCAC"]),

        "t_Vmax_DC_s": t_vmax_dc_s,
        "t_Vmax_DCAC_s": t_vmax_dcac_s,
        "t_Vmax_shift_s_DC_minus_DCAC": t_vmax_dc_s - t_vmax_dcac_s,
        "boundary_ordering_by_time": boundary_order_time,

        "Q_Vmax_DC_Ah": q_vmax_dc_ah,
        "Q_Vmax_DCAC_Ah": q_vmax_dcac_ah,
        "Q_Vmax_shift_Ah_DC_minus_DCAC": q_vmax_dc_ah - q_vmax_dcac_ah,
        "boundary_ordering_by_Q": boundary_order_Q,

        "Q_boundary_lo_Ah": q_boundary_lo,
        "Q_boundary_hi_Ah": q_boundary_hi,
        "Q_boundary_width_Ah": q_boundary_width,

        "Q_final_DC_Ah": q_final_dc_ah,
        "Q_final_DCAC_Ah": q_final_dcac_ah,
        "Q_common_final_Ah": q_common_final,
        "Q_final_diff_status": str(finalq_pair["Q_final_diff_status"]),
        "Q_final_diff_mAh": float(finalq_pair["Q_final_diff_mAh"]),

        "Q80_nominal_Ah": q80_nominal,
        "Q90_nominal_Ah": q90_nominal,
        "Q80_common_Ah": q80_common,
        "Q90_common_Ah": q90_common,

        "Q80_nominal_region": q80_nominal_region,
        "Q90_nominal_region": q90_nominal_region,
        "Q80_common_region": q80_common_region,
        "Q90_common_region": q90_common_region,

        "G0_Q_lo_Ah": g0_q_lo,
        "G0_Q_hi_Ah": g0_q_hi,
        "G0_Q_grid_count": g0_grid_count,

        "G1_Q_lo_Ah": g1_q_lo,
        "G1_Q_hi_Ah": g1_q_hi,
        "G1_Q_grid_count": g1_grid_count,

        "G2_Q_lo_Ah": g2_q_lo,
        "G2_Q_hi_Ah": g2_q_hi,
        "G2_Q_grid_count": g2_grid_count,

        "generalized_region_framework_status": "G0G1G2_assignment_ok",
        "continued_AC_after_Vmax_caveat": True,
        "not_same_protocol_family_as_Day21A_Day22A": True,
        "temperature_data_status": DAY23A_TEMPERATURE_STATUS,
        "boundary_interpretation": boundary_interpretation,
    }

    assignment_rows.append(row)

df_day23_g_assignment = pd.DataFrame(assignment_rows)
df_day23_g_assignment.to_csv(OUT_DAY23A_G_REGION_ASSIGNMENT, index=False)

print(f"[OK] Wrote Day23A G0/G1/G2 assignment: {OUT_DAY23A_G_REGION_ASSIGNMENT}")

display_cols = [
    "protocol_pair",
    "kappa",
    "protocol_mode_status_DCAC",
    "t_Vmax_DC_s",
    "t_Vmax_DCAC_s",
    "t_Vmax_shift_s_DC_minus_DCAC",
    "boundary_ordering_by_time",
    "Q_Vmax_DC_Ah",
    "Q_Vmax_DCAC_Ah",
    "Q_Vmax_shift_Ah_DC_minus_DCAC",
    "boundary_ordering_by_Q",
    "Q_boundary_width_Ah",
    "Q80_common_Ah",
    "Q80_common_region",
    "Q90_common_Ah",
    "Q90_common_region",
    "Q80_nominal_region",
    "Q90_nominal_region",
    "G0_Q_grid_count",
    "G1_Q_grid_count",
    "G2_Q_grid_count",
    "Q_final_diff_status",
]

print(df_day23_g_assignment[display_cols].to_string(index=False))


# =============================================================================
# 6.4 Hard guards and interpretation warnings
# =============================================================================

if len(df_day23_g_assignment) != 2:
    raise ValueError(f"Expected 2 Day23A DCAC assignment rows, got {len(df_day23_g_assignment)}")

if not (df_day23_g_assignment["protocol_mode_status_DCAC"] == PROTOCOL_MODE_CONTINUED_AC).all():
    raise ValueError("All active Day23A DCAC pairs must carry continued-AC protocol-mode caveat.")

if not (df_day23_g_assignment["generalized_region_framework_status"] == "G0G1G2_assignment_ok").all():
    raise ValueError("Day23A generalized region assignment failed.")

if (df_day23_g_assignment["G0_Q_grid_count"] < Q_GRID_MIN_COUNT_G0).any():
    print("\n[warning] At least one pair has insufficient G0 grid count for p95 residual.")

if (df_day23_g_assignment["Q_final_diff_status"] == FINAL_Q_MISMATCH_WARNING).any():
    print("\n[warning] At least one pair has final-Q mismatch warning.")
    print("          Later verdict must carry final_Q_mismatch_warning caveat.")

if (df_day23_g_assignment["boundary_ordering_by_time"] != "DCAC_first_by_time").any():
    print("\n[warning] Not all pairs are DCAC-first by time.")

if (df_day23_g_assignment["boundary_ordering_by_Q"] != BOUNDARY_ORDER_DCAC_FIRST).any():
    print("\n[warning] Not all pairs are DCAC-first by Q.")
    print("          This is scientifically important and must be preserved.")

print("[OK] Cell 6 Day23A Q_Vmax extraction and G0/G1/G2 assignment completed.")
print("[OK] No Δt(Q), geometry residual, or verdict performed.")

[OK] Loaded event audit: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step2_MJ1_0p4C_subDC_event_protocol_boundary_audit.csv
[OK] Loaded Q summary: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step3_MJ1_0p4C_subDC_Q_integration_summary.csv
[OK] Loaded final-Q audit: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step3_MJ1_0p4C_subDC_finalQ_pair_audit.csv
[OK] Loaded resolution summary: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step3A_MJ1_0p4C_subDC_resolution_floor_summary.csv
[OK] DC reference file = 0.4C DC.csv
[OK] t_DC,Vmax = 7630.600 s
[OK] Q_DC,Vmax = 2.882411 Ah
[OK] Q_final,DC = 3.333548 Ah
[OK] Wrote Day23A G0/G1/G2 assignment: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step4_MJ1_0p4C_subDC_G0G1G2_assignment.csv
            protocol_pair  kappa     protocol_mode_status_DCAC  t_Vmax_DC_s  t_Vmax_DCAC_s  t_Vmax_shift_s_DC_minus_DCAC boundary_ordering_by_time  Q_Vmax_DC_Ah  Q_Vmax_DCAC_Ah  Q_Vmax_shift_Ah_DC_minus_DCAC boundary_ordering_by_Q 

In [13]:
# Day23A Cell 7 — Δt(Q) and G0 residual audit
#
# Purpose:
# - Compute raw first-passage Δt(Q) for G0/G1/G2 and Q80/Q90 anchors
# - Compute prescribed-geometry residual only in G0
# - Compute fitted-waveform diagnostic residual only in G0
# - Preserve continued-AC-after-Vmax caveats
#
# Explicitly NOT done here:
# - No final verdict
# - No claim of non-geometric electrochemical acceleration
# - No direct Day21A/Day22A protocol-family comparison

OUT_DAY23A_DTQ_LONG = DATA_DIR / "day23A_step5_MJ1_0p4C_subDC_dtQ_Gregion_audit_long.csv"
OUT_DAY23A_DTQ_SUMMARY = DATA_DIR / "day23A_step5_MJ1_0p4C_subDC_dtQ_Gregion_summary.csv"

required_day23A_cell7_inputs = [
    OUT_DAY23A_G_REGION_ASSIGNMENT,
    OUT_DAY23A_RESOLUTION_SUMMARY,
]

missing = [p for p in required_day23A_cell7_inputs if not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        "Cannot run Day23A Cell 7. Missing required files:\n"
        + "\n".join(str(p) for p in missing)
    )

if "TRAJ23" not in globals():
    raise RuntimeError("TRAJ23 not found. Run Day23A Cell 3 first.")

df_day23_g_assignment = pd.read_csv(OUT_DAY23A_G_REGION_ASSIGNMENT)
df_day23_resolution_summary = pd.read_csv(OUT_DAY23A_RESOLUTION_SUMMARY)

DAY23A_RESOLUTION_P95_S = float(
    df_day23_resolution_summary["day23A_self_consistency_resolution_p95_s"].iloc[0]
)
DAY23A_RESOLUTION_MAX_S = float(
    df_day23_resolution_summary["day23A_self_consistency_resolution_max_s"].iloc[0]
)

print(f"[OK] Loaded G0/G1/G2 assignment: {OUT_DAY23A_G_REGION_ASSIGNMENT}")
print(f"[OK] Day23A resolution p95 = {DAY23A_RESOLUTION_P95_S:.6f} s")
print(f"[OK] Day23A resolution max = {DAY23A_RESOLUTION_MAX_S:.6f} s")


# =============================================================================
# 7.1 Helper functions
# =============================================================================

def safe_mean_day23(x):
    arr = np.asarray(x, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.mean(arr)) if len(arr) else np.nan


def safe_median_day23(x):
    arr = np.asarray(x, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.median(arr)) if len(arr) else np.nan


def safe_min_day23(x):
    arr = np.asarray(x, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.min(arr)) if len(arr) else np.nan


def safe_max_day23(x):
    arr = np.asarray(x, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.max(arr)) if len(arr) else np.nan


def safe_max_abs_day23(x):
    arr = np.asarray(x, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.max(np.abs(arr))) if len(arr) else np.nan


def safe_p95_abs_day23(x):
    arr = np.asarray(x, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.percentile(np.abs(arr), 95)) if len(arr) else np.nan


def prescribed_current_day23(t_s, I_DC_A, I_AC_A, frequency_Hz, phase_rad=0.0):
    t = np.asarray(t_s, dtype=float)

    if float(I_AC_A) == 0.0 or float(frequency_Hz) == 0.0:
        return np.full(len(t), float(I_DC_A), dtype=float)

    omega = 2.0 * np.pi * float(frequency_Hz)
    return float(I_DC_A) + float(I_AC_A) * np.sin(omega * t + float(phase_rad))


def prescribed_Q_Ah_day23(t_s, I_DC_A, I_AC_A, frequency_Hz, phase_rad=0.0):
    i_geom = prescribed_current_day23(
        t_s=t_s,
        I_DC_A=I_DC_A,
        I_AC_A=I_AC_A,
        frequency_Hz=frequency_Hz,
        phase_rad=phase_rad,
    )
    return integrate_strict_net_Q_Ah_day23(t_s, i_geom)


def fit_fixed_frequency_current_day23(t_s, i_a, frequency_Hz):
    """
    Fixed-frequency least-squares fit:

    I(t) = c0 + c1*t + a*sin(ωt) + b*cos(ωt)

    Equivalent fitted amplitude:
    A_fit = sqrt(a^2 + b^2)

    Equivalent phase:
    phi = atan2(b, a)

    The linear trend term reduces false AC amplitude estimation during slow current drift.
    """
    t = np.asarray(t_s, dtype=float)
    i = np.asarray(i_a, dtype=float)

    finite = np.isfinite(t) & np.isfinite(i)
    t = t[finite]
    i = i[finite]

    if len(t) < 20:
        return {
            "fit_status": "insufficient_samples",
            "n": int(len(t)),
            "I0_fit_A": np.nan,
            "slope_fit_A_per_s": np.nan,
            "A_fit_A": np.nan,
            "phase_fit_rad": np.nan,
            "fit_rmse_A": np.nan,
            "fit_duration_s": np.nan,
        }

    if not is_finite_number_day23(frequency_Hz) or float(frequency_Hz) <= 0:
        return {
            "fit_status": "invalid_frequency",
            "n": int(len(t)),
            "I0_fit_A": np.nan,
            "slope_fit_A_per_s": np.nan,
            "A_fit_A": np.nan,
            "phase_fit_rad": np.nan,
            "fit_rmse_A": np.nan,
            "fit_duration_s": float(np.nanmax(t) - np.nanmin(t)),
        }

    t_rel = t - t[0]
    omega = 2.0 * np.pi * float(frequency_Hz)

    X = np.column_stack([
        np.ones_like(t_rel),
        t_rel,
        np.sin(omega * t),
        np.cos(omega * t),
    ])

    try:
        beta, *_ = np.linalg.lstsq(X, i, rcond=None)
        pred = X @ beta

        c0, c1, a_sin, b_cos = beta

        A_fit = float(np.sqrt(a_sin**2 + b_cos**2))
        phi_fit = float(np.arctan2(b_cos, a_sin))
        rmse = float(np.sqrt(np.mean((i - pred) ** 2)))

        return {
            "fit_status": "ok",
            "n": int(len(t)),
            "I0_fit_A": float(c0),
            "slope_fit_A_per_s": float(c1),
            "A_fit_A": A_fit,
            "phase_fit_rad": phi_fit,
            "fit_rmse_A": rmse,
            "fit_duration_s": float(np.nanmax(t) - np.nanmin(t)),
        }

    except Exception as exc:
        return {
            "fit_status": f"fit_error:{type(exc).__name__}",
            "n": int(len(t)),
            "I0_fit_A": np.nan,
            "slope_fit_A_per_s": np.nan,
            "A_fit_A": np.nan,
            "phase_fit_rad": np.nan,
            "fit_rmse_A": np.nan,
            "fit_duration_s": float(np.nanmax(t) - np.nanmin(t)),
        }


def fitted_current_Q_Ah_day23(t_s, I0_A, A_A, frequency_Hz, phase_rad):
    t = np.asarray(t_s, dtype=float)

    if len(t) == 0:
        return np.array([], dtype=float)

    omega = 2.0 * np.pi * float(frequency_Hz)
    i_fit = float(I0_A) + float(A_A) * np.sin(omega * t + float(phase_rad))
    return integrate_strict_net_Q_Ah_day23(t, i_fit)


def g0_residual_status_day23(max_abs_s, p95_abs_s, n_points):
    if not is_finite_number_day23(max_abs_s):
        return G0_RESID_UNRESOLVED

    max_abs = abs(float(max_abs_s))

    if max_abs <= G0_RESID_FLOOR_COMPATIBLE_THRESHOLD_S:
        return G0_RESID_FLOOR_COMPATIBLE

    if max_abs < G0_RESID_REOPEN_THRESHOLD_S:
        return G0_RESID_INTERMEDIATE

    if n_points < Q_GRID_MIN_COUNT_G0 or not is_finite_number_day23(p95_abs_s):
        return G0_RESID_SPIKE

    if abs(float(p95_abs_s)) < G0_RESID_REOPEN_THRESHOLD_S:
        return G0_RESID_SPIKE

    return G0_RESID_ABOVE_FLOOR


def region_grids_day23(row):
    grid_g0 = make_fixed_Q_grid_day23(
        float(row["G0_Q_lo_Ah"]),
        float(row["G0_Q_hi_Ah"]),
        Q_GRID_STEP_AH,
    )

    grid_g1 = make_fixed_Q_grid_day23(
        float(row["G1_Q_lo_Ah"]) + Q_GRID_STEP_AH,
        float(row["G1_Q_hi_Ah"]),
        Q_GRID_STEP_AH,
    )

    grid_g2 = make_fixed_Q_grid_day23(
        float(row["G2_Q_lo_Ah"]) + Q_GRID_STEP_AH,
        float(row["G2_Q_hi_Ah"]),
        Q_GRID_STEP_AH,
    )

    return grid_g0, grid_g1, grid_g2


# =============================================================================
# 7.2 Compute Δt(Q), G0 residuals, and anchor values
# =============================================================================

dtq_rows = []
summary_rows = []

for _, pair_row in df_day23_g_assignment.iterrows():
    pair = pair_row["protocol_pair"]

    dc_file = pair_row["file_name_DC"]
    dcac_file = pair_row["file_name_DCAC"]

    dc_traj = TRAJ23[dc_file]
    dcac_traj = TRAJ23[dcac_file]

    t_dc = dc_traj["t_s"].to_numpy(dtype=float)
    q_dc = dc_traj["Q_net_Ah"].to_numpy(dtype=float)

    t_dcac = dcac_traj["t_s"].to_numpy(dtype=float)
    q_dcac = dcac_traj["Q_net_Ah"].to_numpy(dtype=float)

    I_DC_A = float(pair_row["DC_C"]) * ONE_C_A
    I_AC_A = float(pair_row["AC_C"]) * ONE_C_A
    f_hz = float(pair_row["frequency_Hz"])

    # Fit current waveform inside G0 only.
    g0_mask = (
        dcac_traj["t_s"].notna()
        & dcac_traj["Q_net_Ah"].notna()
        & dcac_traj["I_Q_A"].notna()
        & (dcac_traj["Q_net_Ah"] >= float(pair_row["G0_Q_lo_Ah"]))
        & (dcac_traj["Q_net_Ah"] <= float(pair_row["G0_Q_hi_Ah"]))
    )

    fit_g0 = fit_fixed_frequency_current_day23(
        dcac_traj.loc[g0_mask, "t_s"].to_numpy(dtype=float),
        dcac_traj.loc[g0_mask, "I_Q_A"].to_numpy(dtype=float),
        f_hz,
    )

    phase_for_prescribed = (
        fit_g0["phase_fit_rad"]
        if fit_g0["fit_status"] == "ok" and is_finite_number_day23(fit_g0["phase_fit_rad"])
        else 0.0
    )

    phase_status = (
        GEOM_PHASE_VERIFIED
        if is_finite_number_day23(phase_for_prescribed) and abs(float(phase_for_prescribed)) <= 0.05
        else GEOM_PHASE_ESTIMATED
    )

    # Formal prescribed geometry
    q_geom_dc = prescribed_Q_Ah_day23(
        t_s=t_dc,
        I_DC_A=I_DC_A,
        I_AC_A=0.0,
        frequency_Hz=0.0,
        phase_rad=0.0,
    )

    q_geom_dcac = prescribed_Q_Ah_day23(
        t_s=t_dcac,
        I_DC_A=I_DC_A,
        I_AC_A=I_AC_A,
        frequency_Hz=f_hz,
        phase_rad=phase_for_prescribed,
    )

    # Diagnostic fitted waveform geometry
    if fit_g0["fit_status"] == "ok":
        q_geom_dcac_fit = fitted_current_Q_Ah_day23(
            t_s=t_dcac,
            I0_A=fit_g0["I0_fit_A"],
            A_A=fit_g0["A_fit_A"],
            frequency_Hz=f_hz,
            phase_rad=fit_g0["phase_fit_rad"],
        )
    else:
        q_geom_dcac_fit = np.full(len(t_dcac), np.nan)

    grid_g0, grid_g1, grid_g2 = region_grids_day23(pair_row)

    anchors = {
        "Q80_common": (float(pair_row["Q80_common_Ah"]), pair_row["Q80_common_region"]),
        "Q90_common": (float(pair_row["Q90_common_Ah"]), pair_row["Q90_common_region"]),
        "Q80_nominal": (float(pair_row["Q80_nominal_Ah"]), pair_row["Q80_nominal_region"]),
        "Q90_nominal": (float(pair_row["Q90_nominal_Ah"]), pair_row["Q90_nominal_region"]),
    }

    def add_dtq_row(q_target, q_label, region_label, is_anchor):
        t_dc_q = first_passage_time_from_Q_day23(q_target, t_dc, q_dc)
        t_dcac_q = first_passage_time_from_Q_day23(q_target, t_dcac, q_dcac)

        dt_raw = (
            t_dc_q - t_dcac_q
            if is_finite_number_day23(t_dc_q) and is_finite_number_day23(t_dcac_q)
            else np.nan
        )

        if region_label == REGION_G0:
            t_geom_dc_q = first_passage_time_from_Q_day23(q_target, t_dc, q_geom_dc)
            t_geom_dcac_q = first_passage_time_from_Q_day23(q_target, t_dcac, q_geom_dcac)

            dt_geom = (
                t_geom_dc_q - t_geom_dcac_q
                if is_finite_number_day23(t_geom_dc_q) and is_finite_number_day23(t_geom_dcac_q)
                else np.nan
            )

            dt_resid = (
                dt_raw - dt_geom
                if is_finite_number_day23(dt_raw) and is_finite_number_day23(dt_geom)
                else np.nan
            )

            t_geom_dcac_fit_q = first_passage_time_from_Q_day23(
                q_target,
                t_dcac,
                q_geom_dcac_fit,
            )

            dt_geom_fit = (
                t_geom_dc_q - t_geom_dcac_fit_q
                if is_finite_number_day23(t_geom_dc_q) and is_finite_number_day23(t_geom_dcac_fit_q)
                else np.nan
            )

            dt_resid_fit = (
                dt_raw - dt_geom_fit
                if is_finite_number_day23(dt_raw) and is_finite_number_day23(dt_geom_fit)
                else np.nan
            )

        else:
            t_geom_dc_q = np.nan
            t_geom_dcac_q = np.nan
            dt_geom = np.nan
            dt_resid = np.nan
            dt_geom_fit = np.nan
            dt_resid_fit = np.nan

        dtq_rows.append({
            "protocol_pair": pair,
            "protocol_label_DCAC": pair_row["protocol_label_DCAC"],
            "file_name_DCAC": dcac_file,
            "kappa": float(pair_row["kappa"]),
            "Q_label": q_label,
            "Q_Ah": float(q_target),
            "is_anchor": bool(is_anchor),
            "generalized_region": region_label,

            "t_DC_s": t_dc_q,
            "t_DCAC_s": t_dcac_q,
            "dt_raw_s": dt_raw,

            "t_geom_DC_s": t_geom_dc_q,
            "t_geom_DCAC_s": t_geom_dcac_q,
            "dt_geom_s": dt_geom,
            "dt_resid_s": dt_resid,

            "dt_geom_fit_s": dt_geom_fit,
            "dt_resid_fit_s": dt_resid_fit,

            "boundary_ordering_by_Q": pair_row["boundary_ordering_by_Q"],
            "boundary_ordering_by_time": pair_row["boundary_ordering_by_time"],
            "protocol_mode_status_DCAC": pair_row["protocol_mode_status_DCAC"],

            "geometry_phase_reference_status": phase_status,
            "geometry_phase_offset_rad": phase_for_prescribed,

            "fit_status": fit_g0["fit_status"],
            "I0_fit_A": fit_g0["I0_fit_A"],
            "A_fit_A": fit_g0["A_fit_A"],
            "A_fit_ratio_to_expected": (
                fit_g0["A_fit_A"] / I_AC_A
                if I_AC_A > 0 and is_finite_number_day23(fit_g0["A_fit_A"])
                else np.nan
            ),
            "fit_rmse_A": fit_g0["fit_rmse_A"],

            "continued_AC_after_Vmax_caveat": True,
            "not_same_protocol_family_as_Day21A_Day22A": True,
        })

    for q in grid_g0:
        add_dtq_row(q, "G0_grid", REGION_G0, False)

    for q in grid_g1:
        add_dtq_row(q, "G1_grid", REGION_G1, False)

    for q in grid_g2:
        add_dtq_row(q, "G2_grid", REGION_G2, False)

    for label, (q, region) in anchors.items():
        add_dtq_row(q, label, region, True)

    df_pair = pd.DataFrame([r for r in dtq_rows if r["protocol_pair"] == pair])

    g0 = df_pair[(df_pair["generalized_region"] == REGION_G0) & (~df_pair["is_anchor"])]
    g1 = df_pair[(df_pair["generalized_region"] == REGION_G1) & (~df_pair["is_anchor"])]
    g2 = df_pair[(df_pair["generalized_region"] == REGION_G2) & (~df_pair["is_anchor"])]

    g0_resid = g0["dt_resid_s"].dropna().to_numpy(dtype=float)
    g0_resid_fit = g0["dt_resid_fit_s"].dropna().to_numpy(dtype=float)

    g0_count = int(len(g0))

    g0_resid_mean = safe_mean_day23(g0_resid)
    g0_resid_p95_abs = safe_p95_abs_day23(g0_resid) if g0_count >= Q_GRID_MIN_COUNT_G0 else np.nan
    g0_resid_max_abs = safe_max_abs_day23(g0_resid)

    g0_fit_p95_abs = safe_p95_abs_day23(g0_resid_fit) if g0_count >= Q_GRID_MIN_COUNT_G0 else np.nan
    g0_fit_max_abs = safe_max_abs_day23(g0_resid_fit)

    g0_status = g0_residual_status_day23(
        max_abs_s=g0_resid_max_abs,
        p95_abs_s=g0_resid_p95_abs,
        n_points=g0_count,
    )

    audit_resolution_status = (
        "prescribed_p95_above_self_consistency_resolution"
        if is_finite_number_day23(g0_resid_p95_abs) and g0_resid_p95_abs > DAY23A_RESOLUTION_P95_S
        else "prescribed_p95_below_or_equal_self_consistency_resolution"
    )

    fitted_resolution_status = (
        "fitted_p95_above_self_consistency_resolution"
        if is_finite_number_day23(g0_fit_p95_abs) and g0_fit_p95_abs > DAY23A_RESOLUTION_P95_S
        else "fitted_p95_below_or_equal_self_consistency_resolution"
    )

    anchor_rows = df_pair[df_pair["is_anchor"]].copy()
    anchor_dt = {
        row["Q_label"]: row["dt_raw_s"]
        for _, row in anchor_rows.iterrows()
    }

    summary_rows.append({
        "protocol_pair": pair,
        "protocol_label_DCAC": pair_row["protocol_label_DCAC"],
        "kappa": float(pair_row["kappa"]),
        "boundary_ordering_by_Q": pair_row["boundary_ordering_by_Q"],
        "boundary_ordering_by_time": pair_row["boundary_ordering_by_time"],
        "protocol_mode_status_DCAC": pair_row["protocol_mode_status_DCAC"],

        "Q_Vmax_shift_Ah_DC_minus_DCAC": float(pair_row["Q_Vmax_shift_Ah_DC_minus_DCAC"]),
        "t_Vmax_shift_s_DC_minus_DCAC": float(pair_row["t_Vmax_shift_s_DC_minus_DCAC"]),

        "Q80_common_region": pair_row["Q80_common_region"],
        "Q90_common_region": pair_row["Q90_common_region"],
        "Q80_nominal_region": pair_row["Q80_nominal_region"],
        "Q90_nominal_region": pair_row["Q90_nominal_region"],

        "dt_Q80_common_raw_s": anchor_dt.get("Q80_common", np.nan),
        "dt_Q90_common_raw_s": anchor_dt.get("Q90_common", np.nan),
        "dt_Q80_nominal_raw_s": anchor_dt.get("Q80_nominal", np.nan),
        "dt_Q90_nominal_raw_s": anchor_dt.get("Q90_nominal", np.nan),

        "G0_Q_grid_count": g0_count,
        "G0_dt_raw_median_s": safe_median_day23(g0["dt_raw_s"]),
        "G0_dt_raw_max_s": safe_max_day23(g0["dt_raw_s"]),
        "G0_dt_resid_mean_s": g0_resid_mean,
        "G0_dt_resid_p95_abs_s": g0_resid_p95_abs,
        "G0_dt_resid_max_abs_s": g0_resid_max_abs,
        "G0_residual_status": g0_status,

        "G0_dt_resid_fit_p95_abs_s": g0_fit_p95_abs,
        "G0_dt_resid_fit_max_abs_s": g0_fit_max_abs,

        "day23A_self_consistency_resolution_p95_s": DAY23A_RESOLUTION_P95_S,
        "day23A_self_consistency_resolution_max_s": DAY23A_RESOLUTION_MAX_S,
        "audit_resolution_status": audit_resolution_status,
        "fitted_resolution_status": fitted_resolution_status,

        "G1_Q_grid_count": int(len(g1)),
        "G1_dt_raw_median_s": safe_median_day23(g1["dt_raw_s"]),
        "G1_dt_raw_max_s": safe_max_day23(g1["dt_raw_s"]),

        "G2_Q_grid_count": int(len(g2)),
        "G2_dt_raw_median_s": safe_median_day23(g2["dt_raw_s"]),
        "G2_dt_raw_max_s": safe_max_day23(g2["dt_raw_s"]),

        "geometry_phase_reference_status": phase_status,
        "geometry_phase_offset_rad": phase_for_prescribed,
        "fit_status": fit_g0["fit_status"],
        "I0_fit_A": fit_g0["I0_fit_A"],
        "A_fit_A": fit_g0["A_fit_A"],
        "A_fit_ratio_to_expected": (
            fit_g0["A_fit_A"] / I_AC_A
            if I_AC_A > 0 and is_finite_number_day23(fit_g0["A_fit_A"])
            else np.nan
        ),
        "fit_rmse_A": fit_g0["fit_rmse_A"],

        "Q_final_diff_status": pair_row["Q_final_diff_status"],
        "continued_AC_after_Vmax_caveat": True,
        "not_same_protocol_family_as_Day21A_Day22A": True,
        "temperature_data_status": DAY23A_TEMPERATURE_STATUS,
    })


df_day23_dtq_long = pd.DataFrame(dtq_rows)
df_day23_dtq_summary = pd.DataFrame(summary_rows)

df_day23_dtq_long.to_csv(OUT_DAY23A_DTQ_LONG, index=False)
df_day23_dtq_summary.to_csv(OUT_DAY23A_DTQ_SUMMARY, index=False)

print(f"[OK] Wrote Day23A Δt(Q) long table: {OUT_DAY23A_DTQ_LONG}")
print(f"[OK] Wrote Day23A Δt(Q) summary: {OUT_DAY23A_DTQ_SUMMARY}")

display_cols = [
    "protocol_pair",
    "kappa",
    "boundary_ordering_by_Q",
    "protocol_mode_status_DCAC",
    "Q80_common_region",
    "Q90_common_region",
    "dt_Q80_common_raw_s",
    "dt_Q90_common_raw_s",
    "G0_Q_grid_count",
    "G0_dt_resid_mean_s",
    "G0_dt_resid_p95_abs_s",
    "G0_dt_resid_max_abs_s",
    "G0_residual_status",
    "G0_dt_resid_fit_p95_abs_s",
    "audit_resolution_status",
    "fitted_resolution_status",
    "G1_dt_raw_median_s",
    "G2_dt_raw_median_s",
    "A_fit_ratio_to_expected",
    "Q_final_diff_status",
]

print(df_day23_dtq_summary[display_cols].to_string(index=False))


# =============================================================================
# 7.3 Hard guards and warnings
# =============================================================================

if len(df_day23_dtq_summary) != 2:
    raise ValueError(f"Expected 2 Day23A dtQ summary rows, got {len(df_day23_dtq_summary)}")

if not (df_day23_dtq_summary["protocol_mode_status_DCAC"] == PROTOCOL_MODE_CONTINUED_AC).all():
    raise ValueError("Day23A dtQ summary lost continued-AC protocol-mode caveat.")

if not (df_day23_dtq_summary["Q80_common_region"] == REGION_G0).all():
    print("\n[warning] Not all Q80_common anchors are in G0.")

if not (df_day23_dtq_summary["Q90_common_region"] == REGION_G2).all():
    print("\n[warning] Not all Q90_common anchors are in G2.")

if (df_day23_dtq_summary["G0_Q_grid_count"] < Q_GRID_MIN_COUNT_G0).any():
    print("\n[warning] At least one pair has insufficient G0 grid count for p95 residual.")

if (df_day23_dtq_summary["Q_final_diff_status"] == FINAL_Q_MISMATCH_WARNING).any():
    print("\n[warning] Final-Q mismatch warning present in Day23A dtQ summary.")
    print("          Later verdict must carry final_Q_mismatch_warning caveat.")

print("[OK] Cell 7 Day23A Δt(Q) and G0 residual audit completed.")
print("[OK] No final verdict performed.")

[OK] Loaded G0/G1/G2 assignment: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step4_MJ1_0p4C_subDC_G0G1G2_assignment.csv
[OK] Day23A resolution p95 = 1.612597 s
[OK] Day23A resolution max = 11.990032 s
[OK] Wrote Day23A Δt(Q) long table: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step5_MJ1_0p4C_subDC_dtQ_Gregion_audit_long.csv
[OK] Wrote Day23A Δt(Q) summary: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step5_MJ1_0p4C_subDC_dtQ_Gregion_summary.csv
            protocol_pair  kappa boundary_ordering_by_Q     protocol_mode_status_DCAC      Q80_common_region Q90_common_region  dt_Q80_common_raw_s  dt_Q90_common_raw_s  G0_Q_grid_count  G0_dt_resid_mean_s  G0_dt_resid_p95_abs_s  G0_dt_resid_max_abs_s                              G0_residual_status  G0_dt_resid_fit_p95_abs_s                          audit_resolution_status                     fitted_resolution_status  G1_dt_raw_median_s  G2_dt_raw_median_s  A_fit_ratio_to_expected      Q_final_diff_status
0.4C DC vs 0.4

In [14]:
# Day23A Cell 8 — generalized G0/G1/G2 verdict
#
# Purpose:
# - Produce formal generalized verdicts for Day23A
# - Preserve continued-AC-after-Vmax caveat
# - Separate boundary-leading event from state-equivalent gain persistence
# - Avoid Day21A/Day22A AC-off Segment B interpretation
#
# Explicitly NOT done here:
# - No new computation
# - No threshold redefinition
# - No claim of confirmed non-geometric electrochemical acceleration
# - No direct same-protocol-family comparison to Day21A/Day22A

OUT_DAY23A_VERDICT = DATA_DIR / "day23A_step6_MJ1_0p4C_subDC_generalized_verdict.csv"

required_day23A_cell8_inputs = [
    OUT_DAY23A_DTQ_SUMMARY,
    OUT_DAY23A_G_REGION_ASSIGNMENT,
    OUT_DAY23A_RESOLUTION_SUMMARY,
    OUT_DAY23A_FINALQ_PAIR_AUDIT,
    OUT_DAY23A_PROTOCOL_MODE_AUDIT,
    OUT_DAY23A_FRAMEWORK_DECISION,
]

missing = [p for p in required_day23A_cell8_inputs if not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        "Cannot run Day23A Cell 8. Missing required files:\n"
        + "\n".join(str(p) for p in missing)
    )

df_day23_dtq_summary = pd.read_csv(OUT_DAY23A_DTQ_SUMMARY)
df_day23_g_assignment = pd.read_csv(OUT_DAY23A_G_REGION_ASSIGNMENT)
df_day23_resolution = pd.read_csv(OUT_DAY23A_RESOLUTION_SUMMARY)
df_day23_finalq = pd.read_csv(OUT_DAY23A_FINALQ_PAIR_AUDIT)
df_day23_protocol_mode = pd.read_csv(OUT_DAY23A_PROTOCOL_MODE_AUDIT)
df_day23_framework = pd.read_csv(OUT_DAY23A_FRAMEWORK_DECISION)

DAY23A_RESOLUTION_P95_S = float(
    df_day23_resolution["day23A_self_consistency_resolution_p95_s"].iloc[0]
)
DAY23A_RESOLUTION_MAX_S = float(
    df_day23_resolution["day23A_self_consistency_resolution_max_s"].iloc[0]
)

print(f"[OK] Loaded Day23A Δt summary: {OUT_DAY23A_DTQ_SUMMARY}")
print(f"[OK] Loaded G assignment: {OUT_DAY23A_G_REGION_ASSIGNMENT}")
print(f"[OK] Day23A resolution p95 = {DAY23A_RESOLUTION_P95_S:.6f} s")


# =============================================================================
# 8.1 Helpers
# =============================================================================

def append_caveat_day23(existing, new):
    if new is None or str(new).strip() == "":
        return "" if pd.isna(existing) else str(existing)

    if existing is None or pd.isna(existing) or str(existing).strip() in ["", "nan", "NaN", "<NA>"]:
        items = []
    else:
        items = [x.strip() for x in str(existing).split(";") if x.strip()]

    if new not in items:
        items.append(new)

    return ";".join(items)


def one_row_day23(df, mask, label):
    rows = df.loc[mask]
    if len(rows) != 1:
        raise ValueError(f"Expected exactly one row for {label}, found {len(rows)}")
    return rows.iloc[0]


def raw_anchor_resolution_status_day23(dt_s, resolution_p95_s):
    if not is_finite_number_day23(dt_s):
        return "raw_anchor_unresolved"

    if abs(float(dt_s)) <= float(resolution_p95_s):
        return "raw_anchor_within_self_consistency_resolution"

    if float(dt_s) > float(resolution_p95_s):
        return "raw_anchor_positive_above_resolution"

    return "raw_anchor_negative_above_resolution"


def day23_generalized_verdict(dtq_row):
    """
    Day23A-specific generalized verdict logic.

    This logic does not claim Segment-A non-geometric acceleration.
    It classifies:
    - boundary ordering;
    - G0 residual status;
    - whether raw gain persists to Q90/G2;
    - whether effect is below audit resolution.
    """
    q80_raw = dtq_row["dt_Q80_common_raw_s"]
    q90_raw = dtq_row["dt_Q90_common_raw_s"]

    q80_status = raw_anchor_resolution_status_day23(q80_raw, DAY23A_RESOLUTION_P95_S)
    q90_status = raw_anchor_resolution_status_day23(q90_raw, DAY23A_RESOLUTION_P95_S)

    boundary_order = str(dtq_row["boundary_ordering_by_Q"])
    g0_status = str(dtq_row["G0_residual_status"])
    protocol_mode = str(dtq_row["protocol_mode_status_DCAC"])

    if protocol_mode != PROTOCOL_MODE_CONTINUED_AC:
        return {
            "evidence_status": "ambiguous",
            "mechanism_verdict": "ambiguous_protocol_mode_not_continued_AC",
            "interpretation_class": "protocol_mode_unexpected",
            "Q80_raw_resolution_status": q80_status,
            "Q90_raw_resolution_status": q90_status,
        }

    if boundary_order != BOUNDARY_ORDER_DCAC_FIRST:
        return {
            "evidence_status": "ambiguous",
            "mechanism_verdict": "ambiguous_non_DCAC_first_boundary_order",
            "interpretation_class": "non_DCAC_first_or_unresolved_boundary_order",
            "Q80_raw_resolution_status": q80_status,
            "Q90_raw_resolution_status": q90_status,
        }

    # κ low: both Q80 and Q90 are within resolution -> no resolved state gain.
    if (
        q80_status == "raw_anchor_within_self_consistency_resolution"
        and q90_status == "raw_anchor_within_self_consistency_resolution"
    ):
        return {
            "evidence_status": "weak_or_unresolved",
            "mechanism_verdict": "boundary_leading_but_state_gain_unresolved_at_common_anchors",
            "interpretation_class": "subDC_small_perturbation_near_resolution",
            "Q80_raw_resolution_status": q80_status,
            "Q90_raw_resolution_status": q90_status,
        }

    # Early gain but high-Q penalty.
    if (
        q80_status == "raw_anchor_positive_above_resolution"
        and q90_status == "raw_anchor_negative_above_resolution"
    ):
        return {
            "evidence_status": "mixed",
            "mechanism_verdict": "boundary_leading_with_G0_gain_but_post_boundary_penalty",
            "interpretation_class": "boundary_leading_not_gain_preserving",
            "Q80_raw_resolution_status": q80_status,
            "Q90_raw_resolution_status": q90_status,
        }

    # Positive Q90 but not necessarily Q80.
    if q90_status == "raw_anchor_positive_above_resolution":
        return {
            "evidence_status": "partial_support_with_caveats",
            "mechanism_verdict": "positive_post_boundary_raw_gain_under_continued_AC",
            "interpretation_class": "continued_AC_post_boundary_gain_not_ACoff_segmentB",
            "Q80_raw_resolution_status": q80_status,
            "Q90_raw_resolution_status": q90_status,
        }

    # Negative Q90 dominates.
    if q90_status == "raw_anchor_negative_above_resolution":
        return {
            "evidence_status": "not_supported_as_full_protocol_gain",
            "mechanism_verdict": "boundary_leading_but_negative_highQ_raw_gain",
            "interpretation_class": "boundary_event_leads_but_state_equivalent_gain_reverses",
            "Q80_raw_resolution_status": q80_status,
            "Q90_raw_resolution_status": q90_status,
        }

    return {
        "evidence_status": "ambiguous",
        "mechanism_verdict": "ambiguous_generalized_boundary_result",
        "interpretation_class": "unclassified_generalized_G0G1G2_pattern",
        "Q80_raw_resolution_status": q80_status,
        "Q90_raw_resolution_status": q90_status,
    }


# =============================================================================
# 8.2 Build verdict table
# =============================================================================

verdict_rows = []

for _, dtq_row in df_day23_dtq_summary.iterrows():
    pair = dtq_row["protocol_pair"]

    g_row = one_row_day23(
        df_day23_g_assignment,
        df_day23_g_assignment["protocol_pair"] == pair,
        f"G_assignment:{pair}",
    )

    finalq_row = one_row_day23(
        df_day23_finalq,
        df_day23_finalq["protocol_pair"] == pair,
        f"finalQ:{pair}",
    )

    formal = day23_generalized_verdict(dtq_row)

    caveat = ""
    caveat = append_caveat_day23(caveat, DAY23A_CONTINUED_AC_CAVEAT)
    caveat = append_caveat_day23(caveat, DAY23A_NOT_SAME_PROTOCOL_FAMILY_CAVEAT)
    caveat = append_caveat_day23(caveat, DAY23A_TEMPERATURE_CAVEAT)
    caveat = append_caveat_day23(caveat, DAY23A_NOISE_FLOOR_CAVEAT)
    caveat = append_caveat_day23(caveat, DAY23A_EFFECT_SIZE_LIMITATION)

    if str(finalq_row["Q_final_diff_status"]) == FINAL_Q_MISMATCH_WARNING:
        caveat = append_caveat_day23(caveat, "final_Q_mismatch_warning")

    if str(dtq_row["G0_residual_status"]) == G0_RESID_ABOVE_FLOOR:
        caveat = append_caveat_day23(caveat, "G0_prescribed_residual_above_floor")

    if str(dtq_row["G0_residual_status"]) == G0_RESID_INTERMEDIATE:
        caveat = append_caveat_day23(caveat, "G0_prescribed_residual_intermediate")

    if str(dtq_row["fitted_resolution_status"]) == "fitted_p95_above_self_consistency_resolution":
        caveat = append_caveat_day23(caveat, "fitted_G0_residual_above_self_consistency_resolution")

    if formal["Q90_raw_resolution_status"] == "raw_anchor_negative_above_resolution":
        caveat = append_caveat_day23(caveat, "Q90_common_negative_raw_gain")

    if formal["Q80_raw_resolution_status"] == "raw_anchor_within_self_consistency_resolution":
        caveat = append_caveat_day23(caveat, "Q80_common_within_audit_resolution")

    if formal["Q90_raw_resolution_status"] == "raw_anchor_within_self_consistency_resolution":
        caveat = append_caveat_day23(caveat, "Q90_common_within_audit_resolution")

    row = {
        "source_type": SOURCE_TYPE_MJ1,
        "cell_or_param_set": CELL_ID,
        "group_id": DAY23A_GROUP_ID,
        "protocol_pair": pair,
        "protocol_label_DC": g_row["protocol_label_DC"],
        "protocol_label_DCAC": g_row["protocol_label_DCAC"],

        "DC_C": float(g_row["DC_C"]),
        "AC_C": float(g_row["AC_C"]),
        "kappa": float(g_row["kappa"]),
        "m_tau": 1.0,
        "frequency_Hz": float(g_row["frequency_Hz"]),

        "protocol_mode_status_DCAC": g_row["protocol_mode_status_DCAC"],
        "boundary_ordering_by_Q": g_row["boundary_ordering_by_Q"],
        "boundary_ordering_by_time": g_row["boundary_ordering_by_time"],

        "Q_Vmax_DC_Ah": float(g_row["Q_Vmax_DC_Ah"]),
        "Q_Vmax_DCAC_Ah": float(g_row["Q_Vmax_DCAC_Ah"]),
        "Q_Vmax_shift_Ah_DC_minus_DCAC": float(g_row["Q_Vmax_shift_Ah_DC_minus_DCAC"]),
        "t_Vmax_shift_s_DC_minus_DCAC": float(g_row["t_Vmax_shift_s_DC_minus_DCAC"]),

        "Q_final_diff_status": finalq_row["Q_final_diff_status"],
        "Q_final_diff_mAh": float(finalq_row["Q_final_diff_mAh"]),

        "Q80_common_Ah": float(g_row["Q80_common_Ah"]),
        "Q90_common_Ah": float(g_row["Q90_common_Ah"]),
        "Q80_common_region": g_row["Q80_common_region"],
        "Q90_common_region": g_row["Q90_common_region"],
        "Q80_nominal_region": g_row["Q80_nominal_region"],
        "Q90_nominal_region": g_row["Q90_nominal_region"],

        "dt_Q80_common_raw_s": float(dtq_row["dt_Q80_common_raw_s"]),
        "dt_Q90_common_raw_s": float(dtq_row["dt_Q90_common_raw_s"]),
        "Q80_raw_resolution_status": formal["Q80_raw_resolution_status"],
        "Q90_raw_resolution_status": formal["Q90_raw_resolution_status"],

        "G0_residual_status": dtq_row["G0_residual_status"],
        "G0_dt_resid_mean_s": float(dtq_row["G0_dt_resid_mean_s"]),
        "G0_dt_resid_p95_abs_s": float(dtq_row["G0_dt_resid_p95_abs_s"]),
        "G0_dt_resid_max_abs_s": float(dtq_row["G0_dt_resid_max_abs_s"]),
        "G0_dt_resid_fit_p95_abs_s": float(dtq_row["G0_dt_resid_fit_p95_abs_s"]),
        "G0_dt_resid_fit_max_abs_s": np.nan,

        "day23A_self_consistency_resolution_p95_s": DAY23A_RESOLUTION_P95_S,
        "day23A_self_consistency_resolution_max_s": DAY23A_RESOLUTION_MAX_S,
        "audit_resolution_status": dtq_row["audit_resolution_status"],
        "fitted_resolution_status": dtq_row["fitted_resolution_status"],

        "G1_dt_raw_median_s": float(dtq_row["G1_dt_raw_median_s"]),
        "G2_dt_raw_median_s": float(dtq_row["G2_dt_raw_median_s"]),

        "fit_status": dtq_row["fit_status"],
        "A_fit_ratio_to_expected": float(dtq_row["A_fit_ratio_to_expected"]),
        "temperature_data_status": dtq_row["temperature_data_status"],

        "evidence_status": formal["evidence_status"],
        "mechanism_verdict": formal["mechanism_verdict"],
        "interpretation_class": formal["interpretation_class"],
        "caveat": caveat,
    }

    verdict_rows.append(row)

df_day23_verdict = pd.DataFrame(verdict_rows)
df_day23_verdict.to_csv(OUT_DAY23A_VERDICT, index=False)

print(f"[OK] Wrote Day23A generalized verdict table: {OUT_DAY23A_VERDICT}")

display_cols = [
    "protocol_pair",
    "kappa",
    "evidence_status",
    "mechanism_verdict",
    "interpretation_class",
    "boundary_ordering_by_Q",
    "protocol_mode_status_DCAC",
    "Q80_common_region",
    "Q90_common_region",
    "dt_Q80_common_raw_s",
    "dt_Q90_common_raw_s",
    "Q80_raw_resolution_status",
    "Q90_raw_resolution_status",
    "G0_residual_status",
    "G0_dt_resid_p95_abs_s",
    "G0_dt_resid_fit_p95_abs_s",
    "caveat",
]

print(df_day23_verdict[display_cols].to_string(index=False))

print("[OK] Cell 8 Day23A generalized verdict completed.")
print("[OK] No claim of confirmed non-geometric Segment-A/G0 electrochemical acceleration is made.")
print("[OK] Day23A remains a different protocol-mode family from Day21A/Day22A.")

[OK] Loaded Day23A Δt summary: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step5_MJ1_0p4C_subDC_dtQ_Gregion_summary.csv
[OK] Loaded G assignment: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step4_MJ1_0p4C_subDC_G0G1G2_assignment.csv
[OK] Day23A resolution p95 = 1.612597 s
[OK] Wrote Day23A generalized verdict table: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step6_MJ1_0p4C_subDC_generalized_verdict.csv
            protocol_pair  kappa    evidence_status                                            mechanism_verdict                     interpretation_class boundary_ordering_by_Q     protocol_mode_status_DCAC      Q80_common_region Q90_common_region  dt_Q80_common_raw_s  dt_Q90_common_raw_s                     Q80_raw_resolution_status                     Q90_raw_resolution_status                              G0_residual_status  G0_dt_resid_p95_abs_s  G0_dt_resid_fit_p95_abs_s                                                                                          

In [15]:
# Day23A Cell 9A — closure CSV
#
# Purpose:
# - Close Day23A in machine-readable form
# - Preserve generalized G0/G1/G2 verdict
# - Preserve protocol-mode caveats
#
# Explicitly NOT done here:
# - No recomputation
# - No new thresholds
# - No new mechanism claim

required_day23A_closure_files = [
    OUT_DAY23A_FORMAT_INVENTORY,
    OUT_DAY23A_ACTIVE_FORMAT_INVENTORY,
    OUT_DAY23A_EXCLUSION_AUDIT,
    OUT_DAY23A_TIMEBASE_AUDIT,
    OUT_DAY23A_PROTOCOL_MODE_AUDIT,
    OUT_DAY23A_FRAMEWORK_DECISION,
    OUT_DAY23A_AUDIT_CONTRACT_JSON,
    OUT_DAY23A_FILE_INVENTORY,
    OUT_DAY23A_LOAD_SUMMARY,
    OUT_DAY23A_EVENT_AUDIT,
    OUT_DAY23A_Q_SUMMARY,
    OUT_DAY23A_FINALQ_PAIR_AUDIT,
    OUT_DAY23A_RESOLUTION_SUMMARY,
    OUT_DAY23A_G_REGION_ASSIGNMENT,
    OUT_DAY23A_DTQ_SUMMARY,
    OUT_DAY23A_VERDICT,
]

missing = [p for p in required_day23A_closure_files if not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        "Cannot close Day23A. Missing required files:\n"
        + "\n".join(str(p) for p in missing)
    )

df_day23_verdict = pd.read_csv(OUT_DAY23A_VERDICT)
df_day23_g = pd.read_csv(OUT_DAY23A_G_REGION_ASSIGNMENT)
df_day23_dtq = pd.read_csv(OUT_DAY23A_DTQ_SUMMARY)
df_day23_resolution = pd.read_csv(OUT_DAY23A_RESOLUTION_SUMMARY)
df_day23_framework = pd.read_csv(OUT_DAY23A_FRAMEWORK_DECISION)
df_day23_exclusion = pd.read_csv(OUT_DAY23A_EXCLUSION_AUDIT)

closure_cols = [
    "protocol_pair",
    "kappa",
    "evidence_status",
    "mechanism_verdict",
    "interpretation_class",
    "boundary_ordering_by_Q",
    "protocol_mode_status_DCAC",
    "Q80_common_region",
    "Q90_common_region",
    "dt_Q80_common_raw_s",
    "dt_Q90_common_raw_s",
    "Q80_raw_resolution_status",
    "Q90_raw_resolution_status",
    "G0_residual_status",
    "G0_dt_resid_p95_abs_s",
    "G0_dt_resid_fit_p95_abs_s",
    "audit_resolution_status",
    "fitted_resolution_status",
    "Q_final_diff_status",
    "caveat",
]

df_day23_closure = df_day23_verdict[closure_cols].copy()
df_day23_closure.to_csv(OUT_DAY23A_CLOSURE_CSV, index=False)

print(f"[OK] Wrote Day23A closure CSV: {OUT_DAY23A_CLOSURE_CSV}")
print(f"[OK] closure rows = {df_day23_closure.shape[0]}")
print(df_day23_closure.to_string(index=False))

print("[OK] Cell 9A Day23A closure CSV completed.")

[OK] Wrote Day23A closure CSV: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step7_closure_summary.csv
[OK] closure rows = 2
            protocol_pair  kappa    evidence_status                                            mechanism_verdict                     interpretation_class boundary_ordering_by_Q     protocol_mode_status_DCAC      Q80_common_region Q90_common_region  dt_Q80_common_raw_s  dt_Q90_common_raw_s                     Q80_raw_resolution_status                     Q90_raw_resolution_status                              G0_residual_status  G0_dt_resid_p95_abs_s  G0_dt_resid_fit_p95_abs_s                          audit_resolution_status                     fitted_resolution_status      Q_final_diff_status                                                                                                                                                                                                                                                                               

In [16]:
# Day23A Cell 9B — closure Markdown note
#
# Purpose:
# - Write human-readable Day23A closure note
# - Preserve the framework switch and generalized interpretation
#
# Explicitly NOT done here:
# - No recomputation
# - No new thresholds
# - No claim of confirmed non-geometric G0 electrochemical acceleration

now_utc = datetime.now(timezone.utc).isoformat()

closure_table = df_day23_closure.to_string(index=False)

framework_table = df_day23_framework.to_string(index=False)

exclusion_table = df_day23_exclusion[[
    "file_name",
    "protocol_label",
    "protocol_role",
    "use_for_audit",
    "exclusion_status",
]].to_string(index=False)

g_table = df_day23_g[[
    "protocol_pair",
    "kappa",
    "Q_Vmax_DC_Ah",
    "Q_Vmax_DCAC_Ah",
    "Q_Vmax_shift_Ah_DC_minus_DCAC",
    "boundary_ordering_by_Q",
    "Q80_common_region",
    "Q90_common_region",
    "Q_final_diff_status",
]].to_string(index=False)

resolution_table = df_day23_resolution.to_string(index=False)

lines = []

lines.append("# Day23A Closure Note — MJ1 0.4C Sub-DC-Amplitude Protocol-Mode Audit")
lines.append("")
lines.append(f"Generated: `{now_utc}`")
lines.append(f"Git HEAD: `{GIT_HEAD_DAY23A}`")
lines.append(f"Notebook: `{DAY23A_NOTEBOOK_NAME}`")
lines.append("")
lines.append("## 1. Scope")
lines.append("")
lines.append("Day23A audits the MJ1 0.4C group under fixed 1τ excitation where the AC amplitude is smaller than the DC component.")
lines.append("")
lines.append("Active protocols:")
lines.append("")
lines.append("- `0.4C DC`")
lines.append("- `0.4C + 0.1C 1τ`, κ = 0.25")
lines.append("- `0.4C + 0.3C 1τ`, κ = 0.75")
lines.append("")
lines.append("Excluded protocol:")
lines.append("")
lines.append("- `0.4C + 0.2C 1τ`, κ = 0.50")
lines.append("")
lines.append("The excluded record lacks the required pre-Vmax CC segment and cannot support Q_Vmax extraction, first-passage comparison, or boundary-region assignment.")
lines.append("")
lines.append("## 2. Exclusion registry")
lines.append("")
lines.append("```text")
lines.append(exclusion_table)
lines.append("```")
lines.append("")
lines.append("## 3. Protocol-mode decision")
lines.append("")
lines.append("Before applying Day21A/Day22A segmentation logic, Day23A audited whether the DC–AC protocols switch off AC after first reaching 4.2 V.")
lines.append("")
lines.append("The audit showed that both active DC–AC files retain substantial fixed-frequency current components after Vmax.")
lines.append("")
lines.append("Therefore, the original Day21A/Day22A AC-off Segment A/B/D framework is disabled.")
lines.append("")
lines.append("```text")
lines.append(framework_table)
lines.append("```")
lines.append("")
lines.append("Day23A uses a generalized G0/G1/G2 boundary-ordering framework instead.")
lines.append("")
lines.append("## 4. Generalized boundary-ordering framework")
lines.append("")
lines.append("The generalized regions are:")
lines.append("")
lines.append("- `G0`: shared pre-boundary region, Q <= min(Q_DC,Vmax, Q_DCAC,Vmax)")
lines.append("- `G1`: boundary-ordering split region, min(Q_DC,Vmax, Q_DCAC,Vmax) < Q <= max(Q_DC,Vmax, Q_DCAC,Vmax)")
lines.append("- `G2`: post-boundary region, Q > max(Q_DC,Vmax, Q_DCAC,Vmax)")
lines.append("")
lines.append("Geometry-corrected residuals are only meaningful in G0.")
lines.append("")
lines.append("G1 and G2 are not equivalent to Day21A/Day22A Segment B/D because AC continues after Vmax in Day23A.")
lines.append("")
lines.append("## 5. Boundary-ordering result")
lines.append("")
lines.append("```text")
lines.append(g_table)
lines.append("```")
lines.append("")
lines.append("Both active DC–AC protocols are DCAC-first by Q. Thus, even with AC_C < DC_C, the DC–AC trajectory reaches the 4.2 V boundary at lower Q than the DC reference.")
lines.append("")
lines.append("The Q_Vmax shift is amplitude-dependent:")
lines.append("")
lines.append("- κ = 0.25: Q shift ≈ 0.049 Ah")
lines.append("- κ = 0.75: Q shift ≈ 0.153 Ah")
lines.append("")
lines.append("This supports a boundary-shift effect that scales with excitation strength, but not necessarily a gain-preserving full-protocol advantage.")
lines.append("")
lines.append("## 6. Self-consistency audit-resolution estimate")
lines.append("")
lines.append("Day23A uses a DC self-consistency lower-bound estimate from the 0.4C DC reference.")
lines.append("")
lines.append("```text")
lines.append(resolution_table)
lines.append("```")
lines.append("")
lines.append("This is a lower-bound estimate for sampling, interpolation, and first-passage sensitivity. It is not a repeat-based experimental noise floor.")
lines.append("")
lines.append("## 7. Formal generalized verdict")
lines.append("")
lines.append("```text")
lines.append(closure_table)
lines.append("```")
lines.append("")
lines.append("## 8. Interpretation")
lines.append("")
lines.append("For κ = 0.75, Day23A shows a mixed pattern: DC–AC reaches the voltage boundary earlier and has a positive Q80_common raw gain in G0, but Q90_common is negative and the G2 median raw Δt is also negative. This is classified as boundary-leading with G0 gain but post-boundary penalty.")
lines.append("")
lines.append("For κ = 0.25, both Q80_common and Q90_common raw gains are within the Day23A self-consistency resolution. This is classified as boundary-leading but state-gain unresolved at common anchors.")
lines.append("")
lines.append("Therefore, Day23A does not support a persistent full-protocol gain under sub-DC AC amplitude. It also does not confirm non-geometric G0 electrochemical acceleration.")
lines.append("")
lines.append("## 9. Relation to Day21A and Day22A")
lines.append("")
lines.append("Day23A is not directly comparable to Day21A/Day22A as the same protocol family.")
lines.append("")
lines.append("Day21A and Day22A used an AC-off-after-Vmax protocol assumption:")
lines.append("")
lines.append("- DC–AC applied during CC")
lines.append("- AC switched off at first Vmax")
lines.append("- post-Vmax region interpreted as AC-off voltage-boundary / CV-coupled behavior")
lines.append("")
lines.append("Day23A violates this protocol assumption because AC continues after Vmax.")
lines.append("")
lines.append("Thus, Day23A is best interpreted as a protocol-mode contrast:")
lines.append("")
lines.append("- sub-DC AC amplitude")
lines.append("- fixed 1τ excitation")
lines.append("- continued AC after Vmax")
lines.append("- generalized boundary-ordering instead of AC-off segmentation")
lines.append("")
lines.append("## 10. Allowed claims")
lines.append("")
lines.append("Allowed:")
lines.append("")
lines.append("1. Day23A confirms that both active sub-DC AC protocols are DCAC-first by Q.")
lines.append("2. The boundary shift increases from κ = 0.25 to κ = 0.75.")
lines.append("3. κ = 0.25 produces only unresolved common-anchor state gain within audit resolution.")
lines.append("4. κ = 0.75 produces early G0 gain but loses the advantage at Q90/G2.")
lines.append("5. Day23A demonstrates that boundary-leading does not necessarily imply gain preservation.")
lines.append("6. Day23A is a different protocol-mode family from Day21A/Day22A due to continued AC after Vmax.")
lines.append("")
lines.append("## 11. Prohibited claims")
lines.append("")
lines.append("Do not claim:")
lines.append("")
lines.append("1. Day23A proves non-geometric G0 electrochemical acceleration.")
lines.append("2. Day23A is directly comparable to Day21A/Day22A as an identical AC-off protocol.")
lines.append("3. Day23A proves disappearance of DC–AC effects.")
lines.append("4. G2 raw gain or penalty is late-CV preservation in the Day21A/Day22A sense.")
lines.append("5. Temperature effects can be quantified; temperature summaries are missing.")
lines.append("")
lines.append("## 12. Key output files")
lines.append("")
lines.append(f"- Raw format inventory: `{OUT_DAY23A_FORMAT_INVENTORY}`")
lines.append(f"- Exclusion audit: `{OUT_DAY23A_EXCLUSION_AUDIT}`")
lines.append(f"- Active inventory: `{OUT_DAY23A_ACTIVE_FORMAT_INVENTORY}`")
lines.append(f"- Timebase audit: `{OUT_DAY23A_TIMEBASE_AUDIT}`")
lines.append(f"- Protocol-mode audit: `{OUT_DAY23A_PROTOCOL_MODE_AUDIT}`")
lines.append(f"- Framework decision: `{OUT_DAY23A_FRAMEWORK_DECISION}`")
lines.append(f"- Audit contract: `{OUT_DAY23A_AUDIT_CONTRACT_JSON}`")
lines.append(f"- File inventory: `{OUT_DAY23A_FILE_INVENTORY}`")
lines.append(f"- Load sanity: `{OUT_DAY23A_LOAD_SUMMARY}`")
lines.append(f"- Event audit: `{OUT_DAY23A_EVENT_AUDIT}`")
lines.append(f"- Q integration summary: `{OUT_DAY23A_Q_SUMMARY}`")
lines.append(f"- Final-Q audit: `{OUT_DAY23A_FINALQ_PAIR_AUDIT}`")
lines.append(f"- Resolution summary: `{OUT_DAY23A_RESOLUTION_SUMMARY}`")
lines.append(f"- G0/G1/G2 assignment: `{OUT_DAY23A_G_REGION_ASSIGNMENT}`")
lines.append(f"- Δt summary: `{OUT_DAY23A_DTQ_SUMMARY}`")
lines.append(f"- Generalized verdict: `{OUT_DAY23A_VERDICT}`")
lines.append(f"- Closure CSV: `{OUT_DAY23A_CLOSURE_CSV}`")
lines.append("")
lines.append("## 13. Closure status")
lines.append("")
lines.append("Day23A is closed as a generalized sub-DC-amplitude protocol-mode audit.")
lines.append("")
lines.append("Next recommended step:")
lines.append("")
lines.append("Commit Day23A notebook and audit outputs, excluding raw CSV files.")

closure_md = "\n".join(lines)

OUT_DAY23A_CLOSURE_MD.write_text(closure_md, encoding="utf-8")

print(f"[OK] Wrote Day23A closure note: {OUT_DAY23A_CLOSURE_MD}")
print("[OK] Day23A audit closed.")

[OK] Wrote Day23A closure note: /Users/louislu/pybamm-dcac-superimposed/data/day23A_step7_closure_note.md
[OK] Day23A audit closed.
